# ASOC sparse index tracking — Notebook 2: sequential low-turnover maintenance

This self-contained notebook loads the tracker cache generated by Notebook 1 and runs the low-turnover maintenance layer over the rolling holding windows.

The notebook intentionally does **not** include checkpoint/resume machinery, competitor methods, or LaTeX table generation.  It saves final numerical audit outputs only: hold summaries, decisions, trades, recovery diagnostics, selected and full \(\Delta w\)-grid rows, MALA summaries, and the full per-hold portfolio-composition path.

The defaults reproduce the paper-case branch when the same data and Notebook-1 cache are supplied and `RUN_PRODUCTION=True`.  The default `RUN_PRODUCTION=False` is a smoke-test mode.
## Reproducing the paper maintenance run

1. Run Notebook 1 with `RUN_PRODUCTION=True`.
2. Confirm that the cache path is `./outputs/production/tracker_cache/initial_tracker_fit01_nnz150_c25_floor005_53names.npz`.
3. Set `RUN_PRODUCTION=True` in this notebook.
4. Run this notebook top-to-bottom.
5. Use the final validation cell to compare the generated outputs with the paper values.


In [ ]:
# ===============================================================
# GitHub configuration — sequential tracker maintenance
# ===============================================================
# This notebook assumes that Notebook 1 has already exported the initial
# tracker cache.  Support for arbitrary externally supplied trackers is a
# future extension; for now, use the cache generated by Notebook 1.
# ===============================================================

from pathlib import Path
import os

# Reproducibility: set BLAS/thread controls before NumPy/Pandas are imported.
for _thread_env_var in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[_thread_env_var] = "1"


# Optional Colab Drive mount.  Leave False for local/Jupyter use.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f"Google Drive mount skipped: {exc}")

# User-editable paths.  Use the same OUTPUT_ROOT as Notebook 1.
DATA_PATH = Path("./data/merged_sp500_returns_2020_2025.csv")
OUTPUT_ROOT = Path("./outputs")

# Dataset format.  Set INDEX_COL="__LAST__" to use the final column.
INDEX_COL = "SP500"

# Paper-case default windows.  If the locked paper dates are unavailable,
# the notebook falls back to row-count windows.
USE_PAPER_WINDOWS_IF_AVAILABLE = True
FIT_WINDOW_LENGTH = 500
HOLD_WINDOW_LENGTH = 63
ROLL_STEP_LENGTH = 63
ABSORB_FINAL_STUB = True

# Holds to evaluate.  None means use all available rolling windows.
HOLD_START_NUMBER = 1
HOLD_END_NUMBER = None
REBALANCE_ENTERING_START_HOLD = False

# Run mode.  Smoke mode checks that the notebook works; production mode
# is the expensive paper-reproduction setting.
RUN_PRODUCTION = False
RUN_MODE = "production" if RUN_PRODUCTION else "smoke"
OUTPUT_DIR = OUTPUT_ROOT / RUN_MODE
TRACKER_CACHE_PATH = OUTPUT_DIR / "tracker_cache" / "initial_tracker_fit01_nnz150_c25_floor005_53names.npz"
RANDOM_SEED = 12345
VERBOSE = False
MAKE_PLOTS = True

# Calm-mode posterior directional gate: paper-case defaults.
CALM_EPS_DELTA = 1e-4
CALM_P_DIR = 0.60

# Recovery trigger and asymmetric recovery gates: paper-case defaults.
RECOVERY_TARGET_TE_BP = 15.0
RECOVERY_TARGET_GAP_BP = 2.0
RECOVERY_SHOCK_JUMP_BP = 7.0
RECOVERY_EXIT_GAP_BP = 2.0
RECOVERY_EPS_DELTA = 1e-4
RECOVERY_P_DIR = 0.55               # legacy scalar used in logs
RECOVERY_P_DIR_EXISTING = 0.525
RECOVERY_P_DIR_NEW_BUY = 0.575
RECOVERY_P_DIR_SELL = 0.525
RECOVERY_MAX_APPROVED_MOVES = 10
RECOVERY_MAX_NEW_BUY_NAMES = 4
RECOVERY_TARGET_ONE_WAY_TURNOVER = 0.010
RECOVERY_MAX_ONE_WAY_TURNOVER = 0.020
RECOVERY_BUY_TOPUP_ENABLED = True
RECOVERY_MAX_EXTRA_FUNDING_NAMES = 6
RECOVERY_PROTECT_TOP_N_FUNDERS = 10
RECOVERY_PROTECT_WEIGHT_ABOVE = 0.03
RECOVERY_DUST_FLOOR = 0.005

# Delta-w grid and scoring: paper-case defaults.
SIGMA2_DW_BASE = 1e-4
C_GRID_DW = [1.0, 10.0, 25.0, 50.0, 75.0, 100.0, 125.0, 150.0, 200.0]
NNZ_EFF_THRESH_DW = 1e-4
SCORE_TE_IMPROVEMENT_TARGET = 0.05
SCORE_L1_DW_TARGET = 0.03
SCORE_L1_DW_SCALE = 0.02

# Name of the calm-mode candidate generated from CALM_EPS_DELTA and CALM_P_DIR.
def _format_eps_tag_config(eps: float) -> str:
    if abs(eps - 5e-5) <= 1e-12:
        return "5e-5"
    if abs(eps - 1e-4) <= 1e-12:
        return "1e-4"
    if abs(eps - 2.5e-4) <= 1e-12:
        return "2p5e-4"
    if abs(eps - 5e-4) <= 1e-12:
        return "5e-4"
    if abs(eps - 1e-3) <= 1e-12:
        return "1e-3"
    return f"{eps:.0e}".replace("-0", "-")

_selected_prefix = "moderate_dir" if (abs(CALM_EPS_DELTA - 1e-4) <= 1e-12 and abs(CALM_P_DIR - 0.60) <= 1e-12) else "dir"
SELECTED_ROPE_CANDIDATE = f"{_selected_prefix}_eps{_format_eps_tag_config(CALM_EPS_DELTA)}_pi{CALM_P_DIR:.2f}"

# Numerical settings: smoke vs production.
DW_SAPG_N_ITER = 15_000 if RUN_PRODUCTION else 1_000
DW_FISTA_MAX_ITER = 4_000 if RUN_PRODUCTION else 1_000
DW_MALA_N_BURN_STEPS = 50_000 if RUN_PRODUCTION else 2_000
DW_MALA_N_ITER_LONG = 250_000 if RUN_PRODUCTION else 10_000
DW_MALA_THIN = 6 if RUN_PRODUCTION else 5
DW_MALA_BURN_KEPT = 5_000 if RUN_PRODUCTION else 200
DW_MALA_MAX_LAG_ESS = 200 if RUN_PRODUCTION else 50

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_MODE = {RUN_MODE}")
print(f"DATA_PATH = {DATA_PATH}")
print(f"TRACKER_CACHE_PATH = {TRACKER_CACHE_PATH}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — rolling data preparation
# T=500 FIT windows, quarterly holds, final stub absorbed into Hold 16
#
# This block prepares every FIT window:
#   (i) centers y and R,
#   (ii) computes weighted-l1 alpha scales,
#   (iii) stores everything in rolling_fit_prepared.
#
# Convention:
#   - merged dataframe has asset-return columns first;
#   - index-return column is last, unless explicitly supplied.
# ===============================================================

import os
import warnings
import numpy as np
import pandas as pd
from typing import Optional, Dict, Any, List

# ---------------------------------------------------------------
# Runtime config
# ---------------------------------------------------------------

warnings.filterwarnings("ignore")

rng = np.random.default_rng(RANDOM_SEED)

# ---------------------------------------------------------------
# Data — merged returns, assets first and index return as last column
# ---------------------------------------------------------------

from pathlib import Path

DATA_PATH = Path(DATA_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PATH_MERGED = str(DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"DATA_PATH does not exist: {DATA_PATH}\n"
        "Edit the configuration cell before running the notebook."
    )

df_ret = pd.read_csv(
    DATA_PATH,
    index_col=0,
    parse_dates=True,
).sort_index()

# Basic data checks.  Missing values should be handled before using the notebook.
if not df_ret.index.is_monotonic_increasing:
    raise ValueError("The date index must be sorted increasingly.")
if df_ret.isna().any().any():
    raise ValueError("The return matrix contains missing values; clean or impute them before running.")


def split_X_y(
    df: pd.DataFrame,
    idx_col: Optional[str] = None,
):
    """
    Split merged returns dataframe into asset-return matrix R and
    index-return vector y.

    Parameters
    ----------
    df : pd.DataFrame
        Merged returns dataframe.
    idx_col : str, optional
        Name of the index-return column. If None, the final column is used.

    Returns
    -------
    R : np.ndarray, shape (T, p)
        Asset returns.
    y : np.ndarray, shape (T,)
        Index returns.
    idx_col : str
        Name of the index-return column.
    """
    if idx_col is None:
        idx_col = df.columns[-1]

    if idx_col not in df.columns:
        raise KeyError(f"INDEX_COL={idx_col!r} is not present in the dataframe columns.")

    y = df[idx_col].to_numpy(float).reshape(-1)
    R = df.drop(columns=[idx_col]).to_numpy(float)

    return R, y, idx_col


def slice_window(
    df: pd.DataFrame,
    start: str,
    end: str,
    idx_col: Optional[str] = None,
):
    """
    Slice a date window and return dates, R, and y.
    """
    out = df.loc[start:end].copy()

    R, y, _ = split_X_y(out, idx_col=idx_col)

    assert out.index.is_monotonic_increasing
    assert len(out) > 0
    assert not np.isnan(R).any()
    assert not np.isnan(y).any()

    return out.index, R, y


def center_data(
    y: np.ndarray,
    R: np.ndarray,
) -> Dict[str, np.ndarray | float]:
    """
    Center the index and asset returns over one FIT window.

    The centering constants are stored so that tracking error can later
    be evaluated on raw, uncentered returns using the corresponding
    un-centering correction.
    """
    y = np.asarray(y, dtype=float).reshape(-1)
    R = np.asarray(R, dtype=float)

    y_mu = float(y.mean())
    R_mu = R.mean(axis=0)

    y_c = y - y_mu
    R_c = R - R_mu

    return {
        "y_c": y_c,
        "R_c": R_c,
        "y_mu": y_mu,
        "R_mu": R_mu,
    }


def compute_alpha_scales(
    R_c: np.ndarray,
    eps: float = 1e-8,
) -> np.ndarray:
    """
    Compute weighted-l1 alpha scales for one centered FIT design matrix.

    alpha_j is proportional to ||R_{.j}||_2 / sqrt(T), then mean-normalized
    to one. This keeps the scale convention consistent across rolling windows.
    """
    R_c = np.asarray(R_c, dtype=float)
    T, _ = R_c.shape

    s = np.linalg.norm(R_c, axis=0) / max(np.sqrt(T), 1.0)
    s = np.clip(s, eps, None)
    s = s / s.mean()

    return s


# ---------------------------------------------------------------
# Rolling split definition
# ---------------------------------------------------------------

PAPER_ROLLING_WINDOWS = [
    {"k": 1, "fit_start": "2020-01-03", "fit_end": "2021-12-27", "hold_start": "2021-12-28", "hold_end": "2022-03-25"},
    {"k": 2, "fit_start": "2020-04-02", "fit_end": "2022-03-25", "hold_start": "2022-03-28", "hold_end": "2022-06-27"},
    {"k": 3, "fit_start": "2020-07-02", "fit_end": "2022-06-27", "hold_start": "2022-06-28", "hold_end": "2022-09-27"},
    {"k": 4, "fit_start": "2020-10-02", "fit_end": "2022-09-27", "hold_start": "2022-09-28", "hold_end": "2022-12-27"},
    {"k": 5, "fit_start": "2021-01-04", "fit_end": "2022-12-27", "hold_start": "2022-12-28", "hold_end": "2023-03-27"},
    {"k": 6, "fit_start": "2021-04-01", "fit_end": "2023-03-27", "hold_start": "2023-03-28", "hold_end": "2023-06-27"},
    {"k": 7, "fit_start": "2021-07-01", "fit_end": "2023-06-27", "hold_start": "2023-06-28", "hold_end": "2023-09-27"},
    {"k": 8, "fit_start": "2021-10-01", "fit_end": "2023-09-27", "hold_start": "2023-09-28", "hold_end": "2023-12-27"},
    {"k": 9, "fit_start": "2021-12-31", "fit_end": "2023-12-27", "hold_start": "2023-12-28", "hold_end": "2024-03-27"},
    {"k": 10, "fit_start": "2022-03-31", "fit_end": "2024-03-27", "hold_start": "2024-03-28", "hold_end": "2024-06-27"},
    {"k": 11, "fit_start": "2022-07-01", "fit_end": "2024-06-27", "hold_start": "2024-06-28", "hold_end": "2024-09-27"},
    {"k": 12, "fit_start": "2022-10-03", "fit_end": "2024-09-27", "hold_start": "2024-09-30", "hold_end": "2024-12-27"},
    {"k": 13, "fit_start": "2023-01-03", "fit_end": "2024-12-27", "hold_start": "2024-12-30", "hold_end": "2025-03-28"},
    {"k": 14, "fit_start": "2023-03-31", "fit_end": "2025-03-28", "hold_start": "2025-03-31", "hold_end": "2025-06-27"},
    {"k": 15, "fit_start": "2023-06-30", "fit_end": "2025-06-27", "hold_start": "2025-06-30", "hold_end": "2025-09-29"},
    {"k": 16, "fit_start": "2023-10-02", "fit_end": "2025-09-29", "hold_start": "2025-09-30", "hold_end": "2025-12-31"},
]

T_fit = int(FIT_WINDOW_LENGTH)


def _paper_windows_available(df: pd.DataFrame, windows: list[dict]) -> bool:
    """Return True when all locked paper-window boundary dates are in df.index."""
    idx = pd.DatetimeIndex(df.index)
    for win in windows:
        for key in ["fit_start", "fit_end", "hold_start", "hold_end"]:
            if pd.Timestamp(win[key]) not in idx:
                return False
    return True


def make_row_count_rolling_windows(
    df: pd.DataFrame,
    *,
    fit_length: int,
    hold_length: int,
    step_length: Optional[int] = None,
    absorb_final_stub: bool = True,
) -> list[dict]:
    """
    Create rolling FIT/HOLD windows by row counts.

    This is a generic fallback for non-paper datasets.  For exact paper
    reproduction with the supplied data, USE_PAPER_WINDOWS_IF_AVAILABLE=True
    uses the locked date windows above.
    """
    if fit_length <= 0 or hold_length <= 0:
        raise ValueError("fit_length and hold_length must be positive integers.")

    if step_length is None:
        step_length = hold_length
    if step_length <= 0:
        raise ValueError("step_length must be positive.")

    dates = pd.DatetimeIndex(df.index)
    windows = []
    k = 1
    fit_start_i = 0

    while True:
        fit_end_i = fit_start_i + fit_length - 1
        hold_start_i = fit_end_i + 1
        if hold_start_i >= len(dates):
            break

        hold_end_i = min(hold_start_i + hold_length - 1, len(dates) - 1)
        windows.append({
            "k": k,
            "fit_start": str(dates[fit_start_i].date()),
            "fit_end": str(dates[fit_end_i].date()),
            "hold_start": str(dates[hold_start_i].date()),
            "hold_end": str(dates[hold_end_i].date()),
        })

        if hold_end_i >= len(dates) - 1:
            break

        fit_start_i += step_length
        k += 1

    if absorb_final_stub and windows:
        windows[-1]["hold_end"] = str(dates[-1].date())

    return windows


if bool(USE_PAPER_WINDOWS_IF_AVAILABLE) and _paper_windows_available(df_ret, PAPER_ROLLING_WINDOWS):
    rolling_windows = [dict(win) for win in PAPER_ROLLING_WINDOWS]
    rolling_window_source = "locked_paper_windows"
else:
    rolling_windows = make_row_count_rolling_windows(
        df_ret,
        fit_length=int(FIT_WINDOW_LENGTH),
        hold_length=int(HOLD_WINDOW_LENGTH),
        step_length=int(globals().get("ROLL_STEP_LENGTH", HOLD_WINDOW_LENGTH)),
        absorb_final_stub=bool(ABSORB_FINAL_STUB),
    )
    rolling_window_source = "row_count_windows"

if len(rolling_windows) == 0:
    raise RuntimeError("No rolling windows could be constructed from the supplied dataset.")

# ---------------------------------------------------------------
# Old-code-style aliases for the first window
# ---------------------------------------------------------------

idx_col_config = None if INDEX_COL in [None, "", "__LAST__"] else str(INDEX_COL)
R_all, y_all, idx_name = split_X_y(df_ret, idx_col=idx_col_config)

fit_start, fit_end = (
    rolling_windows[0]["fit_start"],
    rolling_windows[0]["fit_end"],
)

hold1_start, hold1_end = (
    rolling_windows[0]["hold_start"],
    rolling_windows[0]["hold_end"],
)

dates_fit, R_fit, y_fit = slice_window(
    df_ret,
    fit_start,
    fit_end,
    idx_col=idx_name,
)

dates_h1, R_hold1, y_hold1 = slice_window(
    df_ret,
    hold1_start,
    hold1_end,
    idx_col=idx_name,
)

print(
    "Full:",
    df_ret.shape,
    "| Assets:",
    df_ret.shape[1] - 1,
    "| Index col:",
    idx_name,
)

print(
    "Date range:",
    df_ret.index.min().date(),
    "->",
    df_ret.index.max().date(),
)

print(f"Rolling windows: {len(rolling_windows)} (source={rolling_window_source})")
print("Initial FIT:", R_fit.shape, "| Hold1:", R_hold1.shape)

# ---------------------------------------------------------------
# Prepare every FIT window
# ---------------------------------------------------------------

rolling_fit_prepared: List[Dict[str, Any]] = []

for win in rolling_windows:
    k = int(win["k"])

    dates_fit_k, R_fit_k, y_fit_k = slice_window(
        df_ret,
        win["fit_start"],
        win["fit_end"],
        idx_col=idx_name,
    )

    # Safety check: every FIT window should have exactly T_fit rows.
    assert len(dates_fit_k) == T_fit, (
        f"Window {k}: expected {T_fit} FIT rows, got {len(dates_fit_k)}."
    )

    cent_k = center_data(y_fit_k, R_fit_k)

    y_c_k = cent_k["y_c"]
    R_c_k = cent_k["R_c"]
    y_mu_k = cent_k["y_mu"]
    R_mu_k = cent_k["R_mu"]

    alpha_k = compute_alpha_scales(R_c_k)

    rolling_fit_prepared.append(
        {
            "k": k,
            "fit_start": win["fit_start"],
            "fit_end": win["fit_end"],
            "hold_start": win["hold_start"],
            "hold_end": win["hold_end"],

            # Raw FIT data
            "dates_fit": dates_fit_k,
            "R_fit": R_fit_k,
            "y_fit": y_fit_k,

            # Centered FIT data
            "R_c": R_c_k,
            "y_c": y_c_k,
            "R_mu": R_mu_k,
            "y_mu": y_mu_k,

            # Weighted-l1 scales
            "alpha": alpha_k,
        }
    )

# ---------------------------------------------------------------
# First-window aliases used by downstream helper cells
# ---------------------------------------------------------------

first_fit = rolling_fit_prepared[0]

y_c = first_fit["y_c"]
R_c = first_fit["R_c"]
y_mu = first_fit["y_mu"]
R_mu = first_fit["R_mu"]
alpha = first_fit["alpha"]

T, p = R_c.shape

print(f"First FIT centered shape: T={T}, p={p}")
print(
    f"First FIT alpha weights: "
    f"min={alpha.min():.3f}, max={alpha.max():.3f}, mean={alpha.mean():.3f}"
)

# ---------------------------------------------------------------
# Compact sanity-check table for all FIT windows
# ---------------------------------------------------------------

fit_prep_summary = pd.DataFrame(
    [
        {
            "k": d["k"],
            "fit_start": d["fit_start"],
            "fit_end": d["fit_end"],
            "fit_n": len(d["dates_fit"]),
            "p": d["R_c"].shape[1],
            "alpha_min": float(d["alpha"].min()),
            "alpha_max": float(d["alpha"].max()),
            "alpha_mean": float(d["alpha"].mean()),
            "y_mean_raw": float(d["y_mu"]),
        }
        for d in rolling_fit_prepared
    ]
)

print(fit_prep_summary.to_string(index=False))

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Block 0B
# Load initial Tracker cache from Notebook 1
#
# Purpose:
#   Load the locked FIT-1 Tracker created by Notebook 1 so that this
#   notebook can start directly from Hold 1.
#
# Default cache:
#   initial_tracker_fit01_nnz150_c25_59names.npz
#
# Naming convention:
#   The loaded objects are exposed under the names expected by the
#   Block-14 sequential pipeline:
#       initial_tracker_w
#       initial_fit_te_target
#       idx_name_for_seq
#       asset_cols_for_seq
#       w_locked_fit
#       S_locked_idx_fit
# ===============================================================

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# 0) Cache location
# ---------------------------------------------------------------

TRACKER_CACHE_PATH = Path(TRACKER_CACHE_PATH)
TRACKER_CACHE_DIR = TRACKER_CACHE_PATH.parent
TRACKER_CACHE_FILENAME = TRACKER_CACHE_PATH.name


def _cache_scalar_to_str(x) -> str:
    """
    Robustly convert a scalar/object/string npz field to str.
    """
    arr = np.asarray(x)

    if arr.shape == ():
        return str(arr.item())

    if arr.size == 1:
        return str(arr.reshape(-1)[0])

    return str(arr.tolist())


def _cache_label_from_path(path: Path) -> str:
    """
    Build a safe label used in figure/output directory names.
    """
    stem = path.stem
    stem = stem.replace("initial_tracker_fit01_", "")
    stem = stem.replace("initial_tracker_", "")
    stem = re.sub(r"[^A-Za-z0-9_\-]+", "_", stem)
    return stem if stem else "tracker_cache"


if not TRACKER_CACHE_PATH.exists():
    raise FileNotFoundError(
        "Initial Tracker cache not found.\n"
        f"Expected: {TRACKER_CACHE_PATH}\n"
        "Either create this file from Notebook 1, or edit TRACKER_CACHE_FILENAME."
    )

tracker_cache = np.load(TRACKER_CACHE_PATH, allow_pickle=True)
tracker_cache_label = _cache_label_from_path(TRACKER_CACHE_PATH)

# ---------------------------------------------------------------
# 1) Required arrays/scalars
# ---------------------------------------------------------------

w_locked_fit = np.asarray(tracker_cache["w_locked_fit"], dtype=float).reshape(-1)
S_locked_idx_fit = np.asarray(tracker_cache["S_locked_idx_fit"], dtype=int).reshape(-1)

initial_tracker_w = w_locked_fit.copy()
initial_fit_te_target = float(np.asarray(tracker_cache["initial_fit_te_target"]).reshape(-1)[0])

asset_cols_for_seq = [str(x) for x in np.asarray(tracker_cache["asset_cols"], dtype=object).reshape(-1)]
idx_name_for_seq = _cache_scalar_to_str(tracker_cache["idx_name"])

# Expose these aliases as well because several helper blocks use them.
asset_cols = asset_cols_for_seq
idx_name_from_tracker_cache = idx_name_for_seq
asset_cols_from_tracker_cache = asset_cols_for_seq

selected_support_rule_fit = _cache_scalar_to_str(
    tracker_cache["selected_support_rule_fit"]
) if "selected_support_rule_fit" in tracker_cache.files else "unknown"

selected_eps_fit = float(np.asarray(tracker_cache["selected_eps_fit"]).reshape(-1)[0]) if "selected_eps_fit" in tracker_cache.files else np.nan
selected_pi_pos_fit = float(np.asarray(tracker_cache["selected_pi_pos_fit"]).reshape(-1)[0]) if "selected_pi_pos_fit" in tracker_cache.files else np.nan

if "metadata_json" in tracker_cache.files:
    tracker_cache_metadata = json.loads(_cache_scalar_to_str(tracker_cache["metadata_json"]))
else:
    tracker_cache_metadata = {}

# ---------------------------------------------------------------
# 2) Validate cache against merged return data
# ---------------------------------------------------------------

if "df_ret" not in globals():
    raise RuntimeError("df_ret is not defined. Run the data-prep block before the cache loader.")

if idx_name_for_seq not in df_ret.columns:
    raise KeyError(
        f"Cache idx_name={idx_name_for_seq!r} is not a column in df_ret. "
        f"Available last column is {df_ret.columns[-1]!r}."
    )

asset_cols_from_data = [str(c) for c in df_ret.columns if str(c) != str(idx_name_for_seq)]

if len(asset_cols_from_data) != len(asset_cols_for_seq):
    raise ValueError(
        "Cache/data asset count mismatch: "
        f"cache has {len(asset_cols_for_seq)}, data has {len(asset_cols_from_data)}."
    )

if asset_cols_from_data != asset_cols_for_seq:
    # This should be strict: the portfolio weights are order-dependent.
    mismatches = [
        (i, a, b)
        for i, (a, b) in enumerate(zip(asset_cols_from_data, asset_cols_for_seq))
        if a != b
    ]
    preview = mismatches[:10]
    raise ValueError(
        "Cache/data asset-column order mismatch. Portfolio weights are order-dependent.\n"
        f"First mismatches: {preview}"
    )

if w_locked_fit.size != len(asset_cols_for_seq):
    raise ValueError(
        f"w_locked_fit has length {w_locked_fit.size}, "
        f"but cache asset list has length {len(asset_cols_for_seq)}."
    )

if not np.isfinite(initial_fit_te_target):
    raise ValueError("initial_fit_te_target is missing or non-finite in the cache.")

# Optional date checks against rolling_windows[0].
if "rolling_windows" in globals() and len(rolling_windows) > 0:
    win0 = rolling_windows[0]
    for key in ["fit_start", "fit_end", "hold1_start", "hold1_end"]:
        if key in tracker_cache.files:
            cached_val = _cache_scalar_to_str(tracker_cache[key])
            win_key = "hold_start" if key == "hold1_start" else "hold_end" if key == "hold1_end" else key
            expected_val = str(win0[win_key])
            if str(pd.Timestamp(cached_val).date()) != str(pd.Timestamp(expected_val).date()):
                print(
                    f"[warning] cache {key}={cached_val} does not match "
                    f"rolling_windows[0][{win_key!r}]={expected_val}"
                )

# Recompute Hold-1 realised TE as a quick sanity check.
if "R_hold1" in globals() and "y_hold1" in globals():
    hold1_active_check = (np.asarray(R_hold1, dtype=float) @ initial_tracker_w) - np.asarray(y_hold1, dtype=float).reshape(-1)
    hold1_te_check = float(np.sqrt(np.mean(hold1_active_check ** 2)))
else:
    hold1_te_check = np.nan

# ---------------------------------------------------------------
# 3) Print cache summary
# ---------------------------------------------------------------

print("\n[Initial Tracker cache loaded]")
print(f"cache path = {TRACKER_CACHE_PATH}")
print(f"tracker_cache_label = {tracker_cache_label}")
print(f"idx_name_for_seq = {idx_name_for_seq}")
print(f"p = {initial_tracker_w.size}")
print(f"|S_locked| = {S_locked_idx_fit.size}")
print(f"selected_support_rule = {selected_support_rule_fit}")
print(f"selected eps = {selected_eps_fit:.1e}")
print(f"selected pi_pos = {selected_pi_pos_fit:.2f}")
print(f"sum(initial_tracker_w) = {initial_tracker_w.sum():+.12f}")
print(f"nnz(initial_tracker_w) = {np.count_nonzero(np.abs(initial_tracker_w) > 1e-12)}")
print(f"shorts(initial_tracker_w) = {np.count_nonzero(initial_tracker_w < -1e-12)}")
print(
    f"initial_fit_te_target = {initial_fit_te_target:.6e} "
    f"({1e4 * initial_fit_te_target:.3f} bp)"
)

if np.isfinite(hold1_te_check):
    print(
        f"Hold-1 realised TE check = {hold1_te_check:.6e} "
        f"({1e4 * hold1_te_check:.3f} bp)"
    )

if tracker_cache_metadata:
    print("\n[Cache metadata]")
    for key in [
        "cache_name",
        "fit_start",
        "fit_end",
        "hold1_start",
        "hold1_end",
        "support_size",
        "hold1_te_bp",
    ]:
        if key in tracker_cache_metadata:
            print(f"  {key}: {tracker_cache_metadata[key]}")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — sequential plotting helpers
# Sequential hold-window evaluation helpers
#
# Purpose:
#   Evaluate the Tracker on realised, uncentred hold returns.
#
# Key conventions:
#   - Portfolio name in reports/plots: "Tracker"
#   - Active return = Tracker return - index return
#   - TE is realised out-of-sample RMSE on raw returns
#   - Vertical dashed lines mark the start of each hold window
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, List, Optional, Sequence, Tuple
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# ---------------------------------------------------------------
# 0) General dataframe/window helpers
# ---------------------------------------------------------------

def infer_asset_columns_from_df(
    df_ret: pd.DataFrame,
    idx_name: Optional[str] = None,
) -> Tuple[List[str], str]:
    """
    Infer asset columns and index column.

    Convention:
      - if idx_name is supplied, use that as the index-return column;
      - otherwise use the last column as the index-return column;
      - assets are all remaining columns.
    """
    if not isinstance(df_ret, pd.DataFrame):
        raise TypeError("df_ret must be a pandas DataFrame.")

    if df_ret.shape[1] < 2:
        raise ValueError("df_ret must contain asset columns and one index column.")

    cols = list(df_ret.columns)

    if idx_name is not None:
        if idx_name not in cols:
            raise KeyError(f"idx_name={idx_name!r} is not a column of df_ret.")
        idx_col = str(idx_name)
    else:
        idx_col = str(cols[-1])

    asset_cols = [str(c) for c in cols if str(c) != idx_col]

    if len(asset_cols) == 0:
        raise ValueError("No asset columns inferred.")

    return asset_cols, idx_col


def _get_window_field(
    win: Dict[str, Any],
    candidates: Sequence[str],
    *,
    required: bool = True,
    default: Any = None,
) -> Any:
    """
    Get a field from a rolling-window dictionary using several possible names.
    """
    for key in candidates:
        if key in win:
            return win[key]

    if required:
        raise KeyError(
            f"Could not find any of these keys in rolling window: {list(candidates)}. "
            f"Available keys: {list(win.keys())}"
        )

    return default


def get_hold_dates_from_window(win: Dict[str, Any]) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """
    Resolve hold start/end dates from a rolling-window dictionary.
    """
    hold_start = _get_window_field(
        win,
        candidates=[
            "hold_start",
            "hold1_start",
            "h_start",
            "test_start",
            "oos_start",
            "start_hold",
        ],
    )

    hold_end = _get_window_field(
        win,
        candidates=[
            "hold_end",
            "hold1_end",
            "h_end",
            "test_end",
            "oos_end",
            "end_hold",
        ],
    )

    return pd.Timestamp(hold_start), pd.Timestamp(hold_end)


def get_fit_dates_from_window(win: Dict[str, Any]) -> Tuple[pd.Timestamp, pd.Timestamp]:
    """
    Resolve fit start/end dates from a rolling-window dictionary.
    """
    fit_start = _get_window_field(
        win,
        candidates=[
            "fit_start",
            "train_start",
            "is_start",
            "start_fit",
        ],
    )

    fit_end = _get_window_field(
        win,
        candidates=[
            "fit_end",
            "train_end",
            "is_end",
            "end_fit",
        ],
    )

    return pd.Timestamp(fit_start), pd.Timestamp(fit_end)


def extract_raw_returns_for_dates(
    df_ret: pd.DataFrame,
    *,
    start: Any,
    end: Any,
    asset_cols: Sequence[str],
    idx_col: str,
) -> Dict[str, Any]:
    """
    Extract realised raw asset/index returns over [start, end].
    """
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    df_local = df_ret.copy()

    if not isinstance(df_local.index, pd.DatetimeIndex):
        df_local.index = pd.to_datetime(df_local.index)

    df_slice = df_local.loc[(df_local.index >= start) & (df_local.index <= end)]

    if df_slice.empty:
        raise ValueError(f"No rows found in df_ret over [{start}, {end}].")

    R_raw = df_slice[list(asset_cols)].to_numpy(dtype=float)
    y_raw = df_slice[idx_col].to_numpy(dtype=float).reshape(-1)
    dates = pd.DatetimeIndex(df_slice.index)

    return {
        "y_raw": y_raw,
        "R_raw": R_raw,
        "dates": dates,
        "start": dates[0],
        "end": dates[-1],
        "n_obs": int(len(dates)),
    }


def extract_hold_data_from_window(
    df_ret: pd.DataFrame,
    win: Dict[str, Any],
    *,
    asset_cols: Sequence[str],
    idx_col: str,
) -> Dict[str, Any]:
    """
    Extract realised raw hold-window returns for one rolling-window record.
    """
    hold_start, hold_end = get_hold_dates_from_window(win)

    return extract_raw_returns_for_dates(
        df_ret,
        start=hold_start,
        end=hold_end,
        asset_cols=asset_cols,
        idx_col=idx_col,
    )


# ---------------------------------------------------------------
# 1) Portfolio return / active return / TE helpers
# ---------------------------------------------------------------

def tracker_returns_raw(
    R_raw: np.ndarray,
    w: np.ndarray,
) -> np.ndarray:
    """
    Realised raw Tracker returns: R_raw @ w.
    """
    R_raw = np.asarray(R_raw, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)

    return (R_raw @ w).reshape(-1)


def active_returns_raw(
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    w: np.ndarray,
) -> np.ndarray:
    """
    Active return = Tracker return - index return.
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    r_tracker = tracker_returns_raw(R_raw, w)

    return r_tracker - y_raw


def realised_te_rmse_raw(
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    w: np.ndarray,
) -> float:
    """
    Realised tracking-error RMSE on uncentred/raw returns.
    """
    active = active_returns_raw(y_raw, R_raw, w)

    return float(np.sqrt(np.mean(active ** 2)))


def rolling_rmse_np(
    x: np.ndarray,
    window: int = 20,
) -> np.ndarray:
    """
    Rolling RMSE with NaN for the first window-1 entries.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    out = np.full(x.size, np.nan, dtype=float)

    if x.size < window:
        return out

    x2 = x ** 2
    csum = np.cumsum(np.insert(x2, 0, 0.0))
    out[window - 1:] = np.sqrt((csum[window:] - csum[:-window]) / float(window))

    return out


def active_drawdown(
    cumulative_active: np.ndarray,
) -> Tuple[float, np.ndarray]:
    """
    Drawdown of cumulative active return.

    Returns
    -------
    max_drawdown : float
        Minimum drawdown value.
    drawdown_series : np.ndarray
        Cumulative-active drawdown path.
    """
    cumulative_active = np.asarray(cumulative_active, dtype=float).reshape(-1)
    running_max = np.maximum.accumulate(cumulative_active)
    dd = cumulative_active - running_max

    return float(np.min(dd)), dd


def portfolio_structure_for_report(
    w: np.ndarray,
    *,
    topk: Sequence[int] = (10, 25),
    tol: float = 1e-12,
) -> Dict[str, Any]:
    """
    Structure diagnostics for a held Tracker.
    """
    w = np.asarray(w, dtype=float).reshape(-1)
    abs_w = np.abs(w)
    L1 = float(abs_w.sum()) + 1e-16
    order = np.argsort(-abs_w)

    out = {
        "sum_weights": float(w.sum()),
        "nnz": int(np.count_nonzero(abs_w > tol)),
        "n_shorts": int(np.count_nonzero(w < -tol)),
        "HHI_abs": float(np.sum((abs_w / L1) ** 2)),
    }

    for K in topk:
        K_eff = min(int(K), w.size)
        out[f"top{K}_L1_share"] = float(np.sum(abs_w[order[:K_eff]]) / L1)

    return out


# ---------------------------------------------------------------
# 2) Hold-window metric evaluation
# ---------------------------------------------------------------

def evaluate_tracker_on_hold(
    *,
    hold_number: int,
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    dates: Sequence[Any],
    w: np.ndarray,
    fit_te_target: Optional[float] = None,
    rebalance_label: str = "initial portfolio",
    rebalance_decision: Optional[Dict[str, Any]] = None,
    roll_window: int = 20,
    portfolio_label: str = "Tracker",
) -> Dict[str, Any]:
    """
    Evaluate one held Tracker on one realised hold window.
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    R_raw = np.asarray(R_raw, dtype=float)
    w = np.asarray(w, dtype=float).reshape(-1)
    dates = pd.DatetimeIndex(dates)

    r_index = y_raw
    r_tracker = tracker_returns_raw(R_raw, w)
    active = r_tracker - r_index

    te_rmse = float(np.sqrt(np.mean(active ** 2)))
    mae = float(np.mean(np.abs(active)))
    mean_active = float(np.mean(active))
    std_active = float(np.std(active, ddof=1)) if active.size > 1 else np.nan

    cumulative_index = np.cumprod(1.0 + r_index)
    cumulative_tracker = np.cumprod(1.0 + r_tracker)
    cumulative_active = np.cumsum(active)

    max_active_dd, active_dd = active_drawdown(cumulative_active)

    roll_te = rolling_rmse_np(active, window=roll_window)

    if fit_te_target is None or not np.isfinite(fit_te_target):
        hit_rate = np.nan
        fit_te_target = np.nan
    else:
        hit_rate = float(np.mean(np.abs(active) < float(fit_te_target)))

    structure = portfolio_structure_for_report(w)

    if rebalance_decision is None:
        no_trade = np.nan
        no_trade_reason = ""
        implementation_case = rebalance_label
        turnover = 0.0
        n_trades = 0
        n_increases = 0
        n_decreases = 0
    else:
        no_trade = bool(rebalance_decision.get("no_trade", False))
        no_trade_reason = str(rebalance_decision.get("no_trade_reason", ""))
        implementation_case = str(rebalance_decision.get("implementation_case", ""))

        trade_summary = rebalance_decision.get("implemented_trade_summary", {})
        turnover = float(trade_summary.get("turnover_half_L1", 0.0))
        n_trades = int(trade_summary.get("nnz_trades", 0))
        n_increases = int(trade_summary.get("n_increases", 0))
        n_decreases = int(trade_summary.get("n_decreases", 0))

    row = {
        "hold": int(hold_number),
        "portfolio": portfolio_label,
        "start": dates[0],
        "end": dates[-1],
        "n_obs": int(len(dates)),
        "TE_RMSE": te_rmse,
        "TE_bp": 1e4 * te_rmse,
        "MAE_bp": 1e4 * mae,
        "mean_active_bp": 1e4 * mean_active,
        "std_active_bp": 1e4 * std_active,
        "FIT_TE_target": float(fit_te_target) if np.isfinite(fit_te_target) else np.nan,
        "FIT_TE_target_bp": 1e4 * float(fit_te_target) if np.isfinite(fit_te_target) else np.nan,
        "hit_rate_vs_FIT_TE": hit_rate,
        "max_active_drawdown": max_active_dd,
        "cum_index_final": float(cumulative_index[-1]),
        "cum_tracker_final": float(cumulative_tracker[-1]),
        "cum_active_final": float(cumulative_active[-1]),
        "rebalance_label": rebalance_label,
        "no_trade": no_trade,
        "no_trade_reason": no_trade_reason,
        "implementation_case": implementation_case,
        "turnover": turnover,
        "n_trades": n_trades,
        "n_increases": n_increases,
        "n_decreases": n_decreases,
        **structure,
    }

    paths = pd.DataFrame(
        {
            "date": dates,
            "hold": int(hold_number),
            "index_return": r_index,
            "tracker_return": r_tracker,
            "active_return": active,
            "abs_active_bp": 1e4 * np.abs(active),
            "cum_index": cumulative_index,
            "cum_tracker": cumulative_tracker,
            "cum_active": cumulative_active,
            "active_drawdown": active_dd,
            "rolling_TE_bp": 1e4 * roll_te,
        }
    )

    return {
        "row": row,
        "paths": paths,
    }


# ---------------------------------------------------------------
# 3) Plotting across multiple holds
# ---------------------------------------------------------------

def _add_hold_start_lines(
    ax,
    hold_start_dates: Sequence[Any],
    *,
    ymin: Optional[float] = None,
    ymax: Optional[float] = None,
    label_holds: bool = True,
) -> None:
    """
    Add vertical dashed lines at hold starts.
    """
    for h, d in enumerate(hold_start_dates, start=1):
        d = pd.Timestamp(d)

        ax.axvline(d, linestyle="--", linewidth=0.8, alpha=0.65)

        if label_holds:
            y_text = ymax if ymax is not None else ax.get_ylim()[1]
            ax.text(
                d,
                y_text,
                f"H{h}",
                rotation=90,
                va="top",
                ha="right",
                fontsize=8,
                alpha=0.75,
            )


def _sparse_date_axis(ax, max_ticks: int = 8) -> None:
    """
    Make date labels readable.
    """
    locator = mdates.AutoDateLocator(minticks=3, maxticks=max_ticks)
    formatter = mdates.ConciseDateFormatter(locator)

    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)


def plot_sequential_tracker_results(
    result: Dict[str, Any],
    *,
    figdir: Path | str = "./fig_sequential_tracker",
    roll_window: int = 20,
    portfolio_label: str = "Tracker",
) -> None:
    """
    Create standard sequential hold-window plots.

    Plots:
      1. cumulative return overlay;
      2. cumulative active return;
      3. daily absolute active return in bp;
      4. rolling TE in bp.
    """
    figdir = Path(figdir)
    figdir.mkdir(parents=True, exist_ok=True)

    paths = result["paths"].copy()
    hold_starts = result["hold_start_dates"]

    dates = pd.DatetimeIndex(paths["date"])

    # -----------------------------------------------------------
    # 1) Cumulative return overlay
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8.0, 3.8))

    ax.plot(dates, paths["cum_index"], lw=1.2, label="Index")
    ax.plot(dates, paths["cum_tracker"], lw=1.1, label=portfolio_label)

    ymin, ymax = ax.get_ylim()
    _add_hold_start_lines(ax, hold_starts, ymin=ymin, ymax=ymax, label_holds=True)
    _sparse_date_axis(ax)

    ax.set_ylabel("Cumulative value")
    ax.set_title("Sequential holds: cumulative realised returns")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(figdir / "sequential_cumulative_returns.png", dpi=160)
    plt.close(fig)

    # -----------------------------------------------------------
    # 2) Cumulative active return
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8.0, 3.6))

    ax.plot(dates, paths["cum_active"], lw=1.1, label=f"{portfolio_label} - Index")

    ymin, ymax = ax.get_ylim()
    _add_hold_start_lines(ax, hold_starts, ymin=ymin, ymax=ymax, label_holds=True)
    _sparse_date_axis(ax)

    ax.axhline(0.0, lw=0.8)
    ax.set_ylabel("Cumulative active return")
    ax.set_title("Sequential holds: cumulative active return")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(figdir / "sequential_cumulative_active.png", dpi=160)
    plt.close(fig)

    # -----------------------------------------------------------
    # 3) Daily absolute active return in bp
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8.0, 3.6))

    ax.plot(dates, paths["abs_active_bp"], lw=0.85, label=portfolio_label)

    ymin, ymax = ax.get_ylim()
    _add_hold_start_lines(ax, hold_starts, ymin=ymin, ymax=ymax, label_holds=True)
    _sparse_date_axis(ax)

    ax.set_ylabel("Daily |active| (bp)")
    ax.set_title("Sequential holds: daily absolute active return")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(figdir / "sequential_daily_abs_active_bp.png", dpi=160)
    plt.close(fig)

    # -----------------------------------------------------------
    # 4) Rolling TE in bp
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8.0, 3.6))

    ax.plot(
        dates,
        paths["rolling_TE_bp"],
        lw=1.0,
        label=f"{portfolio_label}, {roll_window}d RMSE",
    )

    ymin, ymax = ax.get_ylim()
    _add_hold_start_lines(ax, hold_starts, ymin=ymin, ymax=ymax, label_holds=True)
    _sparse_date_axis(ax)

    ax.set_ylabel(f"{roll_window}-day TE RMSE (bp)")
    ax.set_title("Sequential holds: rolling realised tracking error")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(figdir / "sequential_rolling_TE_bp.png", dpi=160)
    plt.close(fig)

    print(f"[Sequential plots] Sequential Tracker figures saved to: {figdir.resolve()}")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — sequential experiment driver
# Sequential Tracker experiment driver
#
# Purpose:
#   Run the Tracker through H1, H2, ..., H_n sequentially.
#
# Logic:
#   - Hold 1 uses the initial locked Tracker from Block 11.
#   - Before Hold h >= 2, call rebalance_step_fn(...) to decide whether
#     to update w_old -> w_new.
#   - Reporting is on realised, uncentred hold-window returns only.
#
# Depends on the sequential plotting helpers:
#   - infer_asset_columns_from_df
#   - get_hold_dates_from_window
#   - extract_hold_data_from_window
#   - evaluate_tracker_on_hold
#   - plot_sequential_tracker_results
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, Callable, Optional, Sequence
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Rebalance-step callable contract
# ---------------------------------------------------------------

def describe_rebalance_step_contract() -> None:
    """
    Print the expected contract for the rebalance_step_fn hook.

    The actual rebalance_step_fn is where the cleaned Blocks 13A--13E
    will eventually be wrapped into one callable.
    """
    print(
        """
Expected rebalance_step_fn contract
-----------------------------------

rebalance_step_fn(
    hold_number: int,
    window: dict,
    previous_w: np.ndarray,
    experiment_state: dict,
) -> dict

It must return a dictionary containing at least:

    {
        "w_new": np.ndarray,          # portfolio to hold in this hold window
        "dw_impl": np.ndarray,        # implemented adjustment, same length as w_new
        "no_trade": bool,             # True if no rebalance
        "no_trade_reason": str,       # reason if no_trade=True
        "implementation_case": str,   # e.g. no_trade, approved_buys_and_sells_only

        "te_fit": {                   # optional but useful
            "new": float              # realised TE on the decision fit window
        },

        "implemented_trade_summary": {
            "turnover_half_L1": float,
            "nnz_trades": int,
            "n_increases": int,
            "n_decreases": int,
            "sum_dw": float,
        },
    }

For compatibility with the maintenance decision interface, the driver also accepts:

    "w_rebalanced" instead of "w_new",
    "te_fit2" instead of "te_fit".

The rebalance function is responsible for running the relevant
FIT-window Δw model, posterior diagnostics, ROPE rules, and locked
local-budget implementation rules.
"""
    )


# ---------------------------------------------------------------
# 1) Small local trade-summary helper
# ---------------------------------------------------------------

def trade_summary_from_delta_seq(
    dw: np.ndarray,
    *,
    tol: float = 1e-12,
) -> Dict[str, Any]:
    """
    Basic diagnostics for an implemented Δw.

    This local version avoids relying on the maintenance decision interface being present.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    return {
        "sum_dw": float(dw.sum()),
        "turnover_half_L1": float(0.5 * np.sum(np.abs(dw))),
        "L1_dw": float(np.sum(np.abs(dw))),
        "nnz_trades": int(np.count_nonzero(np.abs(dw) > tol)),
        "n_increases": int(np.count_nonzero(dw > tol)),
        "n_decreases": int(np.count_nonzero(dw < -tol)),
        "max_abs_dw": float(np.max(np.abs(dw))) if dw.size else 0.0,
    }


# ---------------------------------------------------------------
# 2) Normalise rolling windows
# ---------------------------------------------------------------

def normalise_rolling_windows(
    rolling_windows: Sequence[Dict[str, Any]],
    *,
    n_holds: int,
) -> list[Dict[str, Any]]:
    """
    Return the first n_holds rolling-window records.

    The main experiment always starts from Hold 1 and moves forward
    sequentially. It does not allow skipping ahead.
    """
    if rolling_windows is None:
        raise ValueError("rolling_windows must not be None.")

    windows = list(rolling_windows)

    if len(windows) < int(n_holds):
        raise ValueError(
            f"Requested n_holds={n_holds}, but rolling_windows has only "
            f"{len(windows)} entries."
        )

    return windows[:int(n_holds)]


# ---------------------------------------------------------------
# 3) Decision-row helper
# ---------------------------------------------------------------

def decision_row_from_rebalance_output(
    *,
    hold_number: int,
    decision: Optional[Dict[str, Any]],
    previous_w: np.ndarray,
    new_w: np.ndarray,
) -> Dict[str, Any]:
    """
    Create a one-row decision summary before a hold window.

    Hold 1 is treated as the initial portfolio rather than a rebalance.
    """
    previous_w = np.asarray(previous_w, dtype=float).reshape(-1)
    new_w = np.asarray(new_w, dtype=float).reshape(-1)

    if previous_w.shape != new_w.shape:
        raise ValueError("previous_w and new_w must have the same shape.")

    dw = new_w - previous_w

    if decision is None:
        return {
            "hold": int(hold_number),
            "decision_type": "initial portfolio",
            "no_trade": False,
            "no_trade_reason": "",
            "implementation_case": "initial portfolio",
            "turnover": 0.0,
            "n_trades": 0,
            "n_increases": 0,
            "n_decreases": 0,
            "sum_dw": 0.0,
            "max_abs_dw": 0.0,
            "candidate_name": "",
            "S_size": np.nan,
            "fit_TE_new": np.nan,
            "fit_TE_new_bp": np.nan,
        }

    trade_summary = decision.get("implemented_trade_summary", None)

    if not isinstance(trade_summary, dict) or len(trade_summary) == 0:
        trade_summary = trade_summary_from_delta_seq(dw)

    te_fit = decision.get("te_fit2", decision.get("te_fit", {}))

    if not isinstance(te_fit, dict):
        te_fit = {}

    fit_te_new = te_fit.get("new", np.nan)

    try:
        fit_te_new_float = float(fit_te_new)
    except Exception:
        fit_te_new_float = np.nan

    return {
        "hold": int(hold_number),
        "decision_type": "rebalance decision",
        "no_trade": bool(decision.get("no_trade", False)),
        "no_trade_reason": str(decision.get("no_trade_reason", "")),
        "implementation_case": str(decision.get("implementation_case", "")),
        "turnover": float(trade_summary.get("turnover_half_L1", 0.0)),
        "n_trades": int(trade_summary.get("nnz_trades", 0)),
        "n_increases": int(trade_summary.get("n_increases", 0)),
        "n_decreases": int(trade_summary.get("n_decreases", 0)),
        "sum_dw": float(trade_summary.get("sum_dw", np.sum(dw))),
        "max_abs_dw": float(
            trade_summary.get(
                "max_abs_dw",
                np.max(np.abs(dw)) if dw.size else 0.0,
            )
        ),
        "candidate_name": str(decision.get("candidate_name", "")),
        "S_size": int(decision.get("S_size", len(decision.get("S_idx", [])))),
        "fit_TE_new": fit_te_new_float,
        "fit_TE_new_bp": 1e4 * fit_te_new_float if np.isfinite(fit_te_new_float) else np.nan,
    }


# ---------------------------------------------------------------
# 4) Sequential driver
# ---------------------------------------------------------------

def run_sequential_tracker_experiment(
    *,
    n_holds: int = 4,
    df_ret: pd.DataFrame,
    rolling_windows: Sequence[Dict[str, Any]],
    initial_w: np.ndarray,
    idx_name: Optional[str] = None,
    asset_cols: Optional[Sequence[str]] = None,
    rebalance_step_fn: Optional[Callable[..., Dict[str, Any]]] = None,
    initial_fit_te_target: Optional[float] = None,
    roll_window: int = 20,
    figdir: Path | str = "./fig_sequential_tracker",
    portfolio_label: str = "Tracker",
    make_plots: bool = True,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run sequential hold-window evaluation for the Tracker.

    Parameters
    ----------
    n_holds : int
        Number of hold windows to evaluate, always starting from Hold 1.
    df_ret : pd.DataFrame
        Full returns dataframe, with asset columns and one index column.
    rolling_windows : sequence of dict
        Existing fit/hold window definitions.
    initial_w : np.ndarray
        Tracker entering Hold 1, usually w_locked_fit from Block 11.
    idx_name : Optional[str]
        Index-return column. If None, the last column of df_ret is used.
    asset_cols : Optional[Sequence[str]]
        Asset columns. If None, inferred from df_ret.
    rebalance_step_fn : Optional[callable]
        Function called before Hold h for h >= 2.
    initial_fit_te_target : Optional[float]
        FIT-window TE target attached to the initial Tracker.
    roll_window : int
        Rolling TE window length.
    figdir : Path | str
        Output directory for plots.
    portfolio_label : str
        Label to use in tables/plots. Default: "Tracker".
    make_plots : bool
        Whether to generate standard sequential plots.
    verbose : bool
        Whether to print summaries.

    Returns
    -------
    result : dict
        Contains hold summary, decision summary, paths, final weights,
        and per-hold weights/decisions.
    """
    if int(n_holds) < 1:
        raise ValueError("n_holds must be at least 1.")

    windows = normalise_rolling_windows(
        rolling_windows,
        n_holds=int(n_holds),
    )

    if asset_cols is None:
        inferred_asset_cols, idx_col = infer_asset_columns_from_df(
            df_ret,
            idx_name=idx_name,
        )
    else:
        inferred_asset_cols = [str(c) for c in asset_cols]

        if idx_name is not None:
            idx_col = str(idx_name)
        else:
            idx_col = str(list(df_ret.columns)[-1])

    p = len(inferred_asset_cols)
    w_current = np.asarray(initial_w, dtype=float).reshape(-1)

    if w_current.size != p:
        raise ValueError(
            f"initial_w has length {w_current.size}, but inferred p={p}."
        )

    if int(n_holds) > 1 and rebalance_step_fn is None:
        raise ValueError(
            "n_holds > 1 requires rebalance_step_fn. "
            "Call describe_rebalance_step_contract() for the expected interface."
        )

    hold_rows = []
    decision_rows = []
    path_frames = []
    hold_start_dates = []

    weights_by_hold: Dict[int, np.ndarray] = {}
    decisions_by_hold: Dict[int, Optional[Dict[str, Any]]] = {}

    experiment_state: Dict[str, Any] = {
        "df_ret": df_ret,
        "idx_col": idx_col,
        "asset_cols": inferred_asset_cols,
        "rolling_windows": windows,
        "portfolio_label": portfolio_label,
        "roll_window": int(roll_window),
    }

    for h, win in enumerate(windows, start=1):
        if verbose:
            hold_start, hold_end = get_hold_dates_from_window(win)
            print("\n" + "=" * 72)
            print(f"[Sequential Tracker] Hold {h}/{n_holds}")
            print(f"Hold dates: {hold_start.date()} -> {hold_end.date()}")

        # -------------------------------------------------------
        # Rebalance decision before Hold h
        # -------------------------------------------------------

        if h == 1:
            decision = None
            w_for_hold = w_current.copy()
            fit_te_target = initial_fit_te_target
            rebalance_label = "initial portfolio"

        else:
            decision = rebalance_step_fn(
                hold_number=h,
                window=win,
                previous_w=w_current.copy(),
                experiment_state=experiment_state,
            )

            if not isinstance(decision, dict):
                raise TypeError("rebalance_step_fn must return a dictionary.")

            if "w_new" in decision:
                w_for_hold = np.asarray(decision["w_new"], dtype=float).reshape(-1)
            elif "w_rebalanced" in decision:
                w_for_hold = np.asarray(decision["w_rebalanced"], dtype=float).reshape(-1)
            else:
                raise KeyError(
                    "rebalance_step_fn output must contain either 'w_new' "
                    "or 'w_rebalanced'."
                )

            if w_for_hold.size != p:
                raise ValueError(
                    f"rebalance_step_fn returned weights of length {w_for_hold.size}, "
                    f"but expected p={p}."
                )

            te_fit = decision.get("te_fit2", decision.get("te_fit", {}))

            if isinstance(te_fit, dict):
                fit_te_target = te_fit.get("new", np.nan)
            else:
                fit_te_target = np.nan

            rebalance_label = (
                "no rebalance"
                if bool(decision.get("no_trade", False))
                else "rebalance"
            )

        decision_row = decision_row_from_rebalance_output(
            hold_number=h,
            decision=decision,
            previous_w=w_current,
            new_w=w_for_hold,
        )

        decision_rows.append(decision_row)
        decisions_by_hold[h] = decision

        # The weight entering this hold is now fixed.
        w_current = w_for_hold.copy()
        weights_by_hold[h] = w_current.copy()

        # -------------------------------------------------------
        # Hold-window realised evaluation
        # -------------------------------------------------------

        hold_data = extract_hold_data_from_window(
            df_ret,
            win,
            asset_cols=inferred_asset_cols,
            idx_col=idx_col,
        )

        hold_start_dates.append(hold_data["dates"][0])

        hold_eval = evaluate_tracker_on_hold(
            hold_number=h,
            y_raw=hold_data["y_raw"],
            R_raw=hold_data["R_raw"],
            dates=hold_data["dates"],
            w=w_current,
            fit_te_target=fit_te_target,
            rebalance_label=rebalance_label,
            rebalance_decision=decision,
            roll_window=roll_window,
            portfolio_label=portfolio_label,
        )

        hold_rows.append(hold_eval["row"])
        path_frames.append(hold_eval["paths"])

        if verbose:
            row = hold_eval["row"]

            print(f"Decision entering hold: {rebalance_label}")

            if decision is not None and bool(decision.get("no_trade", False)):
                print(f"No-trade reason: {decision.get('no_trade_reason', '')}")

            print(
                f"TE(realised) = {row['TE_RMSE']:.6e} "
                f"({row['TE_bp']:.3f} bp)"
            )
            print(
                f"mean active = {row['mean_active_bp']:+.3f} bp | "
                f"std active = {row['std_active_bp']:.3f} bp | "
                f"MAE = {row['MAE_bp']:.3f} bp"
            )
            print(
                f"turnover entering hold = {row['turnover']:.6e} | "
                f"trades = {row['n_trades']}"
            )
            print(
                f"nnz = {row['nnz']} | shorts = {row['n_shorts']} | "
                f"sum(w) = {row['sum_weights']:+.12f}"
            )

    df_hold_summary = pd.DataFrame(hold_rows)
    df_decision_summary = pd.DataFrame(decision_rows)

    paths = pd.concat(path_frames, ignore_index=True)

    # Rebase cumulative paths across concatenated holds.
    paths["cum_index_global"] = np.cumprod(1.0 + paths["index_return"].to_numpy())
    paths["cum_tracker_global"] = np.cumprod(1.0 + paths["tracker_return"].to_numpy())
    paths["cum_active_global"] = np.cumsum(paths["active_return"].to_numpy())

    # Use global cumulative columns for plotting.
    paths["cum_index"] = paths["cum_index_global"]
    paths["cum_tracker"] = paths["cum_tracker_global"]
    paths["cum_active"] = paths["cum_active_global"]

    result = {
        "n_holds": int(n_holds),
        "df_hold_summary": df_hold_summary,
        "df_decision_summary": df_decision_summary,
        "paths": paths,
        "hold_start_dates": hold_start_dates,
        "weights_by_hold": weights_by_hold,
        "decisions_by_hold": decisions_by_hold,
        "final_w": w_current.copy(),
        "asset_cols": inferred_asset_cols,
        "idx_col": idx_col,
        "rolling_windows_used": windows,
        "portfolio_label": portfolio_label,
        "roll_window": int(roll_window),
    }

    if make_plots:
        plot_sequential_tracker_results(
            result,
            figdir=figdir,
            roll_window=roll_window,
            portfolio_label=portfolio_label,
        )

    if verbose:
        print("\n" + "=" * 72)
        print("[Sequential Tracker] Hold summary")

        display_cols = [
            "hold",
            "portfolio",
            "start",
            "end",
            "TE_bp",
            "MAE_bp",
            "mean_active_bp",
            "std_active_bp",
            "turnover",
            "n_trades",
            "nnz",
            "n_shorts",
            "hit_rate_vs_FIT_TE",
            "max_active_drawdown",
        ]

        display_cols = [c for c in display_cols if c in df_hold_summary.columns]

        print(
            df_hold_summary[display_cols].to_string(
                index=False,
                formatters={
                    "TE_bp": lambda x: f"{x:.3f}",
                    "MAE_bp": lambda x: f"{x:.3f}",
                    "mean_active_bp": lambda x: f"{x:+.3f}",
                    "std_active_bp": lambda x: f"{x:.3f}",
                    "turnover": lambda x: f"{x:.6e}",
                    "hit_rate_vs_FIT_TE": lambda x: "" if pd.isna(x) else f"{x:.3f}",
                    "max_active_drawdown": lambda x: f"{x:+.6e}",
                },
            )
        )

        print("\n[Sequential Tracker] Decision summary")

        display_dec_cols = [
            "hold",
            "decision_type",
            "no_trade",
            "implementation_case",
            "turnover",
            "n_trades",
            "candidate_name",
            "S_size",
            "fit_TE_new_bp",
            "no_trade_reason",
        ]

        display_dec_cols = [c for c in display_dec_cols if c in df_decision_summary.columns]

        print(
            df_decision_summary[display_dec_cols].to_string(
                index=False,
                formatters={
                    "turnover": lambda x: f"{x:.6e}",
                    "fit_TE_new_bp": lambda x: "" if pd.isna(x) else f"{x:.3f}",
                },
            )
        )

    return result


print("[Sequential driver] Sequential Tracker experiment driver defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — rebalancing context helpers
# Rebalancing wrapper interface: configuration + fit-window prep
#
# Purpose:
#   Prepare deterministic inputs for the sequential Δw rebalancing
#   wrapper.
#
# Important convention:
#   For Δw, use the old rebalancing-pipeline alpha convention:
#
#       alpha_dw = compute_alpha_scales(R_c)^2,
#       alpha_dw /= mean(alpha_dw).
#
#   This matches the previous standalone the Delta-w setup behaviour.
# ===============================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any, Optional, Sequence, Tuple
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Rebalancing wrapper configuration
# ---------------------------------------------------------------

@dataclass
class ASOCRebalanceConfig:
    """
    Configuration for the sequential ASOC rebalancing step.

    Current design choices:
      - sigma2_dw_base = 1e-4 is an effective rebalancing variance;
      - the outer c-grid is scored by a smooth FIT-TE improvement term
        and a soft upper-bound penalty on ||Delta w_MAP||_1;
      - ROPE/practical-change candidates choose possible movers;
      - one-directional buy-only candidates are implemented via a local
        funding rule using the smallest existing positive holdings;
      - untouched names are not globally renormalised.
    """
    sigma2_dw_base: float = 1e-4

    # Same c-grid as the full portfolio-fitting stage.
    c_grid_dw: tuple[float, ...] = (
        1.0, 10.0, 25.0, 50.0, 75.0, 100.0, 125.0, 150.0, 200.0
    )

    # Effective nonzero threshold for reporting Delta-w_MAP.
    # This is no longer the main c-grid score target.
    nnz_eff_thresh_dw: float = 1e-4

    # Legacy diagnostic target for effective adjustment size.
    # Kept for reporting, but not used in the new c-grid score.
    nnz_target_fraction_of_old: float = 0.25

    # Smooth Delta-w c-grid scoring.
    # w_TE = min(1, max((TE_old - TE_new)/TE_old, 0) / target)
    # w_L1 = exp(-0.5 * (max(||dw_MAP||_1 - l1_target, 0) / l1_scale)^2)
    score_te_improvement_target: float = 0.05
    score_l1_dw_target: float = 0.03
    score_l1_dw_scale: float = 0.02

    # ROPE candidate used for decision-making.
    selected_rope_candidate: str = "moderate_dir_eps1e-4_pi0.60"

    # Legacy field retained for old summaries/config compatibility.
    # The new buy-only local funding rule no longer rejects solely because
    # selected_min_confidence < funding_min_confidence.
    min_names_to_trade: int = 2
    funding_min_confidence: float = 0.80

    # Local implementation controls.
    max_buy_only_funding_names: int = 3
    close_small_funders: bool = True
    buy_only_dust_tol: float = 1e-5
    max_cardinality_change: int = 3
    allow_new_buy_names: bool = True

    # MALA / UQ defaults.
    n_burn_steps_dw: int = 50_000
    n_iter_long_dw: int = 250_000
    thin_long_dw: int = 6
    burn_kept_long_dw: int = 5_000
    max_lag_ess_dw: int = 200

    verbose: bool = True


asoc_rebalance_config = ASOCRebalanceConfig(
    sigma2_dw_base=SIGMA2_DW_BASE,
    c_grid_dw=tuple(C_GRID_DW),
    nnz_eff_thresh_dw=NNZ_EFF_THRESH_DW,
    score_te_improvement_target=SCORE_TE_IMPROVEMENT_TARGET,
    score_l1_dw_target=SCORE_L1_DW_TARGET,
    score_l1_dw_scale=SCORE_L1_DW_SCALE,
    selected_rope_candidate=SELECTED_ROPE_CANDIDATE,
    n_burn_steps_dw=DW_MALA_N_BURN_STEPS,
    n_iter_long_dw=DW_MALA_N_ITER_LONG,
    thin_long_dw=DW_MALA_THIN,
    burn_kept_long_dw=DW_MALA_BURN_KEPT,
    max_lag_ess_dw=DW_MALA_MAX_LAG_ESS,
    verbose=VERBOSE,
)

print("[Rebalance context] ASOCRebalanceConfig defined.")
print(asoc_rebalance_config)


# ---------------------------------------------------------------
# 1) Fit-window extraction
# ---------------------------------------------------------------

def extract_fit_data_from_window_for_rebalance(
    *,
    df_ret: pd.DataFrame,
    window: Dict[str, Any],
    asset_cols: Sequence[str],
    idx_col: str,
) -> Dict[str, Any]:
    """
    Extract realised/raw data for the fit window used to decide the
    rebalance before a hold.
    """
    fit_start, fit_end = get_fit_dates_from_window(window)

    out = extract_raw_returns_for_dates(
        df_ret,
        start=fit_start,
        end=fit_end,
        asset_cols=asset_cols,
        idx_col=idx_col,
    )

    out["fit_start_requested"] = fit_start
    out["fit_end_requested"] = fit_end

    return out


# ---------------------------------------------------------------
# 2) Centering helper
# ---------------------------------------------------------------

def center_fit_window_for_rebalance(
    *,
    y_raw: np.ndarray,
    R_raw: np.ndarray,
) -> Dict[str, Any]:
    """
    Center the fit-window index and asset returns.

    Returns
    -------
    dict with:
      y_c, R_c, y_mu, R_mu
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    R_raw = np.asarray(R_raw, dtype=float)

    if R_raw.ndim != 2:
        raise ValueError("R_raw must be 2D.")

    if R_raw.shape[0] != y_raw.size:
        raise ValueError("R_raw and y_raw have incompatible row counts.")

    y_mu = float(np.mean(y_raw))
    R_mu = np.mean(R_raw, axis=0)

    y_c = y_raw - y_mu
    R_c = R_raw - R_mu[None, :]

    return {
        "y_c": y_c,
        "R_c": R_c,
        "y_mu": y_mu,
        "R_mu": R_mu,
    }


# ---------------------------------------------------------------
# 3) Δw alpha scales: old standalone-pipeline convention
# ---------------------------------------------------------------

def compute_alpha_dw_scales_for_rebalance(
    R_c: np.ndarray,
    *,
    eps: float = 1e-12,
) -> np.ndarray:
    """
    Compute Δw alpha scales using the old rebalancing convention:

        alpha_base = compute_alpha_scales(R_c)
        alpha_dw   = alpha_base^2
        alpha_dw   = alpha_dw / mean(alpha_dw)

    This is intentionally different from the full-weight FIT-stage
    alpha convention. It matches the previous standalone Δw pipeline.
    """
    R_c = np.asarray(R_c, dtype=float)

    if R_c.ndim != 2:
        raise ValueError("R_c must be 2D.")

    if "compute_alpha_scales" in globals():
        alpha_base = np.asarray(compute_alpha_scales(R_c), dtype=float).reshape(-1)
    else:
        T = R_c.shape[0]
        alpha_base = np.linalg.norm(R_c, axis=0) / np.sqrt(max(T, 1))
        alpha_base = np.maximum(alpha_base, eps)
        alpha_base = alpha_base / np.mean(alpha_base)

    alpha_dw = alpha_base ** 2
    alpha_dw = np.maximum(alpha_dw, eps)
    alpha_dw = alpha_dw / np.mean(alpha_dw)

    return alpha_dw.astype(float)


# ---------------------------------------------------------------
# 4) FIT-window realised TE helper
# ---------------------------------------------------------------

def realised_te_for_rebalance_fit(
    *,
    y_fit: np.ndarray,
    R_fit: np.ndarray,
    w: np.ndarray,
) -> float:
    """
    Realised/raw TE on the rebalancing fit window.
    """
    return realised_te_rmse_raw(y_fit, R_fit, w)


# ---------------------------------------------------------------
# 5) Prepare one rebalance context
# ---------------------------------------------------------------

def prepare_rebalance_context_asoc(
    *,
    hold_number: int,
    window: Dict[str, Any],
    previous_w: np.ndarray,
    experiment_state: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
) -> Dict[str, Any]:
    """
    Prepare all deterministic inputs needed by the Δw rebalancing pipeline.

    This does not run SAPG/FISTA/MALA/ROPE. It only creates the clean
    handoff context for Blocks 14D-2 onward.
    """
    df_ret_local = experiment_state["df_ret"]
    idx_col = experiment_state["idx_col"]
    asset_cols = list(experiment_state["asset_cols"])

    w_old_local = np.asarray(previous_w, dtype=float).reshape(-1)
    p = w_old_local.size

    if len(asset_cols) != p:
        raise ValueError(
            f"previous_w has length {p}, but asset_cols has length {len(asset_cols)}."
        )

    fit_data = extract_fit_data_from_window_for_rebalance(
        df_ret=df_ret_local,
        window=window,
        asset_cols=asset_cols,
        idx_col=idx_col,
    )

    y_fit = np.asarray(fit_data["y_raw"], dtype=float).reshape(-1)
    R_fit = np.asarray(fit_data["R_raw"], dtype=float)

    cent = center_fit_window_for_rebalance(
        y_raw=y_fit,
        R_raw=R_fit,
    )

    y_c = cent["y_c"]
    R_c = cent["R_c"]
    y_mu = cent["y_mu"]
    R_mu = cent["R_mu"]

    alpha_dw = compute_alpha_dw_scales_for_rebalance(R_c)

    te_old_fit = realised_te_for_rebalance_fit(
        y_fit=y_fit,
        R_fit=R_fit,
        w=w_old_local,
    )

    nnz_old = int(np.count_nonzero(np.abs(w_old_local) > 1e-12))
    nnz_target_dw = max(2.0, config.nnz_target_fraction_of_old * float(nnz_old))

    context = {
        "hold_number": int(hold_number),
        "fit_start": fit_data["start"],
        "fit_end": fit_data["end"],
        "fit_n_obs": int(fit_data["n_obs"]),
        "asset_cols": asset_cols,
        "idx_col": idx_col,
        "p": p,

        # Raw fit-window objects.
        "y_fit": y_fit,
        "R_fit": R_fit,
        "dates_fit": fit_data["dates"],

        # Centered fit-window objects.
        "y_c": y_c,
        "R_c": R_c,
        "y_mu": y_mu,
        "R_mu": R_mu,

        # Previous held Tracker.
        "w_old": w_old_local,

        # Δw modelling inputs.
        "alpha_dw": alpha_dw,
        "sigma2_dw_base": float(config.sigma2_dw_base),
        "c_grid_dw": np.asarray(config.c_grid_dw, dtype=float),
        "nnz_eff_thresh_dw": float(config.nnz_eff_thresh_dw),
        "nnz_old": nnz_old,
        "nnz_target_dw": float(nnz_target_dw),

        # Diagnostics.
        "te_old_fit": float(te_old_fit),
    }

    if config.verbose:
        print("\n[Rebalance context | rebalance context]")
        print(f"  hold_number = {hold_number}")
        print(f"  fit dates   = {context['fit_start']} -> {context['fit_end']}")
        print(f"  fit shape   = R:{R_fit.shape}, y:{y_fit.shape}")
        print(f"  p           = {p}")
        print(f"  sum(w_old)  = {w_old_local.sum():+.12f}")
        print(f"  nnz(w_old)  = {nnz_old}/{p}")
        print(f"  TE_old fit  = {te_old_fit:.6e} ({1e4 * te_old_fit:.3f} bp)")
        print(f"  sigma2 base = {config.sigma2_dw_base:.3e}")
        print(f"  c_grid      = {np.asarray(config.c_grid_dw, dtype=float)}")
        print(
            "  alpha_dw    = "
            f"min {alpha_dw.min():.3f}, "
            f"median {np.median(alpha_dw):.3f}, "
            f"max {alpha_dw.max():.3f}, "
            f"mean {alpha_dw.mean():.3f}"
        )

    return context


print("[Rebalance context] Rebalance context helpers defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Delta-w grid engine
# Delta-w c-grid engine for the sequential rebalancing wrapper
#
# Purpose:
#   Given a rebalance context from the rebalancing context helper, run the Delta-w
#   effective-variance grid:
#
#       sigma2_dw(c) = c * sigma2_dw_base.
#
#   For each c:
#     - estimate kappa_EB(c) by a compact SAPG/MYULA routine;
#     - solve the SMOOTHED Delta-w MAP problem using the Moreau-envelope
#       objective from the old standalone rebalancing pipeline;
#     - score the candidate using realised FIT-window TE and
#       effective adjustment size.
#
# Important:
#   This block intentionally uses a smoothed Delta-w MAP solver, not the
#   exact nonsmooth FISTA solver. This matches the previous standalone
#   the Delta-w setup behaviour more closely.
#
# Output:
#   A Block-13A-style dictionary containing:
#     - df_dw_grid
#     - selected row information
#     - sigma2_dw_final
#     - kappa_EB_dw_final
#     - dw_steps_final
#     - dw_map_smoothed
#     - w_map_rebalanced_fit
# ===============================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any, Optional, Tuple

import time
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Step settings container
# ---------------------------------------------------------------

@dataclass
class DeltaWStepSettings:
    """
    Numerical step settings for the Delta-w objective.
    """
    L: float
    lambda_MY: float
    gamma_MYULA: float
    fista_step: float
    sigma2_used: float


# ---------------------------------------------------------------
# 1) Linear algebra helpers
# ---------------------------------------------------------------

def power_largest_eig_RtR_over_sigma2(
    R: np.ndarray,
    sigma2: float,
    *,
    max_iter: int = 200,
    tol: float = 1e-8,
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Estimate the largest eigenvalue of R^T R / sigma2 by power iteration.
    """
    R = np.asarray(R, dtype=float)

    if R.ndim != 2:
        raise ValueError("R must be a 2D array.")

    if sigma2 <= 0:
        raise ValueError("sigma2 must be positive.")

    rng = np.random.default_rng(0) if rng is None else rng

    p = R.shape[1]
    v = rng.normal(size=p)
    v = v / (np.linalg.norm(v) + 1e-15)

    prev = 0.0

    for _ in range(max_iter):
        Av = R.T @ (R @ v) / sigma2
        val = float(v @ Av)
        nrm = float(np.linalg.norm(Av))

        if nrm <= 1e-15:
            return 1.0

        v_new = Av / nrm

        if abs(val - prev) <= tol * max(1.0, abs(val)):
            return max(float(val), 1e-12)

        v = v_new
        prev = val

    return max(float(prev), 1e-12)


def make_delta_w_steps(
    R_c: np.ndarray,
    sigma2: float,
    *,
    rng: Optional[np.random.Generator] = None,
) -> DeltaWStepSettings:
    """
    Build numerical step settings for the Delta-w fit at a given sigma2.
    """
    L = power_largest_eig_RtR_over_sigma2(R_c, sigma2, rng=rng)

    lambda_MY = 1.0 / L
    gamma_MYULA = 0.9 / (2.0 * L)

    # For exact nonsmooth FISTA this would be 1/L.
    # The smoothed MAP solver below uses 1/(L + 1/lambda_MY).
    fista_step = 1.0 / L

    return DeltaWStepSettings(
        L=float(L),
        lambda_MY=float(lambda_MY),
        gamma_MYULA=float(gamma_MYULA),
        fista_step=float(fista_step),
        sigma2_used=float(sigma2),
    )


# ---------------------------------------------------------------
# 2) Weighted soft-thresholding with sum-zero constraint
# ---------------------------------------------------------------

def weighted_soft_threshold(
    x: np.ndarray,
    thresh: np.ndarray,
) -> np.ndarray:
    """
    Coordinatewise weighted soft-thresholding.
    """
    x = np.asarray(x, dtype=float)
    thresh = np.asarray(thresh, dtype=float)

    return np.sign(x) * np.maximum(np.abs(x) - thresh, 0.0)


def prox_sumzero_weighted_l1(
    v: np.ndarray,
    *,
    tau_kappa: float,
    alpha_dw: np.ndarray,
    target_sum: float = 0.0,
    max_iter: int = 100,
    tol: float = 1e-12,
) -> np.ndarray:
    """
    Proximal map of

        tau_kappa * sum_j alpha_j |u_j| + I{sum_j u_j = target_sum}

    at v.

    It solves

        argmin_u 0.5 ||u - v||^2 + tau_kappa sum_j alpha_j |u_j|
        subject to sum_j u_j = target_sum.

    The Lagrange multiplier is found by bisection.
    """
    v = np.asarray(v, dtype=float).reshape(-1)
    alpha_dw = np.asarray(alpha_dw, dtype=float).reshape(-1)

    if v.size != alpha_dw.size:
        raise ValueError("v and alpha_dw must have the same length.")

    if tau_kappa < 0:
        raise ValueError("tau_kappa must be non-negative.")

    thresh = tau_kappa * alpha_dw

    def prox_at_nu(nu: float) -> np.ndarray:
        return weighted_soft_threshold(v - nu, thresh)

    def sum_at_nu(nu: float) -> float:
        return float(np.sum(prox_at_nu(nu)))

    # Bracket root of sum(prox(v - nu)) - target_sum = 0.
    scale = float(np.max(np.abs(v)) + np.max(thresh) + abs(target_sum) + 1.0)
    lo = -scale
    hi = scale

    f_lo = sum_at_nu(lo) - target_sum
    f_hi = sum_at_nu(hi) - target_sum

    expand = 0
    while f_lo < 0.0 and expand < 50:
        lo *= 2.0
        f_lo = sum_at_nu(lo) - target_sum
        expand += 1

    expand = 0
    while f_hi > 0.0 and expand < 50:
        hi *= 2.0
        f_hi = sum_at_nu(hi) - target_sum
        expand += 1

    if not (f_lo >= 0.0 and f_hi <= 0.0):
        # Fallback: unconstrained prox plus uniform correction.
        u = weighted_soft_threshold(v, thresh)
        u = u + (target_sum - float(np.sum(u))) / max(u.size, 1)
        return u

    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        f_mid = sum_at_nu(mid) - target_sum

        if abs(f_mid) <= tol:
            return prox_at_nu(mid)

        if f_mid > 0.0:
            lo = mid
        else:
            hi = mid

    return prox_at_nu(0.5 * (lo + hi))


# Compatibility alias for older blocks, if not already defined.
if "prox_sumzero_l1" not in globals():
    def prox_sumzero_l1(
        x: np.ndarray,
        *,
        tau_kappa: float,
        alpha_dw: np.ndarray,
    ) -> np.ndarray:
        """
        Backwards-compatible alias used by older Delta-w blocks.
        """
        return prox_sumzero_weighted_l1(
            x,
            tau_kappa=tau_kappa,
            alpha_dw=alpha_dw,
            target_sum=0.0,
        )


# ---------------------------------------------------------------
# 3) Projection and objective helpers
# ---------------------------------------------------------------

def project_sumzero_asoc(v: np.ndarray) -> np.ndarray:
    """
    Orthogonal projection onto {1^T v = 0}.
    """
    v = np.asarray(v, dtype=float).reshape(-1)

    return v - float(np.mean(v))


def delta_w_objective_exact(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    sigma2: float,
    kappa: float,
    alpha_dw: np.ndarray,
) -> float:
    """
    Exact nonsmooth Delta-w objective, kept for diagnostics only:

        0.5/sigma2 || y_c - R_c (w_old + dw) ||^2
        + kappa sum_j alpha_j |dw_j|.

    The smoothed MAP solver below does not minimise this objective.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    alpha_dw = np.asarray(alpha_dw, dtype=float).reshape(-1)

    resid = y_c - R_c @ (w_old + dw)
    data_term = 0.5 / sigma2 * float(resid @ resid)
    penalty = kappa * float(np.sum(alpha_dw * np.abs(dw)))

    return float(data_term + penalty)


def grad_delta_w_data_term(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    sigma2: float,
) -> np.ndarray:
    """
    Gradient of the smooth data-fidelity term with respect to Delta-w.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)
    resid = R_c @ (w_old + dw) - y_c

    return (R_c.T @ resid) / sigma2


def delta_w_phi_smoothed_asoc(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    sigma2: float,
    kappa: float,
    lambda_MY: float,
) -> tuple[float, float]:
    """
    Smoothed Delta-w objective:

        Phi_sm(dw)
          = 0.5/sigma2 || y_c - R_c (w_old + dw) ||^2
            + g^lambda(dw),

    where

        g(u) = kappa sum_j alpha_j |u_j| + I{1^T u = 0}(u),

    and

        g^lambda(dw)
          = 0.5/lambda ||dw - prox_{lambda g}(dw)||^2
            + kappa sum_j alpha_j |prox_{lambda g}(dw)_j|.

    Returns
    -------
    Phi : float
        Smoothed objective value.
    S_val : float
        Weighted l1 statistic at the prox point.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    resid = y_c - R_c @ (w_old + dw)
    data_term = 0.5 / float(sigma2) * float(resid @ resid)

    u = prox_sumzero_weighted_l1(
        dw,
        tau_kappa=float(lambda_MY) * float(kappa),
        alpha_dw=alpha_dw,
        target_sum=0.0,
    )

    diff = dw - u
    S_val = float(np.sum(alpha_dw * np.abs(u)))

    g_sm = (
        0.5 / float(lambda_MY) * float(diff @ diff)
        + float(kappa) * S_val
    )

    return float(data_term + g_sm), S_val


def grad_delta_w_phi_smoothed_asoc(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    sigma2: float,
    kappa: float,
    lambda_MY: float,
) -> np.ndarray:
    """
    Gradient of the smoothed Delta-w objective, projected onto the
    sum-zero subspace.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    resid = R_c @ (w_old + dw) - y_c
    grad_f_full = R_c.T @ resid / float(sigma2)
    grad_f = project_sumzero_asoc(grad_f_full)

    u = prox_sumzero_weighted_l1(
        dw,
        tau_kappa=float(lambda_MY) * float(kappa),
        alpha_dw=alpha_dw,
        target_sum=0.0,
    )

    grad_g_sm = (dw - u) / float(lambda_MY)

    return project_sumzero_asoc(grad_f + grad_g_sm)


# ---------------------------------------------------------------
# 4) Smoothed Delta-w MAP solver
# ---------------------------------------------------------------

def fista_delta_w_map_sumzero(
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    sigma2: float,
    kappa: float,
    steps: DeltaWStepSettings,
    x0: Optional[np.ndarray] = None,
    max_iter: int = 4_000,
    tol: float = 1e-6,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Smoothed Delta-w MAP solver.

    This keeps the old function name because run_delta_w_cgrid_asoc(...)
    calls fista_delta_w_map_sumzero(...), but the objective solved here is
    the smoothed Moreau-envelope objective used in the old standalone
    rebalancing pipeline:

        min_dw Phi_sm(dw),  subject to 1^T dw = 0.

    The step size is

        1 / (L_data + 1/lambda_MY).

    Parameters
    ----------
    x0 : Optional[np.ndarray]
        Optional warm start. If supplied, it is projected to sum zero.
    """
    p = R_c.shape[1]

    if x0 is None:
        dw = np.zeros(p, dtype=float)
    else:
        dw = project_sumzero_asoc(np.asarray(x0, dtype=float).reshape(-1))

    z = dw.copy()
    t = 1.0

    L_data = float(steps.L)
    lambda_MY = float(steps.lambda_MY)

    L_smooth = L_data + 1.0 / lambda_MY
    step = 1.0 / L_smooth

    obj_trace = []
    S_trace = []
    sum_trace = []

    t0 = time.perf_counter()
    k_last = int(max_iter)

    for k in range(1, int(max_iter) + 1):
        grad = grad_delta_w_phi_smoothed_asoc(
            z,
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            sigma2=sigma2,
            kappa=kappa,
            lambda_MY=lambda_MY,
        )

        dw_next = project_sumzero_asoc(z - step * grad)

        t_next = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t * t))
        z = project_sumzero_asoc(
            dw_next + ((t - 1.0) / t_next) * (dw_next - dw)
        )

        Phi, S_val = delta_w_phi_smoothed_asoc(
            dw_next,
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            sigma2=sigma2,
            kappa=kappa,
            lambda_MY=lambda_MY,
        )

        obj_trace.append(Phi)
        S_trace.append(S_val)
        sum_trace.append(float(dw_next.sum()))

        if k > 1:
            diff = abs(obj_trace[-2] - obj_trace[-1])

            if diff <= tol * max(1.0, abs(obj_trace[-1])):
                k_last = k
                dw = dw_next
                t = t_next
                break

        dw = dw_next
        t = t_next

        if verbose and (k % 500 == 0):
            print(
                f"    [Delta-w smoothed MAP] iter={k:5d}, "
                f"Phi={Phi:.6e}, S={S_val:.3e}, sum(dw)={dw.sum():+.3e}"
            )

    wall = time.perf_counter() - t0

    trace = {
        "obj": np.asarray(obj_trace, dtype=float),
        "S": np.asarray(S_trace, dtype=float),
        "sum_dw": np.asarray(sum_trace, dtype=float),
        "wall_seconds": float(wall),
        "iters": int(k_last),
        "step": float(step),
        "L_smooth": float(L_smooth),
    }

    return {
        "dw_map": dw,
        "n_iter": int(k_last),
        "obj_trace": np.asarray(obj_trace, dtype=float),
        "obj_final": float(obj_trace[-1]) if len(obj_trace) else np.nan,
        "fista_trace": trace,
    }


# ---------------------------------------------------------------
# 5) Compact SAPG/MYULA estimator for kappa
# ---------------------------------------------------------------

def myula_step_delta_w_smoothed(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    sigma2: float,
    kappa: float,
    steps: DeltaWStepSettings,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    One MYULA-style step for the smoothed Delta-w posterior.

    This is used only inside SAPG to estimate kappa conditional on sigma2.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    grad_f = grad_delta_w_data_term(
        dw,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        sigma2=sigma2,
    )

    prox = prox_sumzero_weighted_l1(
        dw,
        tau_kappa=steps.lambda_MY * kappa,
        alpha_dw=alpha_dw,
        target_sum=0.0,
    )

    grad_moreau = (dw - prox) / steps.lambda_MY

    gamma = float(steps.gamma_MYULA)

    dw_new = (
        dw
        - gamma * (grad_f + grad_moreau)
        + np.sqrt(2.0 * gamma) * rng.normal(size=dw.shape)
    )

    return dw_new


def sapg_estimate_kappa_delta_w(
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    sigma2: float,
    steps: DeltaWStepSettings,
    kappa0: float,
    kappa_bounds: Tuple[float, float],
    n_iter: int = 15_000,
    burn_in: Optional[int] = None,
    q_weight: float = 1.0,
    rng: Optional[np.random.Generator] = None,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Compact SAPG estimator for the Delta-w Laplace rate kappa.

    The score uses the positive-homogeneous form

        dim - kappa * sum_j alpha_j |prox_j|,

    with dim = p - 1 because Delta-w is sum-zero.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    p = R_c.shape[1]
    dim = max(p - 1, 1)

    kappa_min, kappa_max = map(float, kappa_bounds)

    if not (0.0 < kappa_min < kappa_max):
        raise ValueError("Require 0 < kappa_min < kappa_max.")

    eta = float(np.log(np.clip(kappa0, kappa_min, kappa_max)))
    eta_min = float(np.log(kappa_min))
    eta_max = float(np.log(kappa_max))

    dw = np.zeros(p, dtype=float)

    if burn_in is None:
        burn_in = max(1_000, n_iter // 2)

    eta_trace = np.empty(n_iter, dtype=float)
    score_trace = np.empty(n_iter, dtype=float)
    S_trace = np.empty(n_iter, dtype=float)

    pr_num = 0.0
    pr_den = 0.0

    for k in range(1, n_iter + 1):
        kappa = float(np.exp(eta))

        dw = myula_step_delta_w_smoothed(
            dw,
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            sigma2=sigma2,
            kappa=kappa,
            steps=steps,
            rng=rng,
        )

        prox = prox_sumzero_weighted_l1(
            dw,
            tau_kappa=steps.lambda_MY * kappa,
            alpha_dw=alpha_dw,
            target_sum=0.0,
        )

        S_val = float(np.sum(alpha_dw * np.abs(prox)))
        score = float(dim - kappa * S_val)

        # Robbins-Monro step on log-kappa.
        rho = 0.8 / ((50.0 + k) ** 0.7)
        eta = float(np.clip(eta + rho * score, eta_min, eta_max))

        eta_trace[k - 1] = eta
        score_trace[k - 1] = score
        S_trace[k - 1] = S_val

        if k > burn_in:
            weight = float((k - burn_in) ** q_weight)
            pr_num += weight * eta
            pr_den += weight

        if verbose and (k % 5_000 == 0):
            print(
                f"    [SAPG Delta-w] iter={k:6d}, "
                f"kappa={np.exp(eta):.3e}, score={score:+.3e}, S={S_val:.3e}"
            )

    if pr_den > 0.0:
        eta_pr = pr_num / pr_den
    else:
        eta_pr = float(np.mean(eta_trace))

    kappa_EB = float(np.exp(np.clip(eta_pr, eta_min, eta_max)))

    return {
        "kappa_EB": kappa_EB,
        "eta_trace": eta_trace,
        "score_trace": score_trace,
        "S_trace": S_trace,
        "kappa_bounds": (kappa_min, kappa_max),
        "kappa0": float(kappa0),
        "hit_lower": bool(np.isclose(kappa_EB, kappa_min)),
        "hit_upper": bool(np.isclose(kappa_EB, kappa_max)),
    }


# ---------------------------------------------------------------
# 6) Delta-w LS reference and kappa initialisation
# ---------------------------------------------------------------

def sumzero_ls_delta_reference(
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    ridge: float = 1e-10,
) -> Dict[str, Any]:
    """
    Compute a diagnostic constrained LS Delta-w reference with sum(Delta-w)=0.

    This is used only to initialise kappa, not as a variance estimator.
    """
    y_res = (
        np.asarray(y_c, dtype=float).reshape(-1)
        - R_c @ np.asarray(w_old, dtype=float).reshape(-1)
    )

    p = R_c.shape[1]
    A = R_c.T @ R_c + ridge * np.eye(p)
    b = R_c.T @ y_res

    # Solve:
    #   A d + lambda 1 = b
    #   1^T d = 0
    KKT = np.block(
        [
            [A, np.ones((p, 1))],
            [np.ones((1, p)), np.zeros((1, 1))],
        ]
    )
    rhs = np.concatenate([b, np.array([0.0])])

    sol = np.linalg.solve(KKT, rhs)
    dw_ls = sol[:p]

    resid = y_res - R_c @ dw_ls
    mse = float(np.mean(resid ** 2))

    return {
        "dw_ls": dw_ls,
        "mse": mse,
        "sum_dw_ls": float(dw_ls.sum()),
        "min_abs_dw_ls": float(np.min(np.abs(dw_ls))),
        "max_abs_dw_ls": float(np.max(np.abs(dw_ls))),
    }


def initialise_kappa_from_delta_ls(
    *,
    dw_ls: np.ndarray,
    alpha_dw: np.ndarray,
    bound_factor: float = 10.0,
) -> Dict[str, Any]:
    """
    Initial kappa heuristic:

        kappa0 = (p - 1) / sum_j alpha_j |Delta-w_LS,j|.
    """
    dw_ls = np.asarray(dw_ls, dtype=float).reshape(-1)
    alpha_dw = np.asarray(alpha_dw, dtype=float).reshape(-1)

    p = dw_ls.size
    dim = max(p - 1, 1)

    S_LS = float(np.sum(alpha_dw * np.abs(dw_ls)))
    kappa0 = float(dim / max(S_LS, 1e-12))

    bounds = (
        kappa0 / float(bound_factor),
        kappa0 * float(bound_factor),
    )

    return {
        "S_LS": S_LS,
        "kappa0": kappa0,
        "kappa_bounds": bounds,
    }


# ---------------------------------------------------------------
# 7) Main Delta-w c-grid runner
# ---------------------------------------------------------------

def run_delta_w_cgrid_asoc(
    *,
    context: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
    sapg_n_iter: int = 15_000,
    fista_max_iter: int = 4_000,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, Any]:
    """
    Run the Delta-w c-grid for one sequential rebalance context.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    y_c = np.asarray(context["y_c"], dtype=float).reshape(-1)
    R_c = np.asarray(context["R_c"], dtype=float)
    y_fit = np.asarray(context["y_fit"], dtype=float).reshape(-1)
    R_fit = np.asarray(context["R_fit"], dtype=float)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    alpha_dw = np.asarray(context["alpha_dw"], dtype=float).reshape(-1)

    sigma2_base = float(context["sigma2_dw_base"])
    c_grid = np.asarray(context["c_grid_dw"], dtype=float)
    nnz_eff_thresh = float(context["nnz_eff_thresh_dw"])
    te_old_fit = float(context["te_old_fit"])
    nnz_target = float(context["nnz_target_dw"])

    score_te_target = max(float(config.score_te_improvement_target), 1e-12)
    score_l1_target = max(float(config.score_l1_dw_target), 0.0)
    score_l1_scale = max(float(config.score_l1_dw_scale), 1e-12)

    p = R_c.shape[1]

    if config.verbose:
        print("\n[Delta-w grid | Delta-w c-grid settings]")
        print(f"  sigma2_dw_base = {sigma2_base:.3e}")
        print(f"  c_grid_dw = {c_grid}")
        print(f"  SAPG iterations per c = {sapg_n_iter}")
        print(f"  smoothed MAP max_iter per c = {fista_max_iter}")
        print(f"  TE_old(realised fit) = {te_old_fit:.6e} ({1e4 * te_old_fit:.3f} bp)")
        print(f"  old nnz = {context['nnz_old']}")
        print(f"  target effective Delta-w nnz = {nnz_target:.2f} (diagnostic only)")
        print(
            "  score = w_TE * w_L1, with "
            f"TE target={score_te_target:.3f}, "
            f"L1 target={score_l1_target:.3e}, "
            f"L1 scale={score_l1_scale:.3e}"
        )

    # Diagnostic LS reference and kappa initialisation.
    ls_ref = sumzero_ls_delta_reference(
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
    )

    kappa_init = initialise_kappa_from_delta_ls(
        dw_ls=ls_ref["dw_ls"],
        alpha_dw=alpha_dw,
        bound_factor=10.0,
    )

    if config.verbose:
        print("\n[Delta-w grid | Delta-w LS diagnostic]")
        print(f"  sum(dw_LS) = {ls_ref['sum_dw_ls']:+.3e}")
        print(f"  diagnostic MSE residual variance = {ls_ref['mse']:.3e}")
        print(f"  sigma2_dw_base USED = {sigma2_base:.3e}")
        print("  note: diagnostic LS variance is not used as rebalancing variance.")
        print(
            f"  Delta-w LS reference: "
            f"min|dw_LS|={ls_ref['min_abs_dw_ls']:.3e}, "
            f"max|dw_LS|={ls_ref['max_abs_dw_ls']:.3e}"
        )
        print("\n[Delta-w grid | kappa initialisation]")
        print(f"  S_LS = {kappa_init['S_LS']:.3e}")
        print(f"  kappa0 = {kappa_init['kappa0']:.3e}")
        print(
            "  kappa bounds = "
            f"[{kappa_init['kappa_bounds'][0]:.3e}, "
            f"{kappa_init['kappa_bounds'][1]:.3e}]"
        )

    rows = []
    per_c_outputs: Dict[float, Dict[str, Any]] = {}

    # Warm start across c-grid rows.
    x0_map = np.zeros(p, dtype=float)

    if config.verbose:
        print("\n[Delta-w grid | running Delta-w c-grid]")

    for c in c_grid:
        c = float(c)
        sigma2 = c * sigma2_base

        steps = make_delta_w_steps(R_c, sigma2, rng=rng)

        sapg_out = sapg_estimate_kappa_delta_w(
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            sigma2=sigma2,
            steps=steps,
            kappa0=kappa_init["kappa0"],
            kappa_bounds=kappa_init["kappa_bounds"],
            n_iter=sapg_n_iter,
            rng=rng,
            verbose=False,
        )

        kappa_EB = float(sapg_out["kappa_EB"])

        map_out = fista_delta_w_map_sumzero(
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            sigma2=sigma2,
            kappa=kappa_EB,
            steps=steps,
            x0=x0_map,
            max_iter=fista_max_iter,
            verbose=False,
        )

        dw_map = np.asarray(map_out["dw_map"], dtype=float).reshape(-1)
        x0_map = dw_map.copy()

        w_new_map = w_old + dw_map

        te_new_fit = realised_te_rmse_raw(y_fit, R_fit, w_new_map)

        nnz_raw = int(np.count_nonzero(dw_map))
        nnz_eff = int(np.count_nonzero(np.abs(dw_map) >= nnz_eff_thresh))
        L1_dw = float(np.sum(np.abs(dw_map)))

        # Smooth realised-TE improvement score.
        # Positive relative improvement receives credit up to score_te_target;
        # worsening proposals receive zero TE credit.
        if te_old_fit > 0.0 and np.isfinite(te_old_fit) and np.isfinite(te_new_fit):
            TE_rel_improvement = float((te_old_fit - te_new_fit) / te_old_fit)
        else:
            TE_rel_improvement = 0.0

        w_TE = float(
            min(
                1.0,
                max(TE_rel_improvement, 0.0) / score_te_target,
            )
        )

        # Soft upper-bound penalty on gross MAP movement.
        # Since one-way turnover is roughly 0.5 * ||Delta w||_1, this
        # discourages aggressive proposal landscapes without forcing MAP
        # sparsity via an effective-nnz target.
        L1_excess = max(L1_dw - score_l1_target, 0.0)
        w_L1_dw = float(np.exp(-0.5 * (L1_excess / score_l1_scale) ** 2))

        score = float(w_TE * w_L1_dw)

        row = {
            "c": c,
            "sigma2_dw": sigma2,
            "kappa_EB": kappa_EB,
            "TE_fit": te_new_fit,
            "TE_fit_bp": 1e4 * te_new_fit,
            "TE_old_fit": te_old_fit,
            "TE_old_fit_bp": 1e4 * te_old_fit,
            "nnz_raw": nnz_raw,
            "nnz_eff": nnz_eff,
            "L1_dw": L1_dw,
            "sum_dw": float(dw_map.sum()),
            "max_abs_dw": float(np.max(np.abs(dw_map))) if dw_map.size else 0.0,
            "TE_rel_improvement": TE_rel_improvement,
            "score_te_target": score_te_target,
            "w_TE": w_TE,
            "score_l1_target": score_l1_target,
            "score_l1_scale": score_l1_scale,
            "w_L1_dw": w_L1_dw,
            "nnz_target_diagnostic": nnz_target,
            "score": score,
            "map_iter": int(map_out["n_iter"]),
            "L": float(steps.L),
            "lambda_MY": float(steps.lambda_MY),
            "gamma_MYULA": float(steps.gamma_MYULA),
            "sapg_hit_lower": bool(sapg_out["hit_lower"]),
            "sapg_hit_upper": bool(sapg_out["hit_upper"]),
        }

        rows.append(row)

        per_c_outputs[c] = {
            "row": row,
            "steps": steps,
            "sapg_out": sapg_out,
            "map_out": map_out,
            "fista_out": map_out,  # legacy alias
            "dw_map": dw_map,
            "w_new_map": w_new_map,
        }

        if config.verbose:
            print(
                f"[Delta-w grid] c={c:7.2f} | "
                f"sigma2={sigma2:.3e} | "
                f"kappa_EB={kappa_EB:.3e} | "
                f"TE_FIT={te_new_fit:.6e} | "
                f"nnz_raw={nnz_raw:4d} | "
                f"nnz_eff={nnz_eff:4d} | "
                f"L1={L1_dw:.3e} | "
                f"TE_imp={TE_rel_improvement:+.3f} | "
                f"w_TE={w_TE:.3f} | "
                f"w_L1={w_L1_dw:.3f} | "
                f"score={score:.3f}"
            )

    df_grid = pd.DataFrame(rows)

    # Select highest score. If tied, choose the smallest c.
    df_grid_sorted = df_grid.sort_values(
        ["score", "c"],
        ascending=[False, True],
    ).reset_index(drop=True)

    selected = df_grid_sorted.iloc[0].to_dict()
    c_star = float(selected["c"])

    selected_obj = per_c_outputs[c_star]
    dw_star = np.asarray(selected_obj["dw_map"], dtype=float).reshape(-1)
    w_star = np.asarray(selected_obj["w_new_map"], dtype=float).reshape(-1)
    steps_star = selected_obj["steps"]

    out = {
        "context": context,
        "config": config,
        "ls_ref": ls_ref,
        "kappa_init": kappa_init,
        "df_dw_grid": df_grid,
        "per_c_outputs": per_c_outputs,
        "selected_row": selected,
        "c_star_dw": c_star,
        "sigma2_dw_final": float(selected["sigma2_dw"]),
        "kappa_EB_dw_final": float(selected["kappa_EB"]),
        "dw_steps_final": steps_star,
        "dw_map_smoothed": dw_star,
        "dw_map": dw_star,
        "w_map_rebalanced_fit": w_star,
        "te_old_fit": te_old_fit,
        "te_new_fit": float(selected["TE_fit"]),
    }

    if config.verbose:
        print("\n[Delta-w grid | selected Delta-w setting]")
        print(f"  c_star = {out['c_star_dw']:.2f}")
        print(f"  sigma2_dw_final = {out['sigma2_dw_final']:.6e}")
        print(f"  kappa_EB_dw_final = {out['kappa_EB_dw_final']:.6e}")
        print(f"  TE_old fit = {out['te_old_fit']:.6e}")
        print(f"  TE_new fit = {out['te_new_fit']:.6e}")
        print(f"  nnz_eff = {int(selected['nnz_eff'])}")
        print(f"  ||dw_MAP||_1 = {float(selected['L1_dw']):.6e}")
        print(f"  TE rel improvement = {float(selected['TE_rel_improvement']):+.3f}")
        print(f"  w_TE = {float(selected['w_TE']):.3f}")
        print(f"  w_L1_dw = {float(selected['w_L1_dw']):.3f}")
        print(f"  max |dw_MAP| = {float(selected['max_abs_dw']):.6e}")
        print(f"  sum(dw_MAP) = {float(selected['sum_dw']):+.3e}")

    return out


print("[Delta-w grid] Delta-w c-grid engine with smoothed MAP solver defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Delta-w MALA helper functions
# Delta-w MALA/UQ helpers for the sequential rebalancing wrapper
#
# Purpose:
#   Define the smoothed Delta-w posterior and preconditioned MALA
#   machinery used after the c-grid has selected sigma2 and kappa.
#
# Depends on the Delta-w grid engine:
#   - DeltaWStepSettings
#   - prox_sumzero_weighted_l1
#   - realised_te_rmse_raw / realised_te_for_rebalance_fit
# ===============================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Any, Optional
import time
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) MALA settings
# ---------------------------------------------------------------

@dataclass
class DeltaWMalaSettings:
    """
    Settings for the smoothed Delta-w MALA target.
    """
    sigma2: float
    lambda_MY: float
    kappa: float
    P: np.ndarray


# ---------------------------------------------------------------
# 1) Diagonal Jacobi preconditioner
# ---------------------------------------------------------------

def build_diag_preconditioner_delta_w(
    R_c: np.ndarray,
    sigma2: float,
    *,
    eps: float = 1e-10,
) -> np.ndarray:
    """
    Diagonal Jacobi preconditioner for the Delta-w MALA run.

    Uses the diagonal of R_c^T R_c / sigma2:
        P_j = 1 / sqrt(diag_j + eps).
    """
    R_c = np.asarray(R_c, dtype=float)

    diag_h = np.sum(R_c * R_c, axis=0) / float(sigma2)
    diag_h = np.maximum(diag_h, eps)

    return (1.0 / np.sqrt(diag_h)).astype(float)


def precond_hessian_matvec_delta_w(
    v: np.ndarray,
    *,
    R_c: np.ndarray,
    sigma2: float,
    P: np.ndarray,
) -> np.ndarray:
    """
    Matvec for

        P (R_c^T R_c / sigma2) P v.
    """
    v = np.asarray(v, dtype=float).reshape(-1)
    P = np.asarray(P, dtype=float).reshape(-1)

    Pv = P * v
    return P * (R_c.T @ (R_c @ Pv) / float(sigma2))


def power_largest_eig_precond_delta_w(
    R_c: np.ndarray,
    sigma2: float,
    P: np.ndarray,
    *,
    max_iter: int = 200,
    tol: float = 1e-8,
    rng: Optional[np.random.Generator] = None,
) -> float:
    """
    Estimate largest eigenvalue of the preconditioned data Hessian:

        P (R_c^T R_c / sigma2) P.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    p = R_c.shape[1]
    v = rng.normal(size=p)
    v = v / (np.linalg.norm(v) + 1e-15)

    prev = 0.0

    for _ in range(max_iter):
        Av = precond_hessian_matvec_delta_w(
            v,
            R_c=R_c,
            sigma2=sigma2,
            P=P,
        )

        val = float(v @ Av)
        nrm = float(np.linalg.norm(Av))

        if nrm <= 1e-15:
            return 1.0

        v_next = Av / nrm

        if abs(val - prev) <= tol * max(1.0, abs(val)):
            return max(float(val), 1e-12)

        v = v_next
        prev = val

    return max(float(prev), 1e-12)


# ---------------------------------------------------------------
# 2) Smoothed Delta-w target
# ---------------------------------------------------------------

def logpost_smoothed_delta_w(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
) -> float:
    """
    Log posterior of the smoothed Delta-w target, up to a constant.

    The energy is

        Phi_sm(dw)
          = 0.5 / sigma2 || y_c - R_c (w_old + dw) ||^2
            + g^lambda(dw),

    where

        g(u) = kappa sum_j alpha_j |u_j| + I{1^T u = 0}(u),

    and

        g^lambda(dw)
          = g(prox_{lambda g}(dw))
            + 0.5 / lambda ||dw - prox_{lambda g}(dw)||^2.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)
    y_c = np.asarray(y_c, dtype=float).reshape(-1)
    R_c = np.asarray(R_c, dtype=float)
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    alpha_dw = np.asarray(alpha_dw, dtype=float).reshape(-1)

    sigma2 = float(settings.sigma2)
    lam = float(settings.lambda_MY)
    kappa = float(settings.kappa)

    resid = y_c - R_c @ (w_old + dw)
    data_term = 0.5 / sigma2 * float(resid @ resid)

    prox = prox_sumzero_weighted_l1(
        dw,
        tau_kappa=lam * kappa,
        alpha_dw=alpha_dw,
        target_sum=0.0,
    )

    diff = dw - prox

    g_smooth = (
        0.5 / lam * float(diff @ diff)
        + kappa * float(np.sum(alpha_dw * np.abs(prox)))
    )

    return -float(data_term + g_smooth)


def grad_phi_smoothed_delta_w(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
) -> np.ndarray:
    """
    Gradient of the smoothed Delta-w energy Phi_sm.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    sigma2 = float(settings.sigma2)
    lam = float(settings.lambda_MY)
    kappa = float(settings.kappa)

    resid = R_c @ (w_old + dw) - y_c
    grad_f = R_c.T @ resid / sigma2

    prox = prox_sumzero_weighted_l1(
        dw,
        tau_kappa=lam * kappa,
        alpha_dw=alpha_dw,
        target_sum=0.0,
    )

    grad_g_sm = (dw - prox) / lam

    return grad_f + grad_g_sm


# ---------------------------------------------------------------
# 3) One preconditioned MALA step
# ---------------------------------------------------------------

def delta_w_mala_step(
    dw: np.ndarray,
    *,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
    gamma: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, bool, float]:
    """
    One preconditioned MALA step for the smoothed Delta-w posterior:

        dw' = dw - gamma P^2 grad Phi(dw)
              + sqrt(2 gamma) P xi.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    P = np.asarray(settings.P, dtype=float).reshape(-1)
    P2 = P * P

    gamma = float(gamma)

    grad = grad_phi_smoothed_delta_w(
        dw,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
    )

    mean_fwd = dw - gamma * (P2 * grad)
    cand = mean_fwd + np.sqrt(2.0 * gamma) * P * rng.normal(size=dw.shape)

    def proposal_quad(x: np.ndarray, m: np.ndarray) -> float:
        z = (x - m) / (np.sqrt(2.0 * gamma) * P)
        return 0.5 * float(z @ z)

    phi_dw = -logpost_smoothed_delta_w(
        dw,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
    )

    phi_cand = -logpost_smoothed_delta_w(
        cand,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
    )

    grad_cand = grad_phi_smoothed_delta_w(
        cand,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
    )

    mean_rev = cand - gamma * (P2 * grad_cand)

    log_q_fwd = -proposal_quad(cand, mean_fwd)
    log_q_rev = -proposal_quad(dw, mean_rev)

    log_alpha = -(phi_cand - phi_dw) + (log_q_rev - log_q_fwd)

    accept = bool(np.log(rng.random()) < log_alpha)
    dw_next = cand if accept else dw

    return dw_next, accept, float(log_alpha)


# ---------------------------------------------------------------
# 4) ACF / ESS helpers
# ---------------------------------------------------------------

def acf_1d_simple(
    x: np.ndarray,
    *,
    max_lag: int = 200,
) -> np.ndarray:
    """
    Simple autocorrelation function for a 1D series.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    x = x - float(np.mean(x))

    n = x.size

    if n == 0:
        return np.array([], dtype=float)

    var = float((x @ x) / max(n, 1))

    if var <= 1e-30:
        out = np.zeros(max_lag + 1, dtype=float)
        out[0] = 1.0
        return out

    max_lag_eff = min(int(max_lag), n - 1)
    r = np.empty(max_lag_eff + 1, dtype=float)

    for lag in range(max_lag_eff + 1):
        r[lag] = float((x[: n - lag] @ x[lag:]) / (n * var + 1e-15))

    return r


def ess_1d_initial_positive(
    x: np.ndarray,
    *,
    max_lag: int = 200,
) -> float:
    """
    Initial-positive-sequence ESS estimate for a 1D series.
    """
    x = np.asarray(x, dtype=float).reshape(-1)

    if x.size <= 1:
        return float(x.size)

    r = acf_1d_simple(x, max_lag=max_lag)

    s = 0.0

    for lag in range(1, len(r)):
        if r[lag] < 0.0:
            break
        s += float(r[lag])

    tau = 1.0 + 2.0 * s

    return float(x.size / max(tau, 1e-12))


# ---------------------------------------------------------------
# 5) Lift Delta-w samples to portfolio samples and compute TE path
# ---------------------------------------------------------------

def lift_delta_w_samples_to_portfolios(
    W_delta: np.ndarray,
    *,
    w_old: np.ndarray,
    target_sum: float = 1.0,
) -> np.ndarray:
    """
    Lift Delta-w samples to full portfolio samples.

    A tiny uniform correction is applied to each lifted sample so that
    the portfolio sum is exactly target_sum. This is for diagnostics only.
    """
    W_delta = np.asarray(W_delta, dtype=float)
    w_old = np.asarray(w_old, dtype=float).reshape(-1)

    W = W_delta + w_old[None, :]
    correction = (target_sum - W.sum(axis=1, keepdims=True)) / W.shape[1]

    return W + correction


def te_path_realised_raw_for_weights(
    *,
    y_raw: np.ndarray,
    R_raw: np.ndarray,
    W: np.ndarray,
) -> np.ndarray:
    """
    Realised/raw TE path for a matrix of portfolio weights.
    """
    y_raw = np.asarray(y_raw, dtype=float).reshape(-1)
    R_raw = np.asarray(R_raw, dtype=float)
    W = np.asarray(W, dtype=float)

    preds = R_raw @ W.T
    resid = preds - y_raw[:, None]

    return np.sqrt(np.mean(resid ** 2, axis=0)).reshape(-1)


print("[Delta-w MALA helpers] Delta-w MALA/UQ helper functions defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Delta-w MALA/UQ runner
# Delta-w MALA/UQ runner for the sequential rebalancing wrapper
#
# Purpose:
#   Given a selected Delta-w c-grid output from the Delta-w grid engine, run:
#
#     1. burn-in from Delta-w MAP;
#     2. post-burn gamma tuning;
#     3. long preconditioned MALA run;
#     4. post-burn TE / ESS diagnostics.
#
# Output:
#   A Block-13B-style dictionary containing W_post, TE path, ESS,
#   acceptance rate, gamma, and diagnostic metadata.
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, Optional
import time

import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Burn-in
# ---------------------------------------------------------------

def run_delta_w_mala_burnin_asoc(
    *,
    dw_start: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
    gamma_burn: float,
    n_burn: int,
    rng: Optional[np.random.Generator] = None,
    print_every: int = 5_000,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Burn-in run for the smoothed Delta-w MALA target.

    No samples are kept. The final state is returned.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    dw = np.asarray(dw_start, dtype=float).reshape(-1).copy()

    acc = 0
    t0 = time.perf_counter()

    for k in range(1, int(n_burn) + 1):
        dw, accepted, _ = delta_w_mala_step(
            dw,
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            settings=settings,
            gamma=gamma_burn,
            rng=rng,
        )

        acc += int(accepted)

        if verbose and print_every and (k % print_every == 0):
            elapsed = time.perf_counter() - t0
            print(
                f"[Delta-w MALA | burn-in] "
                f"{k:7d}/{n_burn}, acc={acc / k:5.3f}, "
                f"elapsed={elapsed:7.1f}s"
            )

    wall = time.perf_counter() - t0
    acc_rate = acc / max(int(n_burn), 1)

    if verbose:
        print(
            f"[Delta-w MALA | burn-in] done: "
            f"n_burn={n_burn}, acc_rate={acc_rate:.3f}, wall={wall:.1f}s"
        )

    return {
        "dw_final": dw,
        "acc_rate": float(acc_rate),
        "wall_seconds": float(wall),
        "gamma_burn": float(gamma_burn),
        "n_burn": int(n_burn),
    }


# ---------------------------------------------------------------
# 1) Post-burn gamma tuning
# ---------------------------------------------------------------

def tune_gamma_delta_w_postburn_asoc(
    *,
    dw_start: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
    gamma_init: float,
    target_accept: float = 0.60,
    n_stages: int = 6,
    n_iter_stage: int = 5_000,
    rng: Optional[np.random.Generator] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Tune the MALA step size after burn-in.

    Rule:
      - if acceptance is too high, increase gamma;
      - if acceptance is too low, decrease gamma;
      - otherwise stop.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    dw = np.asarray(dw_start, dtype=float).reshape(-1).copy()
    gamma = float(gamma_init)

    records = []

    for stage in range(1, int(n_stages) + 1):
        acc = 0

        for _ in range(int(n_iter_stage)):
            dw, accepted, _ = delta_w_mala_step(
                dw,
                y_c=y_c,
                R_c=R_c,
                w_old=w_old,
                alpha_dw=alpha_dw,
                settings=settings,
                gamma=gamma,
                rng=rng,
            )

            acc += int(accepted)

        acc_rate = acc / max(int(n_iter_stage), 1)

        records.append(
            {
                "stage": int(stage),
                "gamma": float(gamma),
                "acc_rate": float(acc_rate),
            }
        )

        if verbose:
            print(
                f"[Delta-w MALA | tune] "
                f"stage={stage:2d}, gamma={gamma:.3e}, acc={acc_rate:.3f}"
            )

        if acc_rate > target_accept + 0.05:
            gamma *= 1.25
        elif acc_rate < target_accept - 0.10:
            gamma *= 0.50
        else:
            break

    if verbose:
        print(f"[Delta-w MALA | tune] final gamma ≈ {gamma:.6e}")

    return {
        "gamma_star": float(gamma),
        "dw_final": dw,
        "df_tune": pd.DataFrame(records),
        "target_accept": float(target_accept),
    }


# ---------------------------------------------------------------
# 2) Long MALA run
# ---------------------------------------------------------------

def run_delta_w_mala_long_asoc(
    *,
    dw_start: np.ndarray,
    y_c: np.ndarray,
    R_c: np.ndarray,
    y_fit: np.ndarray,
    R_fit: np.ndarray,
    w_old: np.ndarray,
    alpha_dw: np.ndarray,
    settings: DeltaWMalaSettings,
    gamma: float,
    n_iter: int,
    thin: int,
    burn_kept: int,
    max_lag_ess: int = 200,
    rng: Optional[np.random.Generator] = None,
    print_every_kept: int = 2_000,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Long preconditioned MALA run for the smoothed Delta-w posterior.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    dw = np.asarray(dw_start, dtype=float).reshape(-1).copy()
    p = dw.size

    n_iter = int(n_iter)
    thin = int(thin)
    burn_kept = int(burn_kept)

    if thin <= 0:
        raise ValueError("thin must be positive.")

    n_keep_alloc = n_iter // thin

    W = np.empty((n_keep_alloc, p), dtype=float)
    L1_trace = np.empty(n_keep_alloc, dtype=float)
    LP_trace = np.empty(n_keep_alloc, dtype=float)

    acc = 0
    kept = 0
    t0 = time.perf_counter()

    for k in range(1, n_iter + 1):
        dw, accepted, _ = delta_w_mala_step(
            dw,
            y_c=y_c,
            R_c=R_c,
            w_old=w_old,
            alpha_dw=alpha_dw,
            settings=settings,
            gamma=gamma,
            rng=rng,
        )

        acc += int(accepted)

        if k % thin == 0:
            W[kept, :] = dw
            L1_trace[kept] = float(np.sum(np.abs(dw)))
            LP_trace[kept] = logpost_smoothed_delta_w(
                dw,
                y_c=y_c,
                R_c=R_c,
                w_old=w_old,
                alpha_dw=alpha_dw,
                settings=settings,
            )

            kept += 1

            if verbose and print_every_kept and (kept % print_every_kept == 0):
                elapsed = time.perf_counter() - t0
                print(
                    f"[Delta-w MALA | long] kept={kept:7d}/{n_keep_alloc}, "
                    f"acc={acc / k:5.3f}, "
                    f"||dw||1={L1_trace[kept - 1]:.3e}, "
                    f"elapsed={elapsed:7.1f}s"
                )

    wall = time.perf_counter() - t0
    acc_rate = acc / max(n_iter, 1)

    W = W[:kept, :]
    L1_trace = L1_trace[:kept]
    LP_trace = LP_trace[:kept]

    if burn_kept >= W.shape[0] - 5:
        raise RuntimeError(
            f"burn_kept={burn_kept} is too large for kept={W.shape[0]}."
        )

    W_post = W[burn_kept:, :]
    L1_post = L1_trace[burn_kept:]
    LP_post = LP_trace[burn_kept:]

    W_port_post = lift_delta_w_samples_to_portfolios(
        W_post,
        w_old=w_old,
        target_sum=1.0,
    )

    te_fit_path = te_path_realised_raw_for_weights(
        y_raw=y_fit,
        R_raw=R_fit,
        W=W_port_post,
    )

    ess_te = ess_1d_initial_positive(
        te_fit_path,
        max_lag=max_lag_ess,
    )

    top_idx = np.argsort(-np.abs(dw_start))[:4]
    ess_coords = {}

    for j in top_idx:
        ess_coords[int(j)] = ess_1d_initial_positive(
            W_post[:, int(j)],
            max_lag=max_lag_ess,
        )

    te_q05, te_q50, te_q95 = np.quantile(te_fit_path, [0.05, 0.50, 0.95])

    if verbose:
        print("\n[Delta-w MALA | post-burn summary]")
        print(f"  proposals       = {n_iter}")
        print(f"  thin            = {thin}")
        print(f"  kept total      = {W.shape[0]}")
        print(f"  burn kept       = {burn_kept}")
        print(f"  kept post-burn  = {W_post.shape[0]}")
        print(f"  gamma           = {gamma:.6e}")
        print(f"  accept rate     = {acc_rate:.3f}")
        print(f"  wall seconds    = {wall:.1f}")
        print(
            f"  TE mean raw     = {np.mean(te_fit_path):.6e} "
            f"({1e4 * np.mean(te_fit_path):.3f} bp)"
        )
        print(
            f"  TE median raw   = {te_q50:.6e} "
            f"({1e4 * te_q50:.3f} bp)"
        )
        print(
            f"  TE 90% interval = "
            f"[{te_q05:.6e}, {te_q95:.6e}]"
        )
        print(f"  ESS(TE)         = {ess_te:.1f}")
        print(
            "  ESS sentinel Delta-w = "
            + ", ".join(
                [f"j={j}: {ess_coords[int(j)]:.1f}" for j in top_idx]
            )
        )

    return {
        "W": W,
        "W_post": W_post,
        "L1_trace": L1_trace,
        "L1_post": L1_post,
        "LP": LP_trace,
        "LP_post": LP_post,
        "te_fit_path": te_fit_path,
        "ess_te": float(ess_te),
        "ess_coords": ess_coords,
        "acc_rate": float(acc_rate),
        "wall_seconds": float(wall),
        "gamma": float(gamma),
        "thin": int(thin),
        "n_iter": int(n_iter),
        "burn_kept": int(burn_kept),
        "top_idx": np.asarray(top_idx, dtype=int),
        "te_summary": {
            "mean": float(np.mean(te_fit_path)),
            "median": float(te_q50),
            "q05": float(te_q05),
            "q95": float(te_q95),
        },
    }


# ---------------------------------------------------------------
# 3) Full MALA/UQ driver from c-grid output
# ---------------------------------------------------------------

def run_delta_w_mala_uq_asoc(
    *,
    cgrid_out: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
    rng: Optional[np.random.Generator] = None,
) -> Dict[str, Any]:
    """
    Run the full Delta-w MALA/UQ stage after the c-grid selection.
    """
    rng = np.random.default_rng(0) if rng is None else rng

    context = cgrid_out["context"]

    y_c = np.asarray(context["y_c"], dtype=float).reshape(-1)
    R_c = np.asarray(context["R_c"], dtype=float)
    y_fit = np.asarray(context["y_fit"], dtype=float).reshape(-1)
    R_fit = np.asarray(context["R_fit"], dtype=float)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    alpha_dw = np.asarray(context["alpha_dw"], dtype=float).reshape(-1)

    sigma2 = float(cgrid_out["sigma2_dw_final"])
    kappa = float(cgrid_out["kappa_EB_dw_final"])
    lambda_MY = float(cgrid_out["dw_steps_final"].lambda_MY)
    dw_map_start = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)

    P = build_diag_preconditioner_delta_w(
        R_c,
        sigma2=sigma2,
    )

    L_pre = power_largest_eig_precond_delta_w(
        R_c,
        sigma2=sigma2,
        P=P,
        rng=rng,
    )

    gamma0_pre = 0.9 / (2.0 * L_pre)

    settings = DeltaWMalaSettings(
        sigma2=sigma2,
        lambda_MY=lambda_MY,
        kappa=kappa,
        P=P,
    )

    if config.verbose:
        print("\n[Delta-w MALA run | Delta-w MALA resolved inputs]")
        print(f"  sigma2_dw = {sigma2:.3e}")
        print(f"  kappa_dw  = {kappa:.3e}")
        print(f"  lambda_MY = {lambda_MY:.3e}")
        print(f"  FIT shape = T={R_c.shape[0]}, p={R_c.shape[1]}")
        print("\n[Delta-w MALA run | preconditioner]")
        print(
            f"  P_dw: min={P.min():.3e}, "
            f"median={np.median(P):.3e}, max={P.max():.3e}"
        )
        print(f"  L_pre_dw = {L_pre:.6e}")
        print(f"  gamma0_pre_dw = {gamma0_pre:.6e}")
        print("\n[Delta-w MALA run | run settings]")
        print(f"  n_burn_steps_dw   = {config.n_burn_steps_dw}")
        print(f"  n_iter_long_dw    = {config.n_iter_long_dw}")
        print(f"  thin_long_dw      = {config.thin_long_dw}")
        print(f"  burn_kept_long_dw = {config.burn_kept_long_dw}")
        print(f"  max_lag_ess_dw    = {config.max_lag_ess_dw}")

    burn_out = run_delta_w_mala_burnin_asoc(
        dw_start=dw_map_start,
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
        gamma_burn=gamma0_pre,
        n_burn=config.n_burn_steps_dw,
        rng=rng,
        verbose=config.verbose,
    )

    tune_out = tune_gamma_delta_w_postburn_asoc(
        dw_start=burn_out["dw_final"],
        y_c=y_c,
        R_c=R_c,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
        gamma_init=gamma0_pre,
        target_accept=0.60,
        n_stages=6,
        n_iter_stage=5_000,
        rng=rng,
        verbose=config.verbose,
    )

    long_out = run_delta_w_mala_long_asoc(
        dw_start=tune_out["dw_final"],
        y_c=y_c,
        R_c=R_c,
        y_fit=y_fit,
        R_fit=R_fit,
        w_old=w_old,
        alpha_dw=alpha_dw,
        settings=settings,
        gamma=tune_out["gamma_star"],
        n_iter=config.n_iter_long_dw,
        thin=config.thin_long_dw,
        burn_kept=config.burn_kept_long_dw,
        max_lag_ess=config.max_lag_ess_dw,
        rng=rng,
        verbose=config.verbose,
    )

    out = {
        **long_out,
        "settings": settings,
        "P": P,
        "L_pre": float(L_pre),
        "gamma0_pre": float(gamma0_pre),
        "burn_out": burn_out,
        "tune_out": tune_out,
        "cgrid_out": cgrid_out,
        "context": context,
        "sigma2_dw": sigma2,
        "kappa_dw": kappa,
        "lambda_MY": lambda_MY,
        "dw_map_start": dw_map_start,
    }

    if config.verbose:
        print("\n[Delta-w MALA run] dw_mala_out_fit2-style output created.")

    return out


print("[Delta-w MALA run] Delta-w MALA/UQ runner defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — Delta-w candidate builder
# Delta-w ROPE / directional practical-change candidates
#
# Purpose:
#   Build posterior-informed rebalancing candidates from Delta-w MAP
#   and post-burn MALA samples.
#
# Rule family:
#
#   For each coordinate j, let s_j = sign(Delta-w_MAP,j).
#
#   A coordinate is directionally approved at (eps, pi_move) if:
#
#       |Delta-w_MAP,j| >= eps
#
#   and, conditional on the MAP direction,
#
#       if Delta-w_MAP,j > 0:
#           P(Delta-w_j > eps | y) >= pi_move
#
#       if Delta-w_MAP,j < 0:
#           P(Delta-w_j < -eps | y) >= pi_move.
#
#   This captures:
#     - practical nonzero movement relative to zero;
#     - confidence in the proposed direction;
#     - no final trade implementation yet.
#
# Output:
#   dw_rope_candidates_fit2-style dictionary containing:
#     - candidate grid table;
#     - named candidate objects;
#     - selected candidate for the local implementation rule / 14D-5.
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, Sequence, Optional, Tuple
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Ticker helper
# ---------------------------------------------------------------

def get_asset_tickers_from_context_asoc(
    context: Dict[str, Any],
) -> list[str]:
    """
    Recover ticker names from a rebalance context.
    """
    if "asset_cols" in context:
        return [str(x) for x in context["asset_cols"]]

    p = int(context["p"])
    return [f"asset_{j}" for j in range(p)]


# ---------------------------------------------------------------
# 1) Stable candidate naming
# ---------------------------------------------------------------

def format_eps_tag_asoc(eps: float) -> str:
    """
    Stable short tag for practical movement thresholds.
    """
    eps = float(eps)

    if np.isclose(eps, 5e-5):
        return "5e-5"
    if np.isclose(eps, 1e-4):
        return "1e-4"
    if np.isclose(eps, 2.5e-4):
        return "2p5e-4"
    if np.isclose(eps, 5e-4):
        return "5e-4"
    if np.isclose(eps, 1e-3):
        return "1e-3"

    return f"{eps:.0e}".replace("-0", "-")


def make_rope_candidate_name_asoc(
    *,
    eps: float,
    pi_move: float,
    prefix: str = "dir",
) -> str:
    """
    Stable candidate-rule name.
    """
    return f"{prefix}_eps{format_eps_tag_asoc(eps)}_pi{pi_move:.2f}"


# ---------------------------------------------------------------
# 2) Build directional posterior probabilities
# ---------------------------------------------------------------

def build_delta_w_directional_probabilities_asoc(
    *,
    W_delta_post: np.ndarray,
    eps_grid: Sequence[float],
) -> Dict[str, Dict[float, np.ndarray]]:
    """
    Compute posterior probabilities for practical positive/negative moves.

    Returns
    -------
    probs : dict
        probs["P_pos"][eps][j] = P(Delta-w_j > eps | y)
        probs["P_neg"][eps][j] = P(Delta-w_j < -eps | y)
        probs["P_zero"][eps][j] = P(|Delta-w_j| <= eps | y)
        probs["P_nonzero"][eps][j] = P(|Delta-w_j| > eps | y)
    """
    W_delta_post = np.asarray(W_delta_post, dtype=float)

    if W_delta_post.ndim != 2:
        raise ValueError("W_delta_post must be a 2D array.")

    P_pos = {}
    P_neg = {}
    P_zero = {}
    P_nonzero = {}

    for eps in eps_grid:
        eps = float(eps)

        P_pos[eps] = np.mean(W_delta_post > eps, axis=0)
        P_neg[eps] = np.mean(W_delta_post < -eps, axis=0)
        P_zero[eps] = np.mean(np.abs(W_delta_post) <= eps, axis=0)
        P_nonzero[eps] = np.mean(np.abs(W_delta_post) > eps, axis=0)

    return {
        "P_pos": P_pos,
        "P_neg": P_neg,
        "P_zero": P_zero,
        "P_nonzero": P_nonzero,
    }


# ---------------------------------------------------------------
# 3) Build one directional candidate
# ---------------------------------------------------------------

def build_one_delta_w_rope_candidate_asoc(
    *,
    dw_map: np.ndarray,
    W_delta_post: np.ndarray,
    w_old: np.ndarray,
    tickers: Sequence[str],
    eps: float,
    pi_move: float,
    candidate_name: str,
    probs: Optional[Dict[str, Dict[float, np.ndarray]]] = None,
) -> Dict[str, Any]:
    """
    Build one directional practical-change candidate.

    This block only identifies candidate moves. It does not enforce the
    budget, no-negativity, funding, or no-trade rules.
    """
    dw_map = np.asarray(dw_map, dtype=float).reshape(-1)
    W_delta_post = np.asarray(W_delta_post, dtype=float)
    w_old = np.asarray(w_old, dtype=float).reshape(-1)

    p = dw_map.size

    if W_delta_post.ndim != 2 or W_delta_post.shape[1] != p:
        raise ValueError("W_delta_post must have shape (n_samples, p).")

    if w_old.size != p:
        raise ValueError("w_old and dw_map must have the same length.")

    if len(tickers) != p:
        raise ValueError("tickers length must equal p.")

    eps = float(eps)
    pi_move = float(pi_move)

    if probs is None:
        probs = build_delta_w_directional_probabilities_asoc(
            W_delta_post=W_delta_post,
            eps_grid=[eps],
        )

    P_pos = np.asarray(probs["P_pos"][eps], dtype=float)
    P_neg = np.asarray(probs["P_neg"][eps], dtype=float)
    P_zero = np.asarray(probs["P_zero"][eps], dtype=float)
    P_nonzero = np.asarray(probs["P_nonzero"][eps], dtype=float)

    is_positive_map = dw_map > 0.0
    is_negative_map = dw_map < 0.0

    dir_prob = np.zeros(p, dtype=float)
    dir_prob[is_positive_map] = P_pos[is_positive_map]
    dir_prob[is_negative_map] = P_neg[is_negative_map]

    practical_size = np.abs(dw_map) >= eps
    directional_confident = dir_prob >= pi_move

    keep = practical_size & directional_confident
    S_idx = np.where(keep)[0]

    buy_idx = S_idx[dw_map[S_idx] > 0.0]
    sell_idx = S_idx[dw_map[S_idx] < 0.0]

    total_buy_map = float(np.sum(np.maximum(dw_map[S_idx], 0.0)))
    total_sell_map = float(np.sum(np.maximum(-dw_map[S_idx], 0.0)))

    # Feasibility before any implementation:
    # A sell is individually feasible if w_old_j + dw_map_j >= 0.
    individually_nonnegative = (w_old[S_idx] + dw_map[S_idx]) >= -1e-14

    df_moves = pd.DataFrame(
        {
            "j": S_idx,
            "ticker": [str(tickers[j]) for j in S_idx],
            "direction": np.where(dw_map[S_idx] > 0.0, "increase", "decrease"),
            "dw_MAP": dw_map[S_idx],
            "abs_dw_MAP": np.abs(dw_map[S_idx]),
            "w_old": w_old[S_idx],
            "w_old_plus_dw_MAP": w_old[S_idx] + dw_map[S_idx],
            "directional_confidence": dir_prob[S_idx],
            "P_pos_eps": P_pos[S_idx],
            "P_neg_eps": P_neg[S_idx],
            "P_zero_eps": P_zero[S_idx],
            "P_nonzero_eps": P_nonzero[S_idx],
            "nonnegative_if_raw_MAP_move": individually_nonnegative,
        }
    )

    if not df_moves.empty:
        df_moves = (
            df_moves
            .sort_values(["directional_confidence", "abs_dw_MAP"], ascending=[False, False])
            .reset_index(drop=True)
        )

    map_move_mass = float(np.sum(np.abs(dw_map[S_idx])))
    map_total_mass = float(np.sum(np.abs(dw_map))) + 1e-16

    out = {
        "candidate_name": str(candidate_name),
        "eps": eps,
        "pi_move": pi_move,
        "S_idx": S_idx,
        "buy_idx": buy_idx,
        "sell_idx": sell_idx,
        "S_size": int(S_idx.size),
        "n_buy": int(buy_idx.size),
        "n_sell": int(sell_idx.size),
        "total_buy_map": total_buy_map,
        "total_sell_map": total_sell_map,
        "net_map_sum_on_S": float(np.sum(dw_map[S_idx])) if S_idx.size else 0.0,
        "map_move_mass": map_move_mass,
        "map_move_mass_share": float(map_move_mass / map_total_mass),
        "min_directional_confidence": (
            float(np.min(dir_prob[S_idx])) if S_idx.size else np.nan
        ),
        "mean_directional_confidence": (
            float(np.mean(dir_prob[S_idx])) if S_idx.size else np.nan
        ),
        "max_P_zero_eps": (
            float(np.max(P_zero[S_idx])) if S_idx.size else np.nan
        ),
        "all_raw_map_moves_nonnegative": bool(np.all(individually_nonnegative)) if S_idx.size else True,
        "df_moves": df_moves,
        "dir_prob": dir_prob,
        "P_pos_eps": P_pos,
        "P_neg_eps": P_neg,
        "P_zero_eps": P_zero,
        "P_nonzero_eps": P_nonzero,
    }

    return out


# ---------------------------------------------------------------
# 4) Build the full ROPE frontier and named candidates
# ---------------------------------------------------------------

def build_delta_w_rope_candidates_asoc(
    *,
    cgrid_out: Dict[str, Any],
    mala_out: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
    eps_grid: Optional[Sequence[float]] = None,
    pi_move_grid: Optional[Sequence[float]] = None,
    selected_candidate_name: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Build ROPE / directional practical-change candidate sets.

    The default selected candidate comes from config.selected_rope_candidate.
    """
    context = cgrid_out["context"]

    dw_map = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    W_delta_post = np.asarray(mala_out["W_post"], dtype=float)

    tickers = get_asset_tickers_from_context_asoc(context)

    if eps_grid is None:
        eps_grid = sorted(set([5e-5, 1e-4, 2.5e-4, 5e-4, 1e-3, float(CALM_EPS_DELTA)]))
    if pi_move_grid is None:
        pi_move_grid = sorted(set([0.50, 0.60, 0.70, 0.80, 0.90, float(CALM_P_DIR)]))

    eps_grid = tuple(float(eps) for eps in eps_grid)
    pi_move_grid = tuple(float(pi) for pi in pi_move_grid)

    probs = build_delta_w_directional_probabilities_asoc(
        W_delta_post=W_delta_post,
        eps_grid=eps_grid,
    )

    frontier_rows = []
    candidates: Dict[str, Dict[str, Any]] = {}

    for eps in eps_grid:
        for pi_move in pi_move_grid:
            # Human-readable tier labels used for the three main candidates.
            if np.isclose(eps, 5e-5) and np.isclose(pi_move, 0.60):
                name = make_rope_candidate_name_asoc(
                    eps=eps,
                    pi_move=pi_move,
                    prefix="broad_dir",
                )
            elif np.isclose(eps, 1e-4) and np.isclose(pi_move, 0.60):
                name = make_rope_candidate_name_asoc(
                    eps=eps,
                    pi_move=pi_move,
                    prefix="moderate_dir",
                )
            elif np.isclose(eps, 2.5e-4) and np.isclose(pi_move, 0.60):
                name = make_rope_candidate_name_asoc(
                    eps=eps,
                    pi_move=pi_move,
                    prefix="conservative_dir",
                )
            else:
                name = make_rope_candidate_name_asoc(
                    eps=eps,
                    pi_move=pi_move,
                    prefix="dir",
                )

            cand = build_one_delta_w_rope_candidate_asoc(
                dw_map=dw_map,
                W_delta_post=W_delta_post,
                w_old=w_old,
                tickers=tickers,
                eps=eps,
                pi_move=pi_move,
                candidate_name=name,
                probs=probs,
            )

            candidates[name] = cand

            frontier_rows.append(
                {
                    "candidate_name": name,
                    "eps": eps,
                    "pi_move": pi_move,
                    "S": cand["S_size"],
                    "n_buy": cand["n_buy"],
                    "n_sell": cand["n_sell"],
                    "map_move_mass": cand["map_move_mass"],
                    "map_move_mass_share": cand["map_move_mass_share"],
                    "total_buy_map": cand["total_buy_map"],
                    "total_sell_map": cand["total_sell_map"],
                    "net_map_sum_on_S": cand["net_map_sum_on_S"],
                    "mean_directional_confidence": cand["mean_directional_confidence"],
                    "min_directional_confidence": cand["min_directional_confidence"],
                    "max_P_zero_eps": cand["max_P_zero_eps"],
                    "all_raw_map_moves_nonnegative": cand["all_raw_map_moves_nonnegative"],
                }
            )

    df_frontier = pd.DataFrame(frontier_rows)

    # Main named candidates. These are convenient aliases for discussion.
    candidate_aliases = {
        "broad_dir_eps5e-5_pi0.60": "broad_dir_eps5e-5_pi0.60",
        "moderate_dir_eps1e-4_pi0.60": "moderate_dir_eps1e-4_pi0.60",
        "conservative_dir_eps2p5e-4_pi0.60": "conservative_dir_eps2p5e-4_pi0.60",
    }

    if selected_candidate_name is None:
        selected_candidate_name = config.selected_rope_candidate

    if selected_candidate_name not in candidates:
        available = list(candidates.keys())
        raise KeyError(
            f"selected_candidate_name={selected_candidate_name!r} not found. "
            f"Available candidates include: {available[:10]} ..."
        )

    selected_candidate = candidates[selected_candidate_name]

    out = {
        "context": context,
        "cgrid_out": cgrid_out,
        "mala_out": mala_out,
        "df_frontier": df_frontier,
        "candidates": candidates,
        "candidate_aliases": candidate_aliases,
        "selected_candidate_name": selected_candidate_name,
        "selected_candidate": selected_candidate,
        "eps_grid": eps_grid,
        "pi_move_grid": pi_move_grid,
        "probs": probs,
        "dw_map": dw_map,
        "W_delta_post": W_delta_post,
    }

    if config.verbose:
        print("\n[Delta-w candidates | Delta-w ROPE / directional candidates]")
        print(f"FIT window = hold transition before Hold {context['hold_number']}")
        print(f"p = {context['p']}")
        print(f"n_samples = {W_delta_post.shape[0]}")
        print(f"selected candidate = {selected_candidate_name}")
        print(f"||Delta-w_MAP||_1 = {np.sum(np.abs(dw_map)):.6e}")
        print(f"max |Delta-w_MAP| = {np.max(np.abs(dw_map)):.6e}")
        print(
            f"nnz_eff(|Delta-w_MAP| >= {context['nnz_eff_thresh_dw']:.1e}) = "
            f"{np.count_nonzero(np.abs(dw_map) >= context['nnz_eff_thresh_dw'])}"
        )
        print(f"sum(Delta-w_MAP) = {dw_map.sum():+.3e}")

        display = df_frontier.copy()
        display["eps"] = display["eps"].map(lambda x: f"{x:.1e}")
        display["pi_move"] = display["pi_move"].map(lambda x: f"{x:.2f}")

        for col in [
            "map_move_mass",
            "map_move_mass_share",
            "total_buy_map",
            "total_sell_map",
            "net_map_sum_on_S",
            "mean_directional_confidence",
            "min_directional_confidence",
            "max_P_zero_eps",
        ]:
            display[col] = display[col].map(
                lambda x: "" if pd.isna(x) else f"{float(x):.3e}"
            )

        print("\n=== Delta-w directional practical-change frontier ===")
        print(
            display[
                [
                    "eps",
                    "pi_move",
                    "S",
                    "n_buy",
                    "n_sell",
                    "map_move_mass",
                    "map_move_mass_share",
                    "mean_directional_confidence",
                    "min_directional_confidence",
                    "max_P_zero_eps",
                ]
            ].to_string(index=False)
        )

        print("\n=== Named Delta-w candidate supports ===")
        for alias in candidate_aliases:
            if alias in candidates:
                cand = candidates[alias]
                print(
                    f"{alias:38s} | "
                    f"|S|={cand['S_size']:3d} | "
                    f"buy={cand['n_buy']:3d} | sell={cand['n_sell']:3d} | "
                    f"mass={cand['map_move_mass']:.3e} | "
                    f"mean conf={cand['mean_directional_confidence']:.3f}"
                )

        print("\n=== Selected candidate moves ===")
        df_moves = selected_candidate["df_moves"].copy()

        if df_moves.empty:
            print("(empty)")
        else:
            df_disp = df_moves.copy()
            for col in [
                "dw_MAP",
                "abs_dw_MAP",
                "w_old",
                "w_old_plus_dw_MAP",
            ]:
                df_disp[col] = df_disp[col].map(lambda x: f"{x:+.4e}")

            for col in [
                "directional_confidence",
                "P_pos_eps",
                "P_neg_eps",
                "P_zero_eps",
                "P_nonzero_eps",
            ]:
                df_disp[col] = df_disp[col].map(lambda x: f"{x:.3f}")

            print(df_disp.to_string(index=False))

        print("\n[Delta-w candidates] dw_rope_candidates_fit2-style object created.")
        print("[Delta-w candidates] No rebalance decision has been locked in this block.")

    return out


print("[Delta-w candidates] Delta-w ROPE / directional candidate builder defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — local implementation rule
# Locked local-budget rebalance decision from ROPE candidates
#
# Purpose:
#   Convert the selected Delta-w ROPE / directional candidate from
#   the Delta-w candidate builder into an implementable portfolio update.
#
# Rebalanced decisions encoded here:
#   1. Empty candidate sets do not trade, but a single approved buy may trade.
#   2. Negative Delta-w_i moves are allowed, but final weights must
#      satisfy w_old_i + Delta-w_i >= 0.
#   3. If any proposed final weight would be negative, set that name's
#      final weight to zero and repair locally if possible.
#   4. If approved moves contain both buys and sells, touch only those
#      approved names and match the traded buy/sell budget locally.
#   5. If approved moves are buy-only, fund them locally from at most a
#      small number of the smallest existing positive holdings.
#   6. Buy-only implementation may open only a bounded number of new names;
#      excess new-name buy candidates are dropped, while existing-name
#      increases are kept.
#   7. If approved moves are sell-only, do not rebalance because there is
#      no posterior-approved destination for the freed budget.
#   8. Never globally renormalise the whole portfolio.
#
# Output:
#   maintenance-driver-style decision dictionary, compatible with the sequential driver.
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, Optional, Sequence, Tuple
import numpy as np
import pandas as pd


# ---------------------------------------------------------------
# 0) Small diagnostics
# ---------------------------------------------------------------

def asoc_trade_summary_from_delta(
    dw: np.ndarray,
    *,
    tol: float = 1e-12,
) -> Dict[str, Any]:
    """
    Basic diagnostics for an implemented Delta-w.
    """
    dw = np.asarray(dw, dtype=float).reshape(-1)

    return {
        "sum_dw": float(dw.sum()),
        "turnover_half_L1": float(0.5 * np.sum(np.abs(dw))),
        "L1_dw": float(np.sum(np.abs(dw))),
        "nnz_trades": int(np.count_nonzero(np.abs(dw) > tol)),
        "n_increases": int(np.count_nonzero(dw > tol)),
        "n_decreases": int(np.count_nonzero(dw < -tol)),
        "max_abs_dw": float(np.max(np.abs(dw))) if dw.size else 0.0,
    }


def asoc_portfolio_structure_summary(
    w: np.ndarray,
    *,
    tol: float = 1e-12,
) -> Dict[str, Any]:
    """
    Basic portfolio diagnostics after a rebalance decision.
    """
    w = np.asarray(w, dtype=float).reshape(-1)
    abs_w = np.abs(w)
    L1 = float(abs_w.sum()) + 1e-16

    return {
        "sum_w": float(w.sum()),
        "nnz_exact": int(np.count_nonzero(abs_w > tol)),
        "nnz_eff_1e_minus_4": int(np.count_nonzero(abs_w >= 1e-4)),
        "n_shorts": int(np.count_nonzero(w < -tol)),
        "min_w": float(np.min(w)) if w.size else np.nan,
        "max_w": float(np.max(w)) if w.size else np.nan,
        "L1": float(abs_w.sum()),
        "HHI_abs": float(np.sum((abs_w / L1) ** 2)),
    }


# ---------------------------------------------------------------
# 1) Funding helper for approved buy-only moves
# ---------------------------------------------------------------

def asoc_choose_small_weight_funders(
    *,
    w_old: np.ndarray,
    excluded_idx: np.ndarray,
    required_budget: float,
    max_funding_names: int = 3,
    close_small_funders: bool = True,
    dust_tol: float = 1e-5,
    tol: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Select the smallest currently held positive weights to fund approved buys.

    The helper uses at most max_funding_names funders. If close_small_funders
    is True, a funding name is closed when doing so overshoots the remaining
    funding need only by dust_tol. This avoids leaving dust positions such as
    a residual 8e-6 holding after a buy-only rebalance.
    """
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    excluded_idx = np.asarray(excluded_idx, dtype=int).reshape(-1)

    p = w_old.size
    max_funding_names = int(max(max_funding_names, 0))
    dust_tol = float(max(dust_tol, 0.0))

    if max_funding_names <= 0 or required_budget <= tol:
        return np.array([], dtype=int), np.array([], dtype=float), 0.0

    excluded_mask = np.zeros(p, dtype=bool)
    excluded_idx = excluded_idx[(excluded_idx >= 0) & (excluded_idx < p)]
    excluded_mask[excluded_idx] = True

    eligible = np.where((w_old > tol) & (~excluded_mask))[0]

    if eligible.size == 0:
        return np.array([], dtype=int), np.array([], dtype=float), 0.0

    eligible = eligible[np.argsort(w_old[eligible])]

    funders = []
    sell_amounts = []
    obtained = 0.0

    for j in eligible:
        if len(funders) >= max_funding_names:
            break

        remaining = float(required_budget - obtained)

        if remaining <= tol:
            break

        available = float(w_old[j])

        if close_small_funders and available <= remaining + dust_tol:
            sell_j = available
        else:
            sell_j = min(available, remaining)

        if sell_j > tol:
            funders.append(int(j))
            sell_amounts.append(float(sell_j))
            obtained += float(sell_j)

        if obtained >= required_budget - tol:
            break

    if len(funders) == 0:
        return np.array([], dtype=int), np.array([], dtype=float), 0.0

    funder_idx = np.asarray(funders, dtype=int)
    sell_amounts = np.asarray(sell_amounts, dtype=float)
    budget_obtained = float(np.sum(sell_amounts))

    return funder_idx, sell_amounts, budget_obtained


# ---------------------------------------------------------------
# 2) Local negative-weight repair
# ---------------------------------------------------------------

def asoc_repair_negative_weights_locally(
    *,
    w_old: np.ndarray,
    dw_impl: np.ndarray,
    preferred_buy_idx: np.ndarray,
    tol: float = 1e-12,
) -> Tuple[np.ndarray, np.ndarray, Optional[str]]:
    """
    Enforce the locked rule:

        if w_new_j < 0, set w_new_j = 0.

    The budget increase caused by clipping is repaired by reducing
    approved buys proportionally. If that is impossible, return an error.
    """
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    dw_impl = np.asarray(dw_impl, dtype=float).reshape(-1)
    preferred_buy_idx = np.asarray(preferred_buy_idx, dtype=int).reshape(-1)

    w_new = w_old + dw_impl

    neg_idx = np.where(w_new < -tol)[0]

    if neg_idx.size == 0:
        tiny_neg_idx = np.where((w_new < 0.0) & (w_new >= -tol))[0]

        if tiny_neg_idx.size > 0:
            w_new[tiny_neg_idx] = 0.0
            dw_impl = w_new - w_old

        return dw_impl, w_new, None

    budget_to_remove = float(np.sum(-w_new[neg_idx]))

    w_new[neg_idx] = 0.0
    dw_impl = w_new - w_old

    if budget_to_remove <= tol:
        return dw_impl, w_new, None

    buy_idx = preferred_buy_idx[
        (preferred_buy_idx >= 0)
        & (preferred_buy_idx < w_old.size)
        & (dw_impl[preferred_buy_idx] > tol)
    ]

    if buy_idx.size == 0:
        return (
            np.zeros_like(dw_impl),
            w_old.copy(),
            "negative-weight clipping required budget repair but no approved buys were available",
        )

    buy_available = float(np.sum(dw_impl[buy_idx]))

    if buy_available + tol < budget_to_remove:
        return (
            np.zeros_like(dw_impl),
            w_old.copy(),
            "negative-weight clipping required more budget repair than approved buys could absorb",
        )

    reduction = budget_to_remove * dw_impl[buy_idx] / buy_available
    dw_impl[buy_idx] -= reduction

    w_new = w_old + dw_impl

    tiny_neg_idx = np.where((w_new < 0.0) & (w_new >= -tol))[0]

    if tiny_neg_idx.size > 0:
        w_new[tiny_neg_idx] = 0.0
        dw_impl = w_new - w_old

    if np.any(w_new < -tol):
        return (
            np.zeros_like(dw_impl),
            w_old.copy(),
            "local repair still produced material negative weights",
        )

    return dw_impl, w_new, None


# ---------------------------------------------------------------
# 3) Locked implementation rule
# ---------------------------------------------------------------

def asoc_implement_locked_rebalance(
    *,
    w_old: np.ndarray,
    dw_map: np.ndarray,
    S_idx: np.ndarray,
    selected_min_confidence: Optional[float],
    candidate_threshold: Optional[float],
    min_names_to_trade: int = 2,
    funding_min_confidence: float = 0.80,
    max_buy_only_funding_names: int = 3,
    close_small_funders: bool = True,
    buy_only_dust_tol: float = 1e-5,
    max_cardinality_change: int = 3,
    allow_new_buy_names: bool = True,
    tol: float = 1e-12,
) -> Dict[str, Any]:
    """
    Implement the local-budget rebalancing rules.

    Notes
    -----
    min_names_to_trade and funding_min_confidence are retained for legacy
    config compatibility and reporting, but the revised buy-only rule does
    not reject solely because |S|=1 or selected_min_confidence < 0.80.
    """
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    dw_map = np.asarray(dw_map, dtype=float).reshape(-1)
    S_idx = np.unique(np.asarray(S_idx, dtype=int).reshape(-1))

    p = w_old.size
    dw_impl = np.zeros(p, dtype=float)

    max_buy_only_funding_names = int(max(max_buy_only_funding_names, 0))
    max_cardinality_change = int(max(max_cardinality_change, 0))

    if dw_map.size != p:
        raise ValueError("dw_map and w_old must have the same length.")

    S_idx = S_idx[(S_idx >= 0) & (S_idx < p)]

    def _base_no_trade(case: str, reason: str, *, buy_idx=None, sell_idx=None, funding_idx=None):
        return {
            "no_trade": True,
            "no_trade_reason": reason,
            "dw_impl": np.zeros(p, dtype=float),
            "w_new": w_old.copy(),
            "implementation_case": case,
            "buy_idx": np.array([], dtype=int) if buy_idx is None else np.asarray(buy_idx, dtype=int),
            "sell_idx": np.array([], dtype=int) if sell_idx is None else np.asarray(sell_idx, dtype=int),
            "funding_idx": np.array([], dtype=int) if funding_idx is None else np.asarray(funding_idx, dtype=int),
            "candidate_threshold": candidate_threshold,
            "selected_min_confidence": selected_min_confidence,
            "legacy_min_names_to_trade": int(min_names_to_trade),
            "legacy_funding_min_confidence": float(funding_min_confidence),
        }

    def _cardinality_ok(w_new: np.ndarray) -> Tuple[bool, int, int, int]:
        nnz_old = int(np.count_nonzero(w_old > tol))
        nnz_new = int(np.count_nonzero(np.asarray(w_new, dtype=float) > tol))
        change = int(nnz_new - nnz_old)
        return abs(change) <= max_cardinality_change, nnz_old, nnz_new, change

    if S_idx.size == 0:
        return _base_no_trade(
            "empty_candidate_set",
            "selected ROPE candidate set is empty",
        )

    buy_idx_raw = S_idx[dw_map[S_idx] > tol]
    sell_idx = S_idx[dw_map[S_idx] < -tol]

    n_buy_raw = int(buy_idx_raw.size)
    n_sell = int(sell_idx.size)

    if n_buy_raw == 0 and n_sell == 0:
        return _base_no_trade(
            "zero_moves",
            "selected set contains no numerically nonzero MAP moves",
            buy_idx=buy_idx_raw,
            sell_idx=sell_idx,
        )

    # -----------------------------------------------------------
    # Case A: approved buys and approved sells.
    # Touch approved movers only.
    # -----------------------------------------------------------

    if n_buy_raw > 0 and n_sell > 0:
        buy_idx = buy_idx_raw
        buy_raw = dw_map[buy_idx].copy()
        sell_raw = -dw_map[sell_idx].copy()

        buy_total = float(np.sum(buy_raw))

        # Cap sells so that final weights are nonnegative.
        sell_capacity = np.maximum(w_old[sell_idx], 0.0)
        sell_feasible = np.minimum(sell_raw, sell_capacity)
        sell_total_feasible = float(np.sum(sell_feasible))

        if buy_total <= tol or sell_total_feasible <= tol:
            return _base_no_trade(
                "mixed_infeasible",
                "approved buy or feasible sell side is numerically zero",
                buy_idx=buy_idx,
                sell_idx=sell_idx,
            )

        q_trade = min(buy_total, sell_total_feasible)

        dw_impl[buy_idx] = q_trade * buy_raw / buy_total
        dw_impl[sell_idx] = -q_trade * sell_feasible / sell_total_feasible

        residual = float(dw_impl.sum())

        if abs(residual) > 1e-14:
            j_fix = buy_idx[np.argmax(np.abs(dw_impl[buy_idx]))]
            dw_impl[j_fix] -= residual

        dw_impl, w_new, repair_error = asoc_repair_negative_weights_locally(
            w_old=w_old,
            dw_impl=dw_impl,
            preferred_buy_idx=buy_idx,
            tol=tol,
        )

        if repair_error is not None:
            return _base_no_trade(
                "mixed_negative_repair_failed",
                repair_error,
                buy_idx=buy_idx,
                sell_idx=sell_idx,
            )

        card_ok, nnz_old, nnz_new, card_change = _cardinality_ok(w_new)

        if not card_ok:
            return _base_no_trade(
                "mixed_cardinality_change_too_large",
                (
                    f"implemented mixed move would change cardinality by "
                    f"{card_change:+d}, exceeding max_cardinality_change={max_cardinality_change}"
                ),
                buy_idx=buy_idx,
                sell_idx=sell_idx,
            )

        return {
            "no_trade": False,
            "no_trade_reason": "",
            "dw_impl": dw_impl,
            "w_new": w_new,
            "implementation_case": "approved_buys_and_sells_only",
            "buy_idx": buy_idx,
            "sell_idx": sell_idx,
            "funding_idx": np.array([], dtype=int),
            "q_trade": float(q_trade),
            "buy_total_raw": buy_total,
            "sell_total_raw": float(np.sum(sell_raw)),
            "sell_total_feasible": sell_total_feasible,
            "candidate_threshold": candidate_threshold,
            "selected_min_confidence": selected_min_confidence,
            "legacy_min_names_to_trade": int(min_names_to_trade),
            "legacy_funding_min_confidence": float(funding_min_confidence),
            "nnz_old": nnz_old,
            "nnz_new": nnz_new,
            "cardinality_change": card_change,
        }

    # -----------------------------------------------------------
    # Case B: approved buys only.
    # Fund locally from the smallest existing positive holdings.
    # -----------------------------------------------------------

    if n_buy_raw > 0 and n_sell == 0:
        existing_buy_idx = buy_idx_raw[w_old[buy_idx_raw] > tol]
        new_buy_idx = buy_idx_raw[w_old[buy_idx_raw] <= tol]

        if allow_new_buy_names and new_buy_idx.size > 0:
            # Keep only a bounded number of new openings. Use MAP move size
            # as a transparent priority rule because the decision object does
            # not need posterior samples at this stage.
            n_new_allowed = min(int(max_cardinality_change), int(new_buy_idx.size))

            if n_new_allowed > 0:
                order = np.argsort(-np.abs(dw_map[new_buy_idx]))
                kept_new_buy_idx = new_buy_idx[order[:n_new_allowed]]
            else:
                kept_new_buy_idx = np.array([], dtype=int)
        else:
            kept_new_buy_idx = np.array([], dtype=int)

        buy_idx = np.unique(
            np.concatenate([existing_buy_idx, kept_new_buy_idx]).astype(int)
        )

        dropped_new_buy_idx = np.setdiff1d(new_buy_idx, kept_new_buy_idx)

        if buy_idx.size == 0:
            return _base_no_trade(
                "buy_only_no_allowed_buys",
                (
                    "approved buys were all new names and no new names were "
                    "allowed under the cardinality rule"
                ),
                buy_idx=buy_idx_raw,
                sell_idx=np.array([], dtype=int),
            )

        buy_raw = dw_map[buy_idx].copy()
        buy_total = float(np.sum(buy_raw))

        if buy_total <= tol:
            return _base_no_trade(
                "buy_only_zero",
                "approved buy side is numerically zero",
                buy_idx=buy_idx,
                sell_idx=np.array([], dtype=int),
            )

        funding_idx, funding_sells, budget_obtained = asoc_choose_small_weight_funders(
            w_old=w_old,
            excluded_idx=buy_idx,
            required_budget=buy_total,
            max_funding_names=max_buy_only_funding_names,
            close_small_funders=close_small_funders,
            dust_tol=buy_only_dust_tol,
            tol=tol,
        )

        if budget_obtained + tol < buy_total or funding_idx.size == 0:
            return _base_no_trade(
                "buy_only_no_funding_capacity",
                (
                    f"could not fund approved buys using at most "
                    f"{max_buy_only_funding_names} smallest existing holdings"
                ),
                buy_idx=buy_idx,
                sell_idx=np.array([], dtype=int),
                funding_idx=funding_idx,
            )

        dw_impl[buy_idx] = buy_raw
        dw_impl[funding_idx] = -funding_sells

        # If we deliberately close a small funding name and overfund by a
        # tiny amount, redistribute the excess across approved buys.
        total_sell = float(np.sum(funding_sells))
        excess_sell = total_sell - buy_total

        if excess_sell > tol:
            buy_weights = buy_raw / max(float(np.sum(buy_raw)), tol)
            dw_impl[buy_idx] += excess_sell * buy_weights

        residual = float(dw_impl.sum())

        if abs(residual) > 1e-14:
            # Final numerical budget correction. Since the trade is buy-only
            # plus funding sells, absorb tiny residual on the largest buy.
            j_fix = buy_idx[np.argmax(np.abs(dw_impl[buy_idx]))]
            dw_impl[j_fix] -= residual

        dw_impl, w_new, repair_error = asoc_repair_negative_weights_locally(
            w_old=w_old,
            dw_impl=dw_impl,
            preferred_buy_idx=buy_idx,
            tol=tol,
        )

        if repair_error is not None:
            return _base_no_trade(
                "buy_only_negative_repair_failed",
                repair_error,
                buy_idx=buy_idx,
                sell_idx=np.array([], dtype=int),
                funding_idx=funding_idx,
            )

        card_ok, nnz_old, nnz_new, card_change = _cardinality_ok(w_new)

        if not card_ok:
            return _base_no_trade(
                "buy_only_cardinality_change_too_large",
                (
                    f"implemented buy-only move would change cardinality by "
                    f"{card_change:+d}, exceeding max_cardinality_change={max_cardinality_change}"
                ),
                buy_idx=buy_idx,
                sell_idx=np.array([], dtype=int),
                funding_idx=funding_idx,
            )

        return {
            "no_trade": False,
            "no_trade_reason": "",
            "dw_impl": dw_impl,
            "w_new": w_new,
            "implementation_case": "buy_only_locally_funded",
            "buy_idx": buy_idx,
            "sell_idx": np.array([], dtype=int),
            "funding_idx": funding_idx,
            "funding_sells": np.asarray(funding_sells, dtype=float),
            "buy_idx_raw": buy_idx_raw,
            "new_buy_idx_raw": new_buy_idx,
            "kept_new_buy_idx": kept_new_buy_idx,
            "dropped_new_buy_idx": dropped_new_buy_idx,
            "buy_total_raw_all_candidates": float(np.sum(dw_map[buy_idx_raw])),
            "buy_total_implemented": float(np.sum(dw_impl[buy_idx])),
            "total_sell": float(np.sum(funding_sells)),
            "excess_sell_redistributed": float(max(excess_sell, 0.0)),
            "candidate_threshold": candidate_threshold,
            "selected_min_confidence": selected_min_confidence,
            "legacy_min_names_to_trade": int(min_names_to_trade),
            "legacy_funding_min_confidence": float(funding_min_confidence),
            "max_buy_only_funding_names": int(max_buy_only_funding_names),
            "close_small_funders": bool(close_small_funders),
            "buy_only_dust_tol": float(buy_only_dust_tol),
            "max_cardinality_change": int(max_cardinality_change),
            "allow_new_buy_names": bool(allow_new_buy_names),
            "nnz_old": nnz_old,
            "nnz_new": nnz_new,
            "cardinality_change": card_change,
        }

    # -----------------------------------------------------------
    # Case C: approved sells only.
    # No posterior-approved destination, so no rebalance.
    # -----------------------------------------------------------

    if n_buy_raw == 0 and n_sell > 0:
        return _base_no_trade(
            "sell_only_no_destination",
            "approved moves are sells only; no posterior-approved destination for freed budget",
            buy_idx=np.array([], dtype=int),
            sell_idx=sell_idx,
        )

    raise RuntimeError("Unhandled rebalance implementation case.")


# ---------------------------------------------------------------
# 4) Decision wrapper from ROPE output
# ---------------------------------------------------------------

def build_asoc_locked_rebalance_decision_from_rope(
    *,
    cgrid_out: Dict[str, Any],
    mala_out: Dict[str, Any],
    rope_out: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
) -> Dict[str, Any]:
    """
    Build a Block-13E-style rebalance decision from the selected
    ROPE / directional Delta-w candidate.
    """
    context = cgrid_out["context"]

    y_fit = np.asarray(context["y_fit"], dtype=float).reshape(-1)
    R_fit = np.asarray(context["R_fit"], dtype=float)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    dw_map = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)

    tickers = get_asset_tickers_from_context_asoc(context)

    selected_name = str(rope_out["selected_candidate_name"])
    selected_candidate = rope_out["selected_candidate"]

    S_idx = np.asarray(selected_candidate["S_idx"], dtype=int).reshape(-1)

    selected_min_confidence = selected_candidate.get("min_directional_confidence", np.nan)
    candidate_threshold = selected_candidate.get("pi_move", np.nan)

    try:
        selected_min_confidence = float(selected_min_confidence)
    except Exception:
        selected_min_confidence = np.nan

    try:
        candidate_threshold = float(candidate_threshold)
    except Exception:
        candidate_threshold = np.nan

    impl = asoc_implement_locked_rebalance(
        w_old=w_old,
        dw_map=dw_map,
        S_idx=S_idx,
        selected_min_confidence=selected_min_confidence,
        candidate_threshold=candidate_threshold,
        min_names_to_trade=config.min_names_to_trade,
        funding_min_confidence=config.funding_min_confidence,
        max_buy_only_funding_names=config.max_buy_only_funding_names,
        close_small_funders=config.close_small_funders,
        buy_only_dust_tol=config.buy_only_dust_tol,
        max_cardinality_change=config.max_cardinality_change,
        allow_new_buy_names=config.allow_new_buy_names,
        tol=1e-12,
    )

    dw_impl = np.asarray(impl["dw_impl"], dtype=float).reshape(-1)
    w_new = np.asarray(impl["w_new"], dtype=float).reshape(-1)

    te_old = realised_te_rmse_raw(y_fit, R_fit, w_old)
    te_map_proposed = realised_te_rmse_raw(y_fit, R_fit, w_old + dw_map)
    te_new = realised_te_rmse_raw(y_fit, R_fit, w_new)

    old_structure = asoc_portfolio_structure_summary(w_old)
    new_structure = asoc_portfolio_structure_summary(w_new)
    map_trade_summary = asoc_trade_summary_from_delta(dw_map)
    implemented_trade_summary = asoc_trade_summary_from_delta(dw_impl)

    # Candidate-set diagnostics.
    if S_idx.size > 0:
        df_candidate = pd.DataFrame(
            {
                "j": S_idx,
                "ticker": [tickers[j] for j in S_idx],
                "w_old": w_old[S_idx],
                "dw_MAP": dw_map[S_idx],
                "w_old_plus_dw_MAP": w_old[S_idx] + dw_map[S_idx],
                "dw_impl": dw_impl[S_idx],
                "w_new": w_new[S_idx],
                "abs_dw_MAP": np.abs(dw_map[S_idx]),
            }
        ).sort_values("abs_dw_MAP", ascending=False).reset_index(drop=True)
    else:
        df_candidate = pd.DataFrame(
            columns=[
                "j",
                "ticker",
                "w_old",
                "dw_MAP",
                "w_old_plus_dw_MAP",
                "dw_impl",
                "w_new",
                "abs_dw_MAP",
            ]
        )

    # Implemented trade diagnostics.
    trade_idx = np.where(np.abs(dw_impl) > 1e-12)[0]

    if trade_idx.size > 0:
        buy_idx = np.asarray(impl.get("buy_idx", np.array([], dtype=int)), dtype=int)
        sell_idx = np.asarray(impl.get("sell_idx", np.array([], dtype=int)), dtype=int)
        funding_idx = np.asarray(impl.get("funding_idx", np.array([], dtype=int)), dtype=int)

        roles = []

        for j in trade_idx:
            if j in buy_idx:
                roles.append("approved_buy")
            elif j in sell_idx:
                roles.append("approved_sell")
            elif j in funding_idx:
                roles.append("funding_sell")
            else:
                roles.append("other")

        df_trades = pd.DataFrame(
            {
                "j": trade_idx,
                "ticker": [tickers[j] for j in trade_idx],
                "role": roles,
                "w_old": w_old[trade_idx],
                "dw_impl": dw_impl[trade_idx],
                "w_new": w_new[trade_idx],
                "dw_MAP": dw_map[trade_idx],
                "abs_dw_impl": np.abs(dw_impl[trade_idx]),
            }
        ).sort_values("abs_dw_impl", ascending=False).reset_index(drop=True)
    else:
        df_trades = pd.DataFrame(
            columns=[
                "j",
                "ticker",
                "role",
                "w_old",
                "dw_impl",
                "w_new",
                "dw_MAP",
                "abs_dw_impl",
            ]
        )

    decision = {
        "fit_k": int(context.get("hold_number", 0)),
        "hold_number": int(context.get("hold_number", 0)),
        "candidate_name": selected_name,
        "candidate_threshold": candidate_threshold,
        "selected_min_confidence": selected_min_confidence,
        "funding_min_confidence": float(config.funding_min_confidence),
        "max_buy_only_funding_names": int(config.max_buy_only_funding_names),
        "buy_only_dust_tol": float(config.buy_only_dust_tol),
        "max_cardinality_change": int(config.max_cardinality_change),
        "allow_new_buy_names": bool(config.allow_new_buy_names),
        "S_idx": S_idx,
        "S_size": int(S_idx.size),
        "S_tickers": [tickers[j] for j in S_idx],
        "candidate_meta": selected_candidate,
        "dw_map": dw_map.copy(),
        "w_old": w_old.copy(),
        "dw_impl": dw_impl.copy(),
        "w_new": w_new.copy(),
        "w_rebalanced": w_new.copy(),
        "no_trade": bool(impl["no_trade"]),
        "no_trade_reason": str(impl["no_trade_reason"]),
        "implementation_case": str(impl["implementation_case"]),
        "implementation_details": impl,
        "te_fit2": {
            "old": float(te_old),
            "map_proposed": float(te_map_proposed),
            "new": float(te_new),
            "map_improvement_vs_old": float(te_old - te_map_proposed),
            "implemented_improvement_vs_old": float(te_old - te_new),
        },
        "old_structure": old_structure,
        "new_structure": new_structure,
        "map_trade_summary": map_trade_summary,
        "implemented_trade_summary": implemented_trade_summary,
        "df_candidate_set": df_candidate,
        "df_implemented_trades": df_trades,
        "cgrid_out": cgrid_out,
        "mala_out": mala_out,
        "rope_out": rope_out,
    }

    if config.verbose:
        print("\n[Local implementation | locked rebalance decision]")
        print(f"Hold transition before Hold {context['hold_number']}")
        print(f"candidate_name = {selected_name}")
        print(f"|S_candidate| = {S_idx.size}")
        print(f"candidate eps = {selected_candidate.get('eps', np.nan):.1e}")
        print(f"candidate pi_move = {candidate_threshold:.2f}")
        print(f"selected min confidence = {selected_min_confidence:.3f}")
        print(
            "buy-only local funding: "
            f"max funders={config.max_buy_only_funding_names}, "
            f"dust_tol={config.buy_only_dust_tol:.1e}, "
            f"max cardinality change={config.max_cardinality_change}"
        )
        print(f"legacy funding_min_confidence = {config.funding_min_confidence:.2f} (not a buy-only gate)")
        print(f"implementation case = {decision['implementation_case']}")

        if decision["no_trade"]:
            print("\nDecision: NO REBALANCE")
            print(f"Reason: {decision['no_trade_reason']}")
        else:
            print("\nDecision: REBALANCE")
            print(f"Implemented trades = {implemented_trade_summary['nnz_trades']}")
            print(f"Turnover = {implemented_trade_summary['turnover_half_L1']:.6e}")
            print(
                f"Increases = {implemented_trade_summary['n_increases']} | "
                f"decreases = {implemented_trade_summary['n_decreases']}"
            )

        print("\n[Realised TE on decision FIT window]")
        print(f"old held Tracker:       {te_old:.6e} ({1e4 * te_old:.3f} bp)")
        print(
            f"MAP proposal w_old+dw:  {te_map_proposed:.6e} "
            f"({1e4 * te_map_proposed:.3f} bp)"
        )
        print(f"implemented Tracker:    {te_new:.6e} ({1e4 * te_new:.3f} bp)")
        print(f"implemented improvement: {te_old - te_new:+.6e}")

        print("\n[Budget / structure checks]")
        print(f"sum(w_old) = {old_structure['sum_w']:+.12f}")
        print(f"sum(w_new) = {new_structure['sum_w']:+.12f}")
        print(f"sum(dw_impl) = {implemented_trade_summary['sum_dw']:+.3e}")
        print(f"nnz(w_old) = {old_structure['nnz_exact']}/{context['p']}")
        print(f"nnz(w_new) = {new_structure['nnz_exact']}/{context['p']}")
        print(f"shorts(w_new) = {new_structure['n_shorts']}")
        print(f"min(w_new) = {new_structure['min_w']:+.6e}")

        if S_idx.size > 0:
            print("\n=== Candidate set diagnostics ===")
            df_disp = df_candidate.copy()

            for col in [
                "w_old",
                "dw_MAP",
                "w_old_plus_dw_MAP",
                "dw_impl",
                "w_new",
                "abs_dw_MAP",
            ]:
                df_disp[col] = df_disp[col].map(lambda x: f"{float(x):+.4e}")

            print(df_disp.to_string(index=False))

        if not df_trades.empty:
            print("\n=== Implemented trades ===")
            df_trade_disp = df_trades.copy()

            for col in ["w_old", "dw_impl", "w_new", "dw_MAP", "abs_dw_impl"]:
                df_trade_disp[col] = df_trade_disp[col].map(lambda x: f"{float(x):+.4e}")

            print(df_trade_disp.to_string(index=False))

        print("\n[Local implementation] Block-13E-style rebalance decision created.")

    return decision


print("[Local implementation] Locked rebalance decision builder defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — maintenance wrapper
# Full rebalance_step_fn_asoc wrapper
#
# Purpose:
#   Connect the sequential experiment driver to the
#   cleaned rebalancing sub-pipeline from Blocks 14D-1--14D-4b.
#
# This callable is used before Hold h >= 2:
#
#   decision = rebalance_step_fn_asoc(
#       hold_number=h,
#       window=rolling_windows[h - 1],
#       previous_w=current_tracker,
#       experiment_state=experiment_state,
#   )
#
# It performs:
#   1. fit-window preparation;
#   2. Delta-w c-grid;
#   3. Delta-w MALA/UQ;
#   4. ROPE / directional practical-change candidate construction;
#   5. locked local-budget rebalance decision.
#
# Output:
#   A maintenance-driver-style decision dictionary compatible with the sequential driver.
# ===============================================================

from __future__ import annotations

from typing import Dict, Any, Optional
import numpy as np


# ---------------------------------------------------------------
# 0) RNG resolver
# ---------------------------------------------------------------

def get_rng_for_asoc_rebalance(
    experiment_state: Dict[str, Any],
    *,
    seed: int = 0,
) -> np.random.Generator:
    """
    Resolve an RNG for the rebalancing wrapper.

    Priority:
      1. experiment_state["rng"], if present;
      2. global rng, if present;
      3. new default_rng(seed).
    """
    if "rng" in experiment_state and isinstance(
        experiment_state["rng"],
        np.random.Generator,
    ):
        return experiment_state["rng"]

    if "rng" in globals() and isinstance(globals()["rng"], np.random.Generator):
        return globals()["rng"]

    return np.random.default_rng(seed)


# ---------------------------------------------------------------
# 1) Main wrapper
# ---------------------------------------------------------------

def rebalance_step_fn_asoc(
    *,
    hold_number: int,
    window: dict,
    previous_w: np.ndarray,
    experiment_state: Dict[str, Any],
    config: ASOCRebalanceConfig = asoc_rebalance_config,
    sapg_n_iter: int = 15_000,
    fista_max_iter: int = 4_000,
    selected_rope_candidate: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Full ASOC rebalancing wrapper for the sequential experiment.

    Parameters
    ----------
    hold_number : int
        Hold number that is about to be entered. For example, this function
        is called before Hold 2 with hold_number=2.
    window : dict
        The rolling-window record associated with this hold.
    previous_w : np.ndarray
        The Tracker currently held before this rebalance decision.
    experiment_state : dict
        State dictionary created by run_sequential_tracker_experiment.
    config : ASOCRebalanceConfig
        Rebalancing configuration.
    sapg_n_iter : int
        SAPG iterations per c-grid row.
    fista_max_iter : int
        FISTA iterations per c-grid row.
    selected_rope_candidate : Optional[str]
        Override for config.selected_rope_candidate.

    Returns
    -------
    decision : dict
        Block-13E-style decision dictionary containing at least:
          - w_new
          - dw_impl
          - no_trade
          - no_trade_reason
          - implementation_case
          - implemented_trade_summary
          - te_fit2
    """
    rng_local = get_rng_for_asoc_rebalance(experiment_state)

    if config.verbose:
        print("\n" + "#" * 78)
        print(f"[Maintenance wrapper] ASOC rebalance wrapper before Hold {hold_number}")
        print("#" * 78)

    # -----------------------------------------------------------
    # 1. Prepare fit-window context
    # -----------------------------------------------------------

    context = prepare_rebalance_context_asoc(
        hold_number=hold_number,
        window=window,
        previous_w=previous_w,
        experiment_state=experiment_state,
        config=config,
    )

    # -----------------------------------------------------------
    # 2. Run Delta-w c-grid
    # -----------------------------------------------------------

    cgrid_out = run_delta_w_cgrid_asoc(
        context=context,
        config=config,
        sapg_n_iter=sapg_n_iter,
        fista_max_iter=fista_max_iter,
        rng=rng_local,
    )

    # -----------------------------------------------------------
    # 3. Run Delta-w MALA/UQ
    # -----------------------------------------------------------

    mala_out = run_delta_w_mala_uq_asoc(
        cgrid_out=cgrid_out,
        config=config,
        rng=rng_local,
    )

    # -----------------------------------------------------------
    # 4. Build ROPE / directional practical-change candidates
    # -----------------------------------------------------------

    rope_candidate_name = (
        selected_rope_candidate
        if selected_rope_candidate is not None
        else config.selected_rope_candidate
    )

    rope_out = build_delta_w_rope_candidates_asoc(
        cgrid_out=cgrid_out,
        mala_out=mala_out,
        config=config,
        selected_candidate_name=rope_candidate_name,
    )

    # -----------------------------------------------------------
    # 5. Apply locked local-budget decision rule
    # -----------------------------------------------------------

    decision = build_asoc_locked_rebalance_decision_from_rope(
        cgrid_out=cgrid_out,
        mala_out=mala_out,
        rope_out=rope_out,
        config=config,
    )

    # -----------------------------------------------------------
    # 6. Add wrapper-level metadata
    # -----------------------------------------------------------

    decision["wrapper_metadata"] = {
        "hold_number": int(hold_number),
        "sapg_n_iter": int(sapg_n_iter),
        "fista_max_iter": int(fista_max_iter),
        "selected_rope_candidate": rope_candidate_name,
        "sigma2_dw_base": float(config.sigma2_dw_base),
        "c_grid_dw": list(map(float, config.c_grid_dw)),
        "min_names_to_trade": int(config.min_names_to_trade),
        "funding_min_confidence": float(config.funding_min_confidence),
        "score_te_improvement_target": float(config.score_te_improvement_target),
        "score_l1_dw_target": float(config.score_l1_dw_target),
        "score_l1_dw_scale": float(config.score_l1_dw_scale),
        "max_buy_only_funding_names": int(config.max_buy_only_funding_names),
        "close_small_funders": bool(config.close_small_funders),
        "buy_only_dust_tol": float(config.buy_only_dust_tol),
        "max_cardinality_change": int(config.max_cardinality_change),
        "allow_new_buy_names": bool(config.allow_new_buy_names),
    }

    # Store per-hold audit history. Full MALA samples are stripped after the decision layer has used them.
    if "rebalance_history" not in experiment_state:
        experiment_state["rebalance_history"] = {}

    experiment_state["rebalance_history"][int(hold_number)] = {
        "context": context,
        "cgrid_out": cgrid_out,
        "mala_out": mala_out,
        "rope_out": rope_out,
        "decision": decision,
    }

    if config.verbose:
        print("\n[Maintenance wrapper] Rebalance wrapper complete.")
        print(f"  hold_number = {hold_number}")
        print(f"  no_trade = {decision['no_trade']}")
        print(f"  implementation_case = {decision['implementation_case']}")
        print(
            "  turnover = "
            f"{decision['implemented_trade_summary']['turnover_half_L1']:.6e}"
        )
        print(
            "  trades = "
            f"{decision['implemented_trade_summary']['nnz_trades']}"
        )

    return decision


print("[Maintenance wrapper] rebalance_step_fn_asoc defined.")

In [ ]:
# ===============================================================
# ASOC Bayesian Index Tracking — paper main sequential maintenance run
# Tracker: Notebook-1 paper branch, 59 names before the one-off floor and 53 after
# Holds: H1--H16 inclusive
#
# Purpose:
#   One-off 0.5% implementability floor at H1 followed by controlled recovery with budgeted buy-dominant funding top-up. Calm mode uses the ordinary posterior-directional gate; recovery mode uses asymmetric gates (existing positions p_dir>=0.525; new buys p_dir>=0.575; sells p_dir>=0.525), ranks by p_dir*|Delta w_MAP|, truncates to 10 moves and 4 new buy names, targets a 1.0% one-way recovery repair when a buy-dominant set is approved, keeps 2% as a hard cap, and may top up funding from weak existing holdings.
#
# Interpretation:
#   The cached Tracker is used as the previous portfolio at the start
#   of the local window. By default, the calm-mode rebalancing rule is
#   not allowed to act before H1; H1 is the first realised evaluation of
#   the construction-floor-adjusted cached Tracker. Set
#   REBALANCE_ENTERING_START_HOLD=True only for local diagnostic windows.
#
# Outputs:
#   - final hold/decision/path CSVs written once after the run;
#   - per-hold full composition CSV;
#   - per-hold top-holdings CSV;
#   - per-hold composition summary CSV.
# ===============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

import re
import shutil

# ---------------------------------------------------------------
# Variant-specific dust/floor helpers
# ---------------------------------------------------------------

def _asoc_weight_structure_for_logs(w, *, tol=1e-12):
    """Small portfolio-structure summary used before/after floor actions."""
    w = np.asarray(w, dtype=float).reshape(-1)
    abs_w = np.abs(w)
    l1 = float(abs_w.sum())
    pos = w[w > tol]
    order = np.argsort(-abs_w)
    return {
        "sum_w": float(w.sum()),
        "nnz": int(np.count_nonzero(abs_w > tol)),
        "nnz_ge_0p5pct": int(np.count_nonzero(w >= 0.005 - tol)),
        "nnz_ge_1pct": int(np.count_nonzero(w >= 0.010 - tol)),
        "n_negative": int(np.count_nonzero(w < -tol)),
        "min_positive": float(np.min(pos)) if pos.size else np.nan,
        "top10_L1_share": float(abs_w[order[:10]].sum() / l1) if l1 > 0 else np.nan,
        "top25_L1_share": float(abs_w[order[:25]].sum() / l1) if l1 > 0 else np.nan,
        "HHI_abs": float(np.sum((abs_w / l1) ** 2)) if l1 > 0 else np.nan,
    }


def apply_min_weight_floor_once_logged(
    w,
    asset_cols,
    *,
    floor,
    hold,
    variant_label,
    action_stage="initial_prune",
    realloc="proportional_survivors",
    tol=1e-12,
):
    """
    One-off implementability-floor action: prune positive holdings below `floor`
    and explicitly reallocate their mass to surviving positive support.

    This is the only place in these variants where redistribution across
    surviving holdings is allowed; it is logged as an implementability-floor action,
    not as a recurring rebalance operation.
    """
    w_old = np.asarray(w, dtype=float).reshape(-1)
    w_new = w_old.copy()
    asset_cols = list(asset_cols)
    floor = float(floor)
    rows = []

    before = _asoc_weight_structure_for_logs(w_new, tol=tol)

    prune_mask = (w_new > tol) & (w_new < floor)
    prune_idx = np.where(prune_mask)[0]
    mass = float(w_new[prune_idx].sum())

    for j in prune_idx:
        rows.append({
            "variant": variant_label,
            "hold": int(hold),
            "action_stage": action_stage,
            "ticker": asset_cols[j],
            "old_weight": float(w_old[j]),
            "new_weight": 0.0,
            "delta_weight": float(-w_old[j]),
            "reason": "below_floor_pruned",
            "floor": floor,
            "realloc": realloc,
            "mass_pruned_total": mass,
        })
        w_new[j] = 0.0

    if mass > tol:
        survivor_idx = np.where(w_new > tol)[0]
        if survivor_idx.size == 0:
            raise ValueError("No surviving holdings available to receive pruned mass.")

        if realloc != "proportional_survivors":
            raise ValueError(f"Unsupported realloc={realloc!r}; only proportional_survivors is implemented.")

        denom = float(w_new[survivor_idx].sum())
        inc = mass * w_new[survivor_idx] / denom
        old_survivor_weights = w_new[survivor_idx].copy()
        w_new[survivor_idx] += inc

        for j, old_j, inc_j in zip(survivor_idx, old_survivor_weights, inc):
            if abs(float(inc_j)) > tol:
                rows.append({
                    "variant": variant_label,
                    "hold": int(hold),
                    "action_stage": action_stage,
                    "ticker": asset_cols[j],
                    "old_weight": float(old_j),
                    "new_weight": float(w_new[j]),
                    "delta_weight": float(inc_j),
                    "reason": "redistributed_pruned_mass_proportional_survivors",
                    "floor": floor,
                    "realloc": realloc,
                    "mass_pruned_total": mass,
                })

    # Local numerical budget correction on the largest survivor only.
    residual = float(w_old.sum() - w_new.sum())
    if abs(residual) > 1e-14:
        survivor_idx = np.where(w_new > tol)[0]
        if survivor_idx.size == 0:
            raise ValueError("No survivor available for numerical budget correction.")
        j_fix = int(survivor_idx[np.argmax(w_new[survivor_idx])])
        old_j = float(w_new[j_fix])
        w_new[j_fix] += residual
        rows.append({
            "variant": variant_label,
            "hold": int(hold),
            "action_stage": action_stage,
            "ticker": asset_cols[j_fix],
            "old_weight": old_j,
            "new_weight": float(w_new[j_fix]),
            "delta_weight": residual,
            "reason": "local_numerical_budget_correction",
            "floor": floor,
            "realloc": realloc,
            "mass_pruned_total": mass,
        })

    w_new[np.abs(w_new) < 1e-15] = 0.0
    after = _asoc_weight_structure_for_logs(w_new, tol=tol)

    print("\n[One-off floor action]")
    print(f"variant          = {variant_label}")
    print(f"floor            = {floor:.4f}")
    print(f"pruned names     = {len(prune_idx)}")
    print(f"mass pruned      = {mass:.6e}")
    print(f"sum before/after = {before['sum_w']:.12f} / {after['sum_w']:.12f}")
    print(f"nnz before/after = {before['nnz']} / {after['nnz']}")
    print(f"min pos before   = {before['min_positive']:.6e}")
    print(f"min pos after    = {after['min_positive']:.6e}")
    print(f"one-way turnover = {0.5 * np.sum(np.abs(w_new - w_old)):.6e}")

    return w_new, pd.DataFrame(rows), {"before": before, "after": after, "mass_pruned": mass}


def build_small_weight_summary(seq_out, *, run_label, thresholds=(0.005, 0.010, 0.015), tol=1e-12):
    """Per-hold implementability and small-weight summary."""
    rows = []
    for hold, w in sorted(seq_out["weights_by_hold"].items()):
        w = np.asarray(w, dtype=float).reshape(-1)
        abs_w = np.abs(w)
        l1 = float(abs_w.sum())
        pos = w[w > tol]
        pos_abs = np.abs(pos)
        order = np.argsort(-abs_w)
        row = {
            "run_label": run_label,
            "hold": int(hold),
            "nnz": int(np.count_nonzero(abs_w > tol)),
            "sum_weights": float(w.sum()),
            "n_negative": int(np.count_nonzero(w < -tol)),
            "min_weight_pct": 100.0 * float(np.min(pos_abs)) if pos_abs.size else np.nan,
            "p05_weight_pct": 100.0 * float(np.quantile(pos_abs, 0.05)) if pos_abs.size else np.nan,
            "p10_weight_pct": 100.0 * float(np.quantile(pos_abs, 0.10)) if pos_abs.size else np.nan,
            "p25_weight_pct": 100.0 * float(np.quantile(pos_abs, 0.25)) if pos_abs.size else np.nan,
            "median_weight_pct": 100.0 * float(np.quantile(pos_abs, 0.50)) if pos_abs.size else np.nan,
            "n_lt_0p5pct": int(np.count_nonzero((w > tol) & (w < 0.005))),
            "n_lt_1pct": int(np.count_nonzero((w > tol) & (w < 0.010))),
            "n_lt_1p5pct": int(np.count_nonzero((w > tol) & (w < 0.015))),
            "top10_L1_share": float(abs_w[order[:10]].sum() / l1) if l1 > 0 else np.nan,
            "top25_L1_share": float(abs_w[order[:25]].sum() / l1) if l1 > 0 else np.nan,
            "HHI_abs": float(np.sum((abs_w / l1) ** 2)) if l1 > 0 else np.nan,
        }
        rows.append(row)
    return pd.DataFrame(rows)


def extract_dust_funding_action_rows(seq_out, *, run_label, dust_floor=0.005, tol=1e-12):
    """Extract explicit funding-use rows from implemented trade diagnostics."""
    asset_cols_local = list(seq_out.get("asset_cols", []))
    rows = []
    for hold, decision in sorted(seq_out.get("decisions_by_hold", {}).items()):
        if not isinstance(decision, dict):
            continue
        df_trades = decision.get("df_implemented_trades", pd.DataFrame())
        if df_trades is None or len(df_trades) == 0:
            continue
        for _, r in df_trades.iterrows():
            if str(r.get("role", "")) != "funding_sell":
                continue
            old_w = float(r.get("w_old", np.nan))
            new_w = float(r.get("w_new", np.nan))
            delta = float(r.get("dw_impl", np.nan))
            reason = "dust_priority_funding" if old_w < dust_floor else "fallback_smallest_positive_funding"
            rows.append({
                "variant": run_label,
                "hold": int(hold),
                "action_stage": "funding_use",
                "ticker": str(r.get("ticker", "")),
                "old_weight": old_w,
                "new_weight": new_w,
                "delta_weight": delta,
                "reason": reason,
                "floor": float(dust_floor),
                "realloc": "local_buy_only_funding",
                "mass_pruned_total": np.nan,
            })
    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# 0) User-facing run controls
# ---------------------------------------------------------------

HOLD_START_NUMBER = int(HOLD_START_NUMBER)
HOLD_END_NUMBER = len(rolling_windows) if HOLD_END_NUMBER is None else int(HOLD_END_NUMBER)

# Full-horizon convention: no extra rebalance before H1.
# The cached Tracker generated by Notebook 1 is held through H1.
REBALANCE_ENTERING_START_HOLD = bool(REBALANCE_ENTERING_START_HOLD)
MAKE_PLOTS = bool(MAKE_PLOTS)


# Variant controls
VARIANT_KIND = "floor005_recoveryAsymP0525New0575_target1pct_turn2pct_fundingTopup_budgeted"
VARIANT_DESCRIPTION = "Sequential maintenance starting from the floor-applied tracker cache generated by Notebook 1. Calm mode uses the ordinary posterior-directional gate; recovery mode uses asymmetric gates (existing positions p_dir>=0.525; new buys p_dir>=0.575; sells p_dir>=0.525), ranks by p_dir*|Delta w_MAP|, truncates to 10 moves and 4 new buy names, targets a 1.0% one-way recovery repair when a buy-dominant set is approved, keeps 2% as a hard cap, and may top up funding from weak existing holdings."
DUST_FLOOR = 0.005
# Notebook 1 exports the floor-applied tracker by default.
# Do not reapply the initial floor in Notebook 2 unless you deliberately
# load an unfloored cache and change this flag.
APPLY_INITIAL_FLOOR = False
INITIAL_FLOOR = 0.005 if APPLY_INITIAL_FLOOR else None
INITIAL_REALLOC = "proportional_survivors"

# Variant controls below describe the public paper run.
# Numerical recovery parameters are defined only in the top configuration cell.
TRACKER_RUN_SHORT_LABEL = "tracker_prefloor59_floor53"
WINDOW_LABEL = f"H{HOLD_START_NUMBER:02d}_H{HOLD_END_NUMBER:02d}"
run_label_seq = "paper_main_tracker_maintenance"
run_output_dir_seq = Path(OUTPUT_DIR) / "maintenance" / run_label_seq
figdir_seq = run_output_dir_seq / "figures"
composition_dir_seq = run_output_dir_seq / "compositions"
run_output_dir_seq.mkdir(parents=True, exist_ok=True)
composition_dir_seq.mkdir(parents=True, exist_ok=True)

print("[Paper main tracker maintenance run]")
print(f"tracker                 = {TRACKER_RUN_SHORT_LABEL}")
print(f"tracker_cache_label     = {tracker_cache_label}")
print(f"tracker_cache_path      = {TRACKER_CACHE_PATH}")
print(f"hold window             = H{HOLD_START_NUMBER:02d}--H{HOLD_END_NUMBER:02d}")
print(f"rebalance entering start hold = {REBALANCE_ENTERING_START_HOLD}")
print(f"variant kind            = {VARIANT_KIND}")
print(f"recovery p_dir legacy   = {RECOVERY_P_DIR:.3f}")
print(f"p_dir existing/sell     = {RECOVERY_P_DIR_EXISTING:.3f} / {RECOVERY_P_DIR_SELL:.3f}")
print(f"p_dir new buy           = {RECOVERY_P_DIR_NEW_BUY:.3f}")
print(f"recovery turnover cap   = {RECOVERY_MAX_ONE_WAY_TURNOVER:.3%}")
print(f"recovery target turnover= {RECOVERY_TARGET_ONE_WAY_TURNOVER:.3%}")
print(f"funding top-up enabled  = {RECOVERY_BUY_TOPUP_ENABLED}")
print(f"extra funding names max = {RECOVERY_MAX_EXTRA_FUNDING_NAMES}")
print(f"protect top-N funders   = {RECOVERY_PROTECT_TOP_N_FUNDERS}")
print(f"variant description     = {VARIANT_DESCRIPTION}")
print(f"run_label_seq          = {run_label_seq}")
print(f"run_output_dir_seq     = {run_output_dir_seq}")
print(f"composition_dir_seq    = {composition_dir_seq}")
print(f"initial nnz             = {np.count_nonzero(np.abs(initial_tracker_w) > 1e-12)}")
print(f"initial sum             = {np.sum(initial_tracker_w):+.12f}")
print(
    f"initial_fit_te_target   = {initial_fit_te_target:.6e} "
    f"({1e4 * initial_fit_te_target:.3f} bp)"
)

# Guard against accidentally mixing smoke-mode and production-mode artefacts.
cache_run_mode = str(tracker_cache_metadata.get("run_mode", "unknown")) if isinstance(tracker_cache_metadata, dict) else "unknown"
if RUN_PRODUCTION and cache_run_mode not in {"production", "unknown"}:
    raise RuntimeError(
        "RUN_PRODUCTION=True but the loaded Notebook-1 cache metadata reports "
        f"run_mode={cache_run_mode!r}. Re-run Notebook 1 with RUN_PRODUCTION=True."
    )
if (not RUN_PRODUCTION) and cache_run_mode == "production":
    print("[Mode check] Loading a production cache while this notebook is in smoke mode.")

if HOLD_START_NUMBER < 1 or HOLD_END_NUMBER < HOLD_START_NUMBER:
    raise ValueError("Require 1 <= HOLD_START_NUMBER <= HOLD_END_NUMBER.")

if HOLD_END_NUMBER > len(rolling_windows):
    raise ValueError(
        f"HOLD_END_NUMBER={HOLD_END_NUMBER} exceeds len(rolling_windows)={len(rolling_windows)}."
    )



# ---------------------------------------------------------------
# 0b) Variant-specific initial portfolio construction
# ---------------------------------------------------------------

initial_w_for_local_run = np.asarray(initial_tracker_w, dtype=float).reshape(-1).copy()
initial_floor_actions_df = pd.DataFrame()
initial_floor_diagnostics = {}

if APPLY_INITIAL_FLOOR:
    initial_w_for_local_run, initial_floor_actions_df, initial_floor_diagnostics = apply_min_weight_floor_once_logged(
        initial_w_for_local_run,
        asset_cols_for_seq,
        floor=float(INITIAL_FLOOR),
        hold=HOLD_START_NUMBER,
        variant_label=run_label_seq,
        action_stage="initial_prune",
        realloc=INITIAL_REALLOC,
    )
else:
    print("\n[Initial portfolio] No one-off floor pruning is applied for this variant.")

print("\n[Initial portfolio after variant construction]")
print(f"sum                 = {initial_w_for_local_run.sum():+.12f}")
print(f"nnz                 = {np.count_nonzero(np.abs(initial_w_for_local_run) > 1e-12)}")
print(f"n >= 0.5%           = {np.count_nonzero(initial_w_for_local_run >= 0.005 - 1e-12)}")
print(f"n >= 1.0%           = {np.count_nonzero(initial_w_for_local_run >= 0.010 - 1e-12)}")
print(f"turnover vs cache   = {0.5 * np.sum(np.abs(initial_w_for_local_run - initial_tracker_w)):.6e}")


# ---------------------------------------------------------------
# 1) Base calm-mode rebalancing configuration plus controlled recovery wrapper
# ---------------------------------------------------------------

asoc_rebalance_config_calm = ASOCRebalanceConfig(
    sigma2_dw_base=SIGMA2_DW_BASE,
    c_grid_dw=tuple(C_GRID_DW),
    nnz_eff_thresh_dw=NNZ_EFF_THRESH_DW,
    nnz_target_fraction_of_old=0.25,
    selected_rope_candidate=SELECTED_ROPE_CANDIDATE,
    min_names_to_trade=2,
    funding_min_confidence=0.80,  # legacy diagnostic; not a buy-only gate in the revised rule
    score_te_improvement_target=SCORE_TE_IMPROVEMENT_TARGET,
    score_l1_dw_target=SCORE_L1_DW_TARGET,
    score_l1_dw_scale=SCORE_L1_DW_SCALE,
    max_buy_only_funding_names=3,
    close_small_funders=True,
    buy_only_dust_tol=1e-5,
    max_cardinality_change=3,
    allow_new_buy_names=True,

    # Long-run UQ settings for the sequential maintenance run.
    n_burn_steps_dw=DW_MALA_N_BURN_STEPS,
    n_iter_long_dw=DW_MALA_N_ITER_LONG,
    thin_long_dw=DW_MALA_THIN,
    burn_kept_long_dw=DW_MALA_BURN_KEPT,
    max_lag_ess_dw=DW_MALA_MAX_LAG_ESS,

    verbose=VERBOSE,
)

print("\n[Paper main tracker maintenance run] Rebalance config:")
print(asoc_rebalance_config_calm)



# ---------------------------------------------------------------
# 1b) Controlled recovery rule helpers
# ---------------------------------------------------------------

from dataclasses import replace as dataclass_replace


def _safe_float_or_nan(x):
    """Convert x to float, returning NaN if conversion fails."""
    try:
        out = float(x)
    except Exception:
        out = np.nan
    return out


def compute_recovery_state_for_decision(
    hold_rows,
    *,
    hold_number: int,
    target_te_bp: float,
    target_gap_bp: float,
    shock_jump_bp: float,
    exit_gap_bp: float,
    p_recovery: float,
    max_turnover: float,
    max_moves: int,
    max_new_buys: int,
):
    """
    Decide whether the rebalance entering `hold_number` is in recovery mode.

    This uses only realised TE from already-completed holds in the current
    local run. It does not use competitor information or future hold outcomes.
    """
    if len(hold_rows) == 0:
        return {
            "hold": int(hold_number),
            "prev_completed_hold": np.nan,
            "prev_hold_te_bp": np.nan,
            "prev_prev_hold_te_bp": np.nan,
            "target_trigger": False,
            "jump_trigger": False,
            "recovery_active_for_decision": False,
            "recovery_anchor_bp": np.nan,
            "exit_condition_met": False,
            "p_recovery_used": float(p_recovery),
            "max_turnover_used": float(max_turnover),
            "max_moves_used": int(max_moves),
            "max_new_buys_used": int(max_new_buys),
            "reason": "no_completed_hold_yet",
        }

    prev = hold_rows[-1]
    prev_te = _safe_float_or_nan(prev.get("TE_bp", np.nan))
    prev_hold = int(prev.get("hold", int(hold_number) - 1))

    if len(hold_rows) >= 2:
        prev_prev = hold_rows[-2]
        prev_prev_te = _safe_float_or_nan(prev_prev.get("TE_bp", np.nan))
    else:
        prev_prev_te = np.nan

    target_trigger = bool(np.isfinite(prev_te) and prev_te >= float(target_te_bp + target_gap_bp))
    jump_trigger = bool(
        np.isfinite(prev_te)
        and np.isfinite(prev_prev_te)
        and (prev_te - prev_prev_te) >= float(shock_jump_bp)
    )

    anchors = []
    if target_trigger:
        anchors.append(float(target_te_bp))
    if jump_trigger and np.isfinite(prev_prev_te):
        anchors.append(float(prev_prev_te))

    recovery_anchor = float(min(anchors)) if anchors else np.nan
    active_raw = bool(target_trigger or jump_trigger)
    exit_condition_met = bool(
        active_raw and np.isfinite(recovery_anchor) and prev_te <= recovery_anchor + float(exit_gap_bp)
    )
    recovery_active = bool(active_raw and not exit_condition_met)

    if recovery_active:
        reasons = []
        if target_trigger:
            reasons.append("target_trigger")
        if jump_trigger:
            reasons.append("jump_trigger")
        reason = "+".join(reasons)
    elif exit_condition_met:
        reason = "exit_condition_met"
    else:
        reason = "calm_mode"

    return {
        "hold": int(hold_number),
        "prev_completed_hold": int(prev_hold),
        "prev_hold_te_bp": float(prev_te),
        "prev_prev_hold_te_bp": float(prev_prev_te) if np.isfinite(prev_prev_te) else np.nan,
        "target_trigger": bool(target_trigger),
        "jump_trigger": bool(jump_trigger),
        "recovery_active_for_decision": bool(recovery_active),
        "recovery_anchor_bp": float(recovery_anchor) if np.isfinite(recovery_anchor) else np.nan,
        "exit_condition_met": bool(exit_condition_met),
        "p_recovery_used": float(p_recovery),
        "max_turnover_used": float(max_turnover),
        "max_moves_used": int(max_moves),
        "max_new_buys_used": int(max_new_buys),
        "reason": reason,
    }


def build_recovery_candidate_table_and_support(
    *,
    hold_number: int,
    cgrid_out: dict,
    mala_out: dict,
    p_recovery: float,
    eps_delta: float,
    max_moves: int,
    max_new_buys: int,
    dust_floor: float,
    p_existing: float,
    p_new_buy: float,
    p_sell: float,
    tol: float = 1e-12,
):
    """
    Build the recovery candidate table and truncate by p_dir * |Delta w_MAP|.

    Recovery-mode gate is asymmetric:
      * existing-position buys/increases require p_existing;
      * existing-position sells/decreases require p_sell;
      * new buy names require p_new_buy;
      * zero-weight sells are never approved.
    """
    context = cgrid_out["context"]
    tickers = get_asset_tickers_from_context_asoc(context)

    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    dw_map = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)
    W_post = np.asarray(mala_out["W_post"], dtype=float)

    if W_post.ndim != 2 or W_post.shape[1] != dw_map.size:
        raise ValueError("mala_out['W_post'] must have shape (n_samples, p).")

    P_pos0 = np.mean(W_post > 0.0, axis=0)
    P_neg0 = np.mean(W_post < 0.0, axis=0)

    is_buy = dw_map > tol
    is_sell = dw_map < -tol
    is_existing = w_old > tol
    is_new_buy = (w_old <= tol) & is_buy
    is_existing_buy = is_existing & is_buy
    is_existing_sell = is_existing & is_sell

    p_dir = np.zeros_like(dw_map, dtype=float)
    p_dir[is_buy] = P_pos0[is_buy]
    p_dir[is_sell] = P_neg0[is_sell]

    # Per-name recovery gate. np.inf means ineligible.
    p_required = np.full_like(dw_map, np.inf, dtype=float)
    gate_type = np.full(dw_map.shape, "ineligible", dtype=object)
    p_required[is_existing_buy] = float(p_existing)
    gate_type[is_existing_buy] = "existing_buy"
    p_required[is_existing_sell] = float(p_sell)
    gate_type[is_existing_sell] = "existing_sell"
    p_required[is_new_buy] = float(p_new_buy)
    gate_type[is_new_buy] = "new_buy"

    abs_dw = np.abs(dw_map)
    score = p_dir * abs_dw
    eligible_direction = is_existing_buy | is_existing_sell | is_new_buy
    approved = (p_dir >= p_required) & (abs_dw >= float(eps_delta)) & eligible_direction

    raw_idx = np.where(approved)[0]
    ranked_idx = raw_idx[np.argsort(-score[raw_idx])] if raw_idx.size else np.array([], dtype=int)

    selected = []
    n_new_buy = 0
    dropped_reasons = {int(j): "low_score_or_below_asymmetric_gate" for j in range(dw_map.size)}

    for j in ranked_idx:
        j = int(j)
        this_is_new_buy = bool(is_new_buy[j])

        if len(selected) >= int(max_moves):
            dropped_reasons[j] = "max_moves"
            continue
        if this_is_new_buy and n_new_buy >= int(max_new_buys):
            dropped_reasons[j] = "max_new_buy_names"
            continue

        selected.append(j)
        if this_is_new_buy:
            n_new_buy += 1
        dropped_reasons[j] = "selected_after_truncation"

    selected_idx = np.asarray(selected, dtype=int)
    selected_set = set(map(int, selected_idx.tolist()))

    rows = []
    for j in range(dw_map.size):
        direction = "buy" if dw_map[j] > tol else ("sell" if dw_map[j] < -tol else "zero")
        selected_j = bool(j in selected_set)
        reason = "" if selected_j else dropped_reasons.get(int(j), "not_selected")
        rows.append({
            "hold": int(hold_number),
            "ticker": str(tickers[j]),
            "j": int(j),
            "current_weight": float(w_old[j]),
            "dw_map": float(dw_map[j]),
            "abs_dw_map": float(abs_dw[j]),
            "p_dir": float(p_dir[j]),
            "p_required_recovery": float(p_required[j]) if np.isfinite(p_required[j]) else np.nan,
            "recovery_gate_type": str(gate_type[j]),
            "P_pos0": float(P_pos0[j]),
            "P_neg0": float(P_neg0[j]),
            "candidate_direction": direction,
            "approved_by_relaxed_gate": bool(approved[j]),
            "selected_after_truncation": selected_j,
            "is_existing_holding": bool(is_existing[j]),
            "is_new_buy": bool(is_new_buy[j]),
            "is_dust": bool((w_old[j] > tol) and (w_old[j] < float(dust_floor))),
            "score": float(score[j]),
            "implemented_dw": 0.0,
            "implemented_trade_abs": 0.0,
            "reason_if_dropped": reason,
        })

    df_candidates = pd.DataFrame(rows)

    return {
        "selected_idx": selected_idx,
        "raw_idx": raw_idx,
        "dropped_idx": np.setdiff1d(raw_idx, selected_idx),
        "df_candidates": df_candidates,
        "p_dir": p_dir,
        "p_required": p_required,
        "gate_type": gate_type,
        "score": score,
        "P_pos0": P_pos0,
        "P_neg0": P_neg0,
    }


def make_recovery_candidate_object(
    *,
    candidate_name: str,
    selected_idx: np.ndarray,
    candidate_pack: dict,
    cgrid_out: dict,
    p_recovery: float,
    eps_delta: float,
):
    """Create a candidate object compatible with build_asoc_locked_rebalance_decision_from_rope."""
    context = cgrid_out["context"]
    tickers = get_asset_tickers_from_context_asoc(context)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    dw_map = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)

    selected_idx = np.asarray(selected_idx, dtype=int).reshape(-1)
    p_dir = np.asarray(candidate_pack["p_dir"], dtype=float).reshape(-1)
    score = np.asarray(candidate_pack["score"], dtype=float).reshape(-1)

    if selected_idx.size:
        df_moves = pd.DataFrame({
            "j": selected_idx,
            "ticker": [tickers[j] for j in selected_idx],
            "direction": np.where(dw_map[selected_idx] > 0.0, "increase", "decrease"),
            "dw_MAP": dw_map[selected_idx],
            "abs_dw_MAP": np.abs(dw_map[selected_idx]),
            "w_old": w_old[selected_idx],
            "w_old_plus_dw_MAP": w_old[selected_idx] + dw_map[selected_idx],
            "directional_confidence": p_dir[selected_idx],
            "rank_score_pdir_abs_dw_MAP": score[selected_idx],
        }).sort_values("rank_score_pdir_abs_dw_MAP", ascending=False).reset_index(drop=True)
    else:
        df_moves = pd.DataFrame(columns=[
            "j", "ticker", "direction", "dw_MAP", "abs_dw_MAP", "w_old",
            "w_old_plus_dw_MAP", "directional_confidence", "rank_score_pdir_abs_dw_MAP"
        ])

    return {
        "candidate_name": candidate_name,
        "candidate_type": "controlled_recovery_sign_confidence_eps_delta",
        "eps": float(eps_delta),
        "pi_move": float(p_recovery),
        "S_idx": selected_idx,
        "S_idx_raw_before_truncation": np.asarray(candidate_pack["raw_idx"], dtype=int),
        "dropped_idx_by_recovery_truncation": np.asarray(candidate_pack["dropped_idx"], dtype=int),
        "buy_idx": selected_idx[dw_map[selected_idx] > 0.0] if selected_idx.size else np.array([], dtype=int),
        "sell_idx": selected_idx[dw_map[selected_idx] < 0.0] if selected_idx.size else np.array([], dtype=int),
        "S_size": int(selected_idx.size),
        "n_buy": int(np.count_nonzero(dw_map[selected_idx] > 0.0)) if selected_idx.size else 0,
        "n_sell": int(np.count_nonzero(dw_map[selected_idx] < 0.0)) if selected_idx.size else 0,
        "total_buy_map": float(np.sum(np.maximum(dw_map[selected_idx], 0.0))) if selected_idx.size else 0.0,
        "total_sell_map": float(np.sum(np.maximum(-dw_map[selected_idx], 0.0))) if selected_idx.size else 0.0,
        "net_map_sum_on_S": float(np.sum(dw_map[selected_idx])) if selected_idx.size else 0.0,
        "map_move_mass": float(np.sum(np.abs(dw_map[selected_idx]))) if selected_idx.size else 0.0,
        "map_move_mass_share": float(np.sum(np.abs(dw_map[selected_idx])) / (np.sum(np.abs(dw_map)) + 1e-16)) if selected_idx.size else 0.0,
        "min_directional_confidence": float(np.min(p_dir[selected_idx])) if selected_idx.size else np.nan,
        "mean_directional_confidence": float(np.mean(p_dir[selected_idx])) if selected_idx.size else np.nan,
        "df_moves": df_moves,
        "dir_prob": p_dir,
        "rank_score_pdir_abs_dw_MAP": score,
    }


def rebuild_trade_dataframe_for_decision(decision: dict) -> pd.DataFrame:
    """Rebuild df_implemented_trades after turnover-cap scaling."""
    w_old = np.asarray(decision["w_old"], dtype=float).reshape(-1)
    w_new = np.asarray(decision["w_new"], dtype=float).reshape(-1)
    dw_impl = np.asarray(decision["dw_impl"], dtype=float).reshape(-1)
    dw_map = np.asarray(decision.get("dw_map", np.zeros_like(dw_impl)), dtype=float).reshape(-1)

    cgrid_out = decision.get("cgrid_out", {})
    context = cgrid_out.get("context", {}) if isinstance(cgrid_out, dict) else {}
    tickers = get_asset_tickers_from_context_asoc(context) if context else [f"asset_{j}" for j in range(dw_impl.size)]

    impl = decision.get("implementation_details", {}) if isinstance(decision.get("implementation_details", {}), dict) else {}
    buy_idx = set(map(int, np.asarray(impl.get("buy_idx", np.array([], dtype=int)), dtype=int).reshape(-1)))
    sell_idx = set(map(int, np.asarray(impl.get("sell_idx", np.array([], dtype=int)), dtype=int).reshape(-1)))
    funding_idx = set(map(int, np.asarray(impl.get("funding_idx", np.array([], dtype=int)), dtype=int).reshape(-1)))

    trade_idx = np.where(np.abs(dw_impl) > 1e-12)[0]
    rows = []
    for j in trade_idx:
        if int(j) in buy_idx:
            role = "approved_buy"
        elif int(j) in sell_idx:
            role = "approved_sell"
        elif int(j) in funding_idx:
            role = "funding_sell"
        else:
            role = "other"
        rows.append({
            "j": int(j),
            "ticker": str(tickers[j]),
            "role": role,
            "w_old": float(w_old[j]),
            "dw_impl": float(dw_impl[j]),
            "w_new": float(w_new[j]),
            "dw_MAP": float(dw_map[j]),
            "abs_dw_impl": float(abs(dw_impl[j])),
        })
    return pd.DataFrame(rows).sort_values("abs_dw_impl", ascending=False).reset_index(drop=True) if rows else pd.DataFrame(columns=["j", "ticker", "role", "w_old", "dw_impl", "w_new", "dw_MAP", "abs_dw_impl"])


def apply_one_way_turnover_cap_to_decision(decision: dict, *, max_turnover: float) -> dict:
    """Scale an implemented recovery trade if it exceeds a one-way turnover cap."""
    if bool(decision.get("no_trade", False)):
        decision["turnover_cap_applied"] = False
        decision["turnover_before_cap"] = 0.0
        decision["turnover_after_cap"] = 0.0
        return decision

    dw_impl = np.asarray(decision["dw_impl"], dtype=float).reshape(-1)
    w_old = np.asarray(decision["w_old"], dtype=float).reshape(-1)
    turnover_before = float(0.5 * np.sum(np.abs(dw_impl)))
    max_turnover = float(max_turnover)

    cap_applied = False
    scale = 1.0
    if max_turnover > 0.0 and turnover_before > max_turnover + 1e-14:
        scale = max_turnover / turnover_before
        dw_impl = dw_impl * scale
        cap_applied = True

    # Numerical budget repair after scaling.
    residual = float(dw_impl.sum())
    if abs(residual) > 1e-14:
        buy_idx = np.where(dw_impl > 1e-12)[0]
        if buy_idx.size:
            j_fix = int(buy_idx[np.argmax(np.abs(dw_impl[buy_idx]))])
        else:
            j_fix = int(np.argmax(np.abs(dw_impl)))
        dw_impl[j_fix] -= residual

    w_new = w_old + dw_impl
    tiny_neg = np.where((w_new < 0.0) & (w_new >= -1e-12))[0]
    if tiny_neg.size:
        w_new[tiny_neg] = 0.0
        dw_impl = w_new - w_old

    if np.any(w_new < -1e-10):
        # Safety fallback: reject the capped implementation rather than allow infeasibility.
        decision["no_trade"] = True
        decision["no_trade_reason"] = "turnover cap scaling produced negative weights"
        decision["implementation_case"] = "turnover_cap_infeasible_no_trade"
        decision["dw_impl"] = np.zeros_like(w_old)
        decision["w_new"] = w_old.copy()
        decision["w_rebalanced"] = w_old.copy()
        decision["implemented_trade_summary"] = asoc_trade_summary_from_delta(np.zeros_like(w_old))
        decision["turnover_cap_applied"] = bool(cap_applied)
        decision["turnover_before_cap"] = turnover_before
        decision["turnover_after_cap"] = 0.0
        return decision

    # Update decision internals.
    decision["dw_impl"] = dw_impl.copy()
    decision["w_new"] = w_new.copy()
    decision["w_rebalanced"] = w_new.copy()
    decision["implemented_trade_summary"] = asoc_trade_summary_from_delta(dw_impl)
    decision["new_structure"] = asoc_portfolio_structure_summary(w_new)
    decision["df_implemented_trades"] = rebuild_trade_dataframe_for_decision(decision)

    cgrid_out = decision.get("cgrid_out", {})
    context = cgrid_out.get("context", {}) if isinstance(cgrid_out, dict) else {}
    if context:
        y_fit = np.asarray(context["y_fit"], dtype=float).reshape(-1)
        R_fit = np.asarray(context["R_fit"], dtype=float)
        dw_map = np.asarray(decision.get("dw_map", np.zeros_like(dw_impl)), dtype=float).reshape(-1)
        te_old = realised_te_rmse_raw(y_fit, R_fit, w_old)
        te_map = realised_te_rmse_raw(y_fit, R_fit, w_old + dw_map)
        te_new = realised_te_rmse_raw(y_fit, R_fit, w_new)
        decision["te_fit2"] = {
            "old": float(te_old),
            "map_proposed": float(te_map),
            "new": float(te_new),
            "map_improvement_vs_old": float(te_old - te_map),
            "implemented_improvement_vs_old": float(te_old - te_new),
        }

    impl = decision.get("implementation_details", {}) if isinstance(decision.get("implementation_details", {}), dict) else {}
    impl["turnover_cap_applied"] = bool(cap_applied)
    impl["turnover_before_cap"] = float(turnover_before)
    impl["turnover_after_cap"] = float(0.5 * np.sum(np.abs(dw_impl)))
    impl["turnover_cap"] = float(max_turnover)
    impl["turnover_cap_scale"] = float(scale)
    decision["implementation_details"] = impl
    decision["turnover_cap_applied"] = bool(cap_applied)
    decision["turnover_before_cap"] = float(turnover_before)
    decision["turnover_after_cap"] = float(0.5 * np.sum(np.abs(dw_impl)))

    if cap_applied and "turnover_capped" not in str(decision.get("implementation_case", "")):
        decision["implementation_case"] = str(decision.get("implementation_case", "")) + "_turnover_capped"

    return decision


def add_recovery_trade_info_to_candidate_table(df_candidates: pd.DataFrame, decision: dict) -> pd.DataFrame:
    """Attach implemented Delta-w and funding-top-up diagnostics to the recovery-candidate table."""
    if df_candidates is None or len(df_candidates) == 0:
        return pd.DataFrame()

    out = df_candidates.copy()
    dw_impl = np.asarray(decision.get("dw_impl", np.array([])), dtype=float).reshape(-1)
    if dw_impl.size:
        j_arr = out["j"].to_numpy(dtype=int)
        out["implemented_dw"] = dw_impl[j_arr]
        out["implemented_trade_abs"] = np.abs(dw_impl[j_arr])

    extra_info = decision.get("extra_funding_info_by_j", {})
    protect_info = decision.get("funding_protection_info_by_j", {})

    def _fund_field(j, field, default):
        try:
            return extra_info.get(int(j), {}).get(field, default)
        except Exception:
            return default

    def _protect_field(j, field, default):
        try:
            return protect_info.get(int(j), {}).get(field, default)
        except Exception:
            return default

    out["used_as_extra_funder"] = out["j"].map(lambda j: bool(_fund_field(j, "used_as_extra_funder", False)))
    out["extra_funding_amount"] = out["j"].map(lambda j: float(_fund_field(j, "extra_funding_amount", 0.0)))
    out["funding_rank"] = out["j"].map(lambda j: _fund_field(j, "funding_rank", np.nan))
    out["funding_reason"] = out["j"].map(lambda j: str(_fund_field(j, "funding_reason", "")))
    out["protected_from_funding"] = out["j"].map(lambda j: bool(_protect_field(j, "protected_from_funding", False)))
    out["funding_protection_reason"] = out["j"].map(lambda j: str(_protect_field(j, "funding_protection_reason", "")))

    out["recovery_decision_no_trade"] = bool(decision.get("no_trade", False))
    out["recovery_implementation_case"] = str(decision.get("implementation_case", ""))
    out["turnover_cap_applied"] = bool(decision.get("turnover_cap_applied", False))
    out["buy_topup_enabled"] = bool(decision.get("buy_topup_enabled", False))
    out["buy_topup_used"] = bool(decision.get("buy_topup_used", False))
    return out


def choose_recovery_extra_funders_for_topup(
    *,
    w_old: np.ndarray,
    dw_map: np.ndarray,
    buy_idx: np.ndarray,
    sell_idx: np.ndarray,
    asset_cols: Sequence[str],
    amount_needed: float,
    dust_floor: float,
    max_extra_funding_names: int,
    protect_top_n: int,
    protect_weight_above: float,
    tol: float = 1e-12,
) -> tuple[dict[int, float], dict[int, dict], dict[int, dict]]:
    """
    Select weak existing holdings as additional funders for buy-dominant
    recovery trades. This uses only current weights and the current Delta-w
    MAP signal; no competitor or future information enters.
    """
    w_old = np.asarray(w_old, dtype=float).reshape(-1)
    dw_map = np.asarray(dw_map, dtype=float).reshape(-1)
    asset_cols = list(asset_cols)

    buy_set = set(map(int, np.asarray(buy_idx, dtype=int).tolist()))
    sell_set = set(map(int, np.asarray(sell_idx, dtype=int).tolist()))
    exclude = buy_set | sell_set

    positive_idx = np.where(w_old > tol)[0]
    top_order = positive_idx[np.argsort(-w_old[positive_idx])] if positive_idx.size else np.array([], dtype=int)
    protected_top = set(map(int, top_order[:int(protect_top_n)].tolist()))

    protection_info: dict[int, dict] = {}
    ranked_candidates = []
    for j0 in positive_idx:
        j = int(j0)
        if j in exclude:
            reason = "protected_buy_candidate" if j in buy_set else "approved_sell_already_used"
            protection_info[j] = {
                "protected_from_funding": True,
                "funding_protection_reason": reason,
            }
            continue
        if j in protected_top:
            protection_info[j] = {
                "protected_from_funding": True,
                "funding_protection_reason": "protected_top10",
            }
            continue
        if w_old[j] >= float(protect_weight_above):
            protection_info[j] = {
                "protected_from_funding": True,
                "funding_protection_reason": "protected_weight_above_threshold",
            }
            continue

        is_dust = bool(w_old[j] < float(dust_floor))
        # Order: dust first, then smaller holdings, then more negative/weaker dw_map.
        ranked_candidates.append((0 if is_dust else 1, float(w_old[j]), float(dw_map[j]), j))
        protection_info[j] = {
            "protected_from_funding": False,
            "funding_protection_reason": "eligible_extra_funder",
        }

    ranked_candidates.sort()

    funding: dict[int, float] = {}
    funding_info: dict[int, dict] = {}
    need = float(max(amount_needed, 0.0))
    rank = 0
    for _, _, _, j in ranked_candidates:
        if need <= tol:
            break
        if rank >= int(max_extra_funding_names):
            break
        take = min(float(w_old[j]), need)
        if take <= tol:
            continue
        rank += 1
        funding[int(j)] = take
        reason = "extra_funder_dust" if w_old[j] < float(dust_floor) else "extra_funder_smallest_weight"
        if dw_map[j] < -tol:
            reason = "extra_funder_negative_dw_map"
        funding_info[int(j)] = {
            "used_as_extra_funder": True,
            "extra_funding_amount": float(take),
            "funding_rank": int(rank),
            "funding_reason": reason,
            "ticker": str(asset_cols[j]),
            "old_weight": float(w_old[j]),
            "dw_map": float(dw_map[j]),
        }
        need -= take

    return funding, funding_info, protection_info


def apply_buy_dominant_funding_topup_to_recovery_decision(
    *,
    decision: dict,
    candidate_pack: dict,
    cgrid_out: dict,
    max_one_way_turnover: float,
    target_one_way_turnover: float,
    dust_floor: float,
    max_extra_funding_names: int,
    protect_top_n: int,
    protect_weight_above: float,
    tol: float = 1e-12,
) -> dict:
    """
    Recovery-mode-only implementation repair for buy-dominant or sell-insufficient
    selected candidate sets.

    If approved buys exceed approved sells, use weak existing holdings as extra
    funders, while preserving nonnegativity, exact budget, and the one-way
    turnover cap. No global renormalisation is used.
    """
    decision["buy_topup_enabled"] = True
    decision["buy_topup_used"] = False
    decision["extra_funding_info_by_j"] = {}
    decision["funding_protection_info_by_j"] = {}

    context = cgrid_out.get("context", {}) if isinstance(cgrid_out, dict) else {}
    if not context:
        decision["buy_topup_skip_reason"] = "missing_cgrid_context"
        return decision

    tickers = get_asset_tickers_from_context_asoc(context)
    w_old = np.asarray(context["w_old"], dtype=float).reshape(-1)
    dw_map = np.asarray(cgrid_out["dw_map_smoothed"], dtype=float).reshape(-1)
    selected_idx = np.asarray(candidate_pack.get("selected_idx", np.array([], dtype=int)), dtype=int)

    if selected_idx.size == 0:
        decision["buy_topup_skip_reason"] = "empty_selected_recovery_set"
        return decision

    buy_idx = selected_idx[dw_map[selected_idx] > tol]
    sell_idx = selected_idx[dw_map[selected_idx] < -tol]

    raw_buy = float(np.sum(dw_map[buy_idx])) if buy_idx.size else 0.0
    raw_sell = float(np.sum(-dw_map[sell_idx])) if sell_idx.size else 0.0

    decision["approved_buy_amount_raw"] = raw_buy
    decision["approved_sell_amount_raw"] = raw_sell

    if raw_buy <= tol:
        decision["buy_topup_skip_reason"] = "no_approved_buys"
        return decision

    if raw_buy <= raw_sell + tol:
        decision["buy_topup_skip_reason"] = "approved_sells_sufficient"
        return decision

    max_turnover = float(max_one_way_turnover)
    target_turnover = float(target_one_way_turnover)
    if target_turnover <= tol:
        target_turnover = raw_buy
    # Budgeted sizing: once the relaxed recovery gate approves a
    # buy-dominant set, use Delta-w MAP for relative direction/ranking but
    # target a meaningful bounded repair.  This prevents approved recovery
    # trades from becoming microscopic solely because the raw Delta-w MAP
    # sum is small.
    target_buy = min(max(raw_buy, target_turnover), max_turnover)

    decision["target_one_way_turnover"] = float(target_turnover)
    decision["raw_buy_scale_to_target"] = float(target_buy / max(raw_buy, tol))

    # Approved sells are used first, capped by current holdings.
    funding: dict[int, float] = {}
    for j0 in sell_idx:
        j = int(j0)
        amount = min(float(-dw_map[j]), float(w_old[j]))
        if amount > tol:
            funding[j] = amount

    approved_sell_amount = float(sum(funding.values()))
    extra_needed = max(0.0, target_buy - approved_sell_amount)

    extra_funding, extra_info, protect_info = choose_recovery_extra_funders_for_topup(
        w_old=w_old,
        dw_map=dw_map,
        buy_idx=buy_idx,
        sell_idx=sell_idx,
        asset_cols=tickers,
        amount_needed=extra_needed,
        dust_floor=dust_floor,
        max_extra_funding_names=max_extra_funding_names,
        protect_top_n=protect_top_n,
        protect_weight_above=protect_weight_above,
        tol=tol,
    )

    for j, amount in extra_funding.items():
        funding[j] = funding.get(j, 0.0) + float(amount)

    total_funding = float(sum(funding.values()))
    extra_funding_amount = float(sum(extra_funding.values()))
    implemented_buy = min(target_buy, total_funding, max_turnover)

    decision["extra_funding_info_by_j"] = extra_info
    decision["funding_protection_info_by_j"] = protect_info
    decision["extra_funding_amount"] = extra_funding_amount
    decision["n_extra_funders"] = int(len(extra_funding))
    decision["target_buy_amount"] = float(target_buy)
    decision["target_one_way_turnover"] = float(target_turnover)
    decision["implemented_buy_amount"] = float(implemented_buy)
    decision["funding_topup_total_funding_available"] = float(total_funding)

    if implemented_buy <= tol or total_funding <= tol:
        decision["buy_topup_skip_reason"] = "no_funding_available"
        return decision

    buy_scale = implemented_buy / max(raw_buy, tol)
    funding_scale = implemented_buy / max(total_funding, tol)

    dw_impl = np.zeros_like(w_old)
    for j0 in buy_idx:
        j = int(j0)
        dw_impl[j] = buy_scale * float(dw_map[j])

    for j, amount in funding.items():
        dw_impl[int(j)] -= funding_scale * float(amount)

    # Exact local budget repair: adjust the largest implemented buy.
    residual = float(dw_impl.sum())
    if abs(residual) > 1e-14:
        if buy_idx.size:
            j_fix = int(buy_idx[np.argmax(np.abs(dw_impl[buy_idx]))])
        else:
            j_fix = int(np.argmax(np.abs(dw_impl)))
        dw_impl[j_fix] -= residual

    turnover = float(0.5 * np.sum(np.abs(dw_impl)))
    if turnover > max_turnover + 1e-12:
        scale = max_turnover / max(turnover, tol)
        dw_impl *= scale
        residual = float(dw_impl.sum())
        if abs(residual) > 1e-14 and buy_idx.size:
            j_fix = int(buy_idx[np.argmax(np.abs(dw_impl[buy_idx]))])
            dw_impl[j_fix] -= residual
        turnover = float(0.5 * np.sum(np.abs(dw_impl)))

    w_new = w_old + dw_impl
    tiny_neg = np.where((w_new < 0.0) & (w_new >= -1e-12))[0]
    if tiny_neg.size:
        w_new[tiny_neg] = 0.0
        dw_impl = w_new - w_old
        turnover = float(0.5 * np.sum(np.abs(dw_impl)))

    if np.any(w_new < -1e-10):
        # Keep the original decision rather than introduce an infeasible repair.
        decision["buy_topup_skip_reason"] = "topup_would_create_negative_weights"
        return decision

    # Replace implementation by the funding top-up trade.
    decision["no_trade"] = False
    decision["no_trade_reason"] = ""
    decision["implementation_case"] = "buy_dominant_with_funding_topup"
    decision["buy_topup_used"] = True
    decision["buy_topup_skip_reason"] = ""
    decision["dw_impl"] = dw_impl.copy()
    decision["w_old"] = w_old.copy()
    decision["w_new"] = w_new.copy()
    decision["w_rebalanced"] = w_new.copy()
    decision["implemented_trade_summary"] = asoc_trade_summary_from_delta(dw_impl)
    decision["new_structure"] = asoc_portfolio_structure_summary(w_new)
    decision["df_implemented_trades"] = rebuild_trade_dataframe_for_decision(decision)
    decision["turnover_before_cap"] = turnover
    decision["turnover_after_cap"] = turnover
    decision["turnover_cap_applied"] = False

    # Fit-window TE diagnostics for the implemented top-up move.
    y_fit = np.asarray(context["y_fit"], dtype=float).reshape(-1)
    R_fit = np.asarray(context["R_fit"], dtype=float)
    te_old = realised_te_rmse_raw(y_fit, R_fit, w_old)
    te_map = realised_te_rmse_raw(y_fit, R_fit, w_old + dw_map)
    te_new = realised_te_rmse_raw(y_fit, R_fit, w_new)
    decision["te_fit2"] = {
        "old": float(te_old),
        "map_proposed": float(te_map),
        "new": float(te_new),
        "map_improvement_vs_old": float(te_old - te_map),
        "implemented_improvement_vs_old": float(te_old - te_new),
    }

    impl = decision.get("implementation_details", {}) if isinstance(decision.get("implementation_details", {}), dict) else {}
    impl.update({
        "implementation_case": "buy_dominant_with_funding_topup",
        "buy_topup_used": True,
        "approved_buy_amount_raw": float(raw_buy),
        "approved_sell_amount_raw": float(raw_sell),
        "approved_sell_amount_capped_by_holdings": float(approved_sell_amount),
        "extra_funding_amount": float(extra_funding_amount),
        "n_extra_funders": int(len(extra_funding)),
        "target_buy_amount": float(target_buy),
        "target_one_way_turnover": float(target_turnover),
        "raw_buy_scale_to_target": float(target_buy / max(raw_buy, tol)),
        "implemented_buy_amount": float(implemented_buy),
        "turnover_cap": float(max_turnover),
        "turnover_used": float(turnover),
        "max_extra_funding_names": int(max_extra_funding_names),
        "protect_top_n": int(protect_top_n),
        "protect_weight_above": float(protect_weight_above),
    })
    decision["implementation_details"] = impl

    return decision


def build_controlled_recovery_decision(
    *,
    hold_number: int,
    base_decision: dict,
    experiment_state: dict,
    config: ASOCRebalanceConfig,
    recovery_state: dict,
) -> dict:
    """
    Replace the ordinary decision by the controlled recovery implementation.
    The expensive c-grid/MALA objects are reused from the ordinary wrapper.
    """
    hist = experiment_state.get("rebalance_history", {}).get(int(hold_number), {})
    if not hist:
        raise KeyError(f"Missing rebalance_history for hold {hold_number}; cannot build recovery decision.")

    cgrid_out = hist["cgrid_out"]
    mala_out = hist["mala_out"]
    rope_out_normal = hist.get("rope_out", {})

    candidate_pack = build_recovery_candidate_table_and_support(
        hold_number=hold_number,
        cgrid_out=cgrid_out,
        mala_out=mala_out,
        p_recovery=RECOVERY_P_DIR,
        eps_delta=RECOVERY_EPS_DELTA,
        max_moves=RECOVERY_MAX_APPROVED_MOVES,
        max_new_buys=RECOVERY_MAX_NEW_BUY_NAMES,
        dust_floor=RECOVERY_DUST_FLOOR,
        p_existing=RECOVERY_P_DIR_EXISTING,
        p_new_buy=RECOVERY_P_DIR_NEW_BUY,
        p_sell=RECOVERY_P_DIR_SELL,
    )

    candidate_name = f"recovery_asym_eps{RECOVERY_EPS_DELTA:.0e}_exist{RECOVERY_P_DIR_EXISTING:.3f}_new{RECOVERY_P_DIR_NEW_BUY:.3f}_sell{RECOVERY_P_DIR_SELL:.3f}".replace("-0", "-").replace(".", "p")
    selected_candidate = make_recovery_candidate_object(
        candidate_name=candidate_name,
        selected_idx=candidate_pack["selected_idx"],
        candidate_pack=candidate_pack,
        cgrid_out=cgrid_out,
        p_recovery=RECOVERY_P_DIR,
        eps_delta=RECOVERY_EPS_DELTA,
    )

    recovery_rope_out = dict(rope_out_normal) if isinstance(rope_out_normal, dict) else {}
    recovery_rope_out["selected_candidate_name"] = candidate_name
    recovery_rope_out["selected_candidate"] = selected_candidate
    recovery_rope_out.setdefault("candidates", {})[candidate_name] = selected_candidate
    recovery_rope_out["controlled_recovery_candidate_table"] = candidate_pack["df_candidates"]

    # Custom recovery implementation caps; ordinary calm-mode config is untouched.
    recovery_impl_config = dataclass_replace(
        config,
        max_buy_only_funding_names=int(RECOVERY_MAX_APPROVED_MOVES),
        close_small_funders=True,
        buy_only_dust_tol=float(RECOVERY_DUST_FLOOR),
        max_cardinality_change=int(RECOVERY_MAX_NEW_BUY_NAMES),
        allow_new_buy_names=True,
    )

    recovery_decision = build_asoc_locked_rebalance_decision_from_rope(
        cgrid_out=cgrid_out,
        mala_out=mala_out,
        rope_out=recovery_rope_out,
        config=recovery_impl_config,
    )

    if RECOVERY_BUY_TOPUP_ENABLED:
        recovery_decision = apply_buy_dominant_funding_topup_to_recovery_decision(
            decision=recovery_decision,
            candidate_pack=candidate_pack,
            cgrid_out=cgrid_out,
            max_one_way_turnover=RECOVERY_MAX_ONE_WAY_TURNOVER,
            target_one_way_turnover=RECOVERY_TARGET_ONE_WAY_TURNOVER,
            dust_floor=RECOVERY_DUST_FLOOR,
            max_extra_funding_names=RECOVERY_MAX_EXTRA_FUNDING_NAMES,
            protect_top_n=RECOVERY_PROTECT_TOP_N_FUNDERS,
            protect_weight_above=RECOVERY_PROTECT_WEIGHT_ABOVE,
        )

    recovery_decision = apply_one_way_turnover_cap_to_decision(
        recovery_decision,
        max_turnover=RECOVERY_MAX_ONE_WAY_TURNOVER,
    )

    df_rec = add_recovery_trade_info_to_candidate_table(
        candidate_pack["df_candidates"],
        recovery_decision,
    )
    experiment_state.setdefault("recovery_candidate_frames", []).append(df_rec)

    recovery_decision["recovery_mode_active"] = True
    recovery_decision["recovery_state_for_decision"] = dict(recovery_state)
    recovery_decision["normal_candidate_name"] = str(base_decision.get("candidate_name", ""))
    recovery_decision["normal_no_trade"] = bool(base_decision.get("no_trade", False))
    recovery_decision["normal_implementation_case"] = str(base_decision.get("implementation_case", ""))
    recovery_decision["controlled_recovery_replaced_normal_decision"] = True
    recovery_decision["S_size_raw_before_truncation"] = int(len(candidate_pack["raw_idx"]))
    recovery_decision["S_size_after_truncation"] = int(len(candidate_pack["selected_idx"]))
    recovery_decision["recovery_max_approved_moves"] = int(RECOVERY_MAX_APPROVED_MOVES)
    recovery_decision["recovery_max_new_buy_names"] = int(RECOVERY_MAX_NEW_BUY_NAMES)
    recovery_decision["recovery_max_one_way_turnover"] = float(RECOVERY_MAX_ONE_WAY_TURNOVER)
    recovery_decision["recovery_target_one_way_turnover"] = float(RECOVERY_TARGET_ONE_WAY_TURNOVER)
    recovery_decision["recovery_rank_by"] = "p_dir_x_abs_dw_MAP"
    recovery_decision["recovery_p_dir_existing"] = float(RECOVERY_P_DIR_EXISTING)
    recovery_decision["recovery_p_dir_new_buy"] = float(RECOVERY_P_DIR_NEW_BUY)
    recovery_decision["recovery_p_dir_sell"] = float(RECOVERY_P_DIR_SELL)
    recovery_decision["buy_topup_enabled"] = bool(RECOVERY_BUY_TOPUP_ENABLED)
    recovery_decision["recovery_max_extra_funding_names"] = int(RECOVERY_MAX_EXTRA_FUNDING_NAMES)
    recovery_decision["recovery_protect_top_n_funders"] = int(RECOVERY_PROTECT_TOP_N_FUNDERS)
    recovery_decision["recovery_protect_weight_above"] = float(RECOVERY_PROTECT_WEIGHT_ABOVE)
    recovery_decision["df_recovery_candidates"] = df_rec

    # Lightweight metadata for summaries.
    wm = dict(recovery_decision.get("wrapper_metadata", {}))
    wm.update({
        "recovery_mode_active": True,
        "recovery_p_dir": float(RECOVERY_P_DIR),
        "recovery_p_dir_existing": float(RECOVERY_P_DIR_EXISTING),
        "recovery_p_dir_new_buy": float(RECOVERY_P_DIR_NEW_BUY),
        "recovery_p_dir_sell": float(RECOVERY_P_DIR_SELL),
        "recovery_eps_delta": float(RECOVERY_EPS_DELTA),
        "recovery_max_approved_moves": int(RECOVERY_MAX_APPROVED_MOVES),
        "recovery_max_new_buy_names": int(RECOVERY_MAX_NEW_BUY_NAMES),
        "recovery_max_one_way_turnover": float(RECOVERY_MAX_ONE_WAY_TURNOVER),
    "recovery_target_one_way_turnover": float(RECOVERY_TARGET_ONE_WAY_TURNOVER),
    "recovery_buy_topup_enabled": bool(RECOVERY_BUY_TOPUP_ENABLED),
    "recovery_max_extra_funding_names": int(RECOVERY_MAX_EXTRA_FUNDING_NAMES),
    "recovery_protect_top_n_funders": int(RECOVERY_PROTECT_TOP_N_FUNDERS),
    "recovery_protect_weight_above": float(RECOVERY_PROTECT_WEIGHT_ABOVE),
        "normal_candidate_name": str(base_decision.get("candidate_name", "")),
        "buy_topup_enabled": bool(RECOVERY_BUY_TOPUP_ENABLED),
        "buy_topup_used": bool(recovery_decision.get("buy_topup_used", False)),
        "recovery_max_extra_funding_names": int(RECOVERY_MAX_EXTRA_FUNDING_NAMES),
    })
    recovery_decision["wrapper_metadata"] = wm

    return recovery_decision



def strip_mala_samples_from_rebalance_history(experiment_state: dict, hold_number: int) -> None:
    """Remove full MALA sample arrays from per-hold history after decisions are made.

    The decision layer uses `W_post` to compute directional probabilities.
    Once a hold decision has been built, keeping the full sample matrix for
    every hold is unnecessary for the public reproducibility outputs and can
    consume substantial memory in production runs.
    """
    hist = experiment_state.get("rebalance_history", {}).get(int(hold_number), {})
    if not isinstance(hist, dict):
        return
    mala = hist.get("mala_out", {})
    if not isinstance(mala, dict) or "W_post" not in mala:
        return

    W_post = np.asarray(mala["W_post"])
    mala_stripped = {k: v for k, v in mala.items() if k != "W_post"}
    mala_stripped["W_post_shape"] = tuple(map(int, W_post.shape))
    mala_stripped["W_post_removed_from_history"] = True
    hist["mala_out"] = mala_stripped


def rebalance_step_fn_asoc_controlled_recovery_local(
    *,
    hold_number: int,
    window: dict,
    previous_w: np.ndarray,
    experiment_state: dict,
) -> dict:
    """
    H1--H16 paper-main maintenance wrapper.

    Calm mode: ordinary posterior-directional gate and ordinary local implementation.
    Recovery mode: replace only the candidate/implementation step by the
    controlled recovery rule p_dir>=0.55, ranked by p_dir*|dw_MAP| and capped.
    """
    base_decision = rebalance_step_fn_asoc(
        hold_number=hold_number,
        window=window,
        previous_w=previous_w,
        experiment_state=experiment_state,
        config=asoc_rebalance_config_calm,
        sapg_n_iter=DW_SAPG_N_ITER,
        fista_max_iter=DW_FISTA_MAX_ITER,
        selected_rope_candidate=asoc_rebalance_config_calm.selected_rope_candidate,
    )

    recovery_state = experiment_state.get("recovery_state_for_current_decision", {})
    recovery_active = bool(recovery_state.get("recovery_active_for_decision", False))

    if not recovery_active:
        base_decision["recovery_mode_active"] = False
        base_decision["controlled_recovery_replaced_normal_decision"] = False
        base_decision["recovery_state_for_decision"] = dict(recovery_state)
        wm = dict(base_decision.get("wrapper_metadata", {}))
        wm.update({
            "recovery_mode_active": False,
            "recovery_p_dir": np.nan,
            "recovery_max_one_way_turnover": np.nan,
        })
        base_decision["wrapper_metadata"] = wm
        strip_mala_samples_from_rebalance_history(experiment_state, hold_number)
        return base_decision

    if asoc_rebalance_config_calm.verbose:
        print("\n[Controlled recovery] Recovery mode active; replacing ordinary candidate/implementation.")
        print(f"  reason       = {recovery_state.get('reason', '')}")
        print(f"  p_dir legacy = {RECOVERY_P_DIR:.3f}")
        print(f"  p_dir exist/sell = {RECOVERY_P_DIR_EXISTING:.3f} / {RECOVERY_P_DIR_SELL:.3f}")
        print(f"  p_dir new buy    = {RECOVERY_P_DIR_NEW_BUY:.3f}")
        print(f"  max moves    = {RECOVERY_MAX_APPROVED_MOVES}")
        print(f"  max new buys = {RECOVERY_MAX_NEW_BUY_NAMES}")
        print(f"  turnover cap = {RECOVERY_MAX_ONE_WAY_TURNOVER:.3%}")
        print(f"  buy top-up   = {RECOVERY_BUY_TOPUP_ENABLED}")
        print(f"  extra funders= {RECOVERY_MAX_EXTRA_FUNDING_NAMES}")

    recovery_decision = build_controlled_recovery_decision(
        hold_number=hold_number,
        base_decision=base_decision,
        experiment_state=experiment_state,
        config=asoc_rebalance_config_calm,
        recovery_state=recovery_state,
    )
    strip_mala_samples_from_rebalance_history(experiment_state, hold_number)
    return recovery_decision


# ---------------------------------------------------------------
# 2) Local/full-window runner with actual hold numbers
# ---------------------------------------------------------------


def run_local_hold_window(
    *,
    hold_start_number: int,
    hold_end_number: int,
    rebalance_entering_start_hold: bool,
    df_ret: pd.DataFrame,
    rolling_windows: list[dict],
    initial_w: np.ndarray,
    idx_name: str,
    asset_cols: list[str],
    rebalance_step_fn,
    initial_fit_te_target: float,
    roll_window: int,
    figdir: Path | str,
    portfolio_label: str,
    make_plots: bool,
    verbose: bool,
) -> dict:
    """Run a local sequence of holds and collect reproducibility outputs."""
    hold_numbers_all = list(range(int(hold_start_number), int(hold_end_number) + 1))
    inferred_asset_cols = [str(c) for c in asset_cols]
    idx_col = str(idx_name)
    p = len(inferred_asset_cols)

    w_current = np.asarray(initial_w, dtype=float).reshape(-1)
    if w_current.size != p:
        raise ValueError(f"initial_w has length {w_current.size}, but expected p={p}.")

    hold_rows = []
    decision_rows = []
    path_frames = []
    hold_start_dates = []
    weights_by_hold = {}
    decisions_by_hold = {}
    experiment_state = {
        "df_ret": df_ret,
        "idx_col": idx_col,
        "asset_cols": inferred_asset_cols,
        "rolling_windows": list(rolling_windows),
        "portfolio_label": portfolio_label,
        "roll_window": int(roll_window),
        "local_hold_start_number": int(hold_start_number),
        "local_hold_end_number": int(hold_end_number),
        "rebalance_entering_start_hold": bool(rebalance_entering_start_hold),
        "recovery_candidate_frames": [],
        "recovery_state_rows": [],
        "recovery_settings": {
            "target_te_bp": float(RECOVERY_TARGET_TE_BP),
            "target_gap_bp": float(RECOVERY_TARGET_GAP_BP),
            "shock_jump_bp": float(RECOVERY_SHOCK_JUMP_BP),
            "exit_gap_bp": float(RECOVERY_EXIT_GAP_BP),
            "p_recovery": float(RECOVERY_P_DIR),
            "eps_delta": float(RECOVERY_EPS_DELTA),
            "max_approved_moves": int(RECOVERY_MAX_APPROVED_MOVES),
            "max_new_buy_names": int(RECOVERY_MAX_NEW_BUY_NAMES),
            "max_one_way_turnover": float(RECOVERY_MAX_ONE_WAY_TURNOVER),
            "target_one_way_turnover": float(RECOVERY_TARGET_ONE_WAY_TURNOVER),
            "dust_floor": float(RECOVERY_DUST_FLOOR),
        },
    }

    for h in hold_numbers_all:
        win = rolling_windows[h - 1]
        hold_start, hold_end = get_hold_dates_from_window(win)
        print("\n" + "=" * 72)
        print(f"[Tracker maintenance] Hold H{h:02d}: {hold_start.date()} -> {hold_end.date()}")

        recovery_state_for_decision = compute_recovery_state_for_decision(
            hold_rows,
            hold_number=h,
            target_te_bp=RECOVERY_TARGET_TE_BP,
            target_gap_bp=RECOVERY_TARGET_GAP_BP,
            shock_jump_bp=RECOVERY_SHOCK_JUMP_BP,
            exit_gap_bp=RECOVERY_EXIT_GAP_BP,
            p_recovery=RECOVERY_P_DIR,
            max_turnover=RECOVERY_MAX_ONE_WAY_TURNOVER,
            max_moves=RECOVERY_MAX_APPROVED_MOVES,
            max_new_buys=RECOVERY_MAX_NEW_BUY_NAMES,
        )
        experiment_state["recovery_state_for_current_decision"] = recovery_state_for_decision
        print(
            "Recovery mode entering hold: "
            f"{recovery_state_for_decision['recovery_active_for_decision']} "
            f"({recovery_state_for_decision['reason']})"
        )

        is_local_start = (h == int(hold_start_number))
        if is_local_start and not bool(rebalance_entering_start_hold):
            decision = None
            w_for_hold = w_current.copy()
            fit_te_target = initial_fit_te_target
            rebalance_label = "starting portfolio"
        else:
            decision = rebalance_step_fn(
                hold_number=h,
                window=win,
                previous_w=w_current.copy(),
                experiment_state=experiment_state,
            )
            if not isinstance(decision, dict):
                raise TypeError("rebalance_step_fn must return a dictionary.")
            if "w_new" in decision:
                w_for_hold = np.asarray(decision["w_new"], dtype=float).reshape(-1)
            elif "w_rebalanced" in decision:
                w_for_hold = np.asarray(decision["w_rebalanced"], dtype=float).reshape(-1)
            else:
                raise KeyError("rebalance_step_fn output must contain 'w_new' or 'w_rebalanced'.")

            te_fit = decision.get("te_fit2", decision.get("te_fit", {}))
            fit_te_target = te_fit.get("new", np.nan) if isinstance(te_fit, dict) else np.nan
            rebalance_label = "no rebalance" if bool(decision.get("no_trade", False)) else "rebalance"

        decision_row = decision_row_from_rebalance_output(
            hold_number=h,
            decision=decision,
            previous_w=w_current,
            new_w=w_for_hold,
        )

        if decision is not None:
            rec_state = decision.get("recovery_state_for_decision", recovery_state_for_decision)
            decision_row.update({
                "recovery_mode_active": bool(decision.get("recovery_mode_active", False)),
                "controlled_recovery_replaced_normal_decision": bool(decision.get("controlled_recovery_replaced_normal_decision", False)),
                "recovery_reason": str(rec_state.get("reason", "")),
                "recovery_prev_hold_te_bp": _safe_float_or_nan(rec_state.get("prev_hold_te_bp", np.nan)),
                "recovery_p_dir": _safe_float_or_nan(decision.get("recovery_p_dir", RECOVERY_P_DIR if bool(decision.get("recovery_mode_active", False)) else np.nan)),
                "recovery_max_one_way_turnover": _safe_float_or_nan(decision.get("recovery_max_one_way_turnover", np.nan)),
                "S_size_raw_before_truncation": int(decision.get("S_size_raw_before_truncation", decision.get("S_size", 0))) if not pd.isna(decision.get("S_size", 0)) else 0,
                "S_size_after_truncation": int(decision.get("S_size_after_truncation", decision.get("S_size", 0))) if not pd.isna(decision.get("S_size", 0)) else 0,
                "turnover_cap_applied": bool(decision.get("turnover_cap_applied", False)),
                "turnover_before_cap": _safe_float_or_nan(decision.get("turnover_before_cap", np.nan)),
                "turnover_after_cap": _safe_float_or_nan(decision.get("turnover_after_cap", np.nan)),
                "buy_topup_enabled": bool(decision.get("buy_topup_enabled", False)),
                "buy_topup_used": bool(decision.get("buy_topup_used", False)),
                "buy_topup_skip_reason": str(decision.get("buy_topup_skip_reason", "")),
                "approved_buy_amount_raw": _safe_float_or_nan(decision.get("approved_buy_amount_raw", np.nan)),
                "approved_sell_amount_raw": _safe_float_or_nan(decision.get("approved_sell_amount_raw", np.nan)),
                "extra_funding_amount": _safe_float_or_nan(decision.get("extra_funding_amount", np.nan)),
                "n_extra_funders": int(decision.get("n_extra_funders", 0) or 0),
                "implemented_buy_amount": _safe_float_or_nan(decision.get("implemented_buy_amount", np.nan)),
            })
            decisions_by_hold[h] = decision
        else:
            decision_row.update({
                "recovery_mode_active": False,
                "controlled_recovery_replaced_normal_decision": False,
                "recovery_reason": str(recovery_state_for_decision.get("reason", "")),
                "recovery_prev_hold_te_bp": np.nan,
                "recovery_p_dir": np.nan,
                "recovery_max_one_way_turnover": np.nan,
                "S_size_raw_before_truncation": 0,
                "S_size_after_truncation": 0,
                "turnover_cap_applied": False,
                "turnover_before_cap": np.nan,
                "turnover_after_cap": np.nan,
                "buy_topup_enabled": False,
                "buy_topup_used": False,
                "buy_topup_skip_reason": "",
                "approved_buy_amount_raw": np.nan,
                "approved_sell_amount_raw": np.nan,
                "extra_funding_amount": np.nan,
                "n_extra_funders": 0,
                "implemented_buy_amount": np.nan,
            })
            decisions_by_hold[h] = None

        decision_rows.append(decision_row)
        w_current = w_for_hold.copy()
        weights_by_hold[h] = w_current.copy()

        hold_data = extract_hold_data_from_window(
            df_ret,
            win,
            asset_cols=inferred_asset_cols,
            idx_col=idx_col,
        )
        hold_start_dates.append(hold_data["dates"][0])

        hold_eval = evaluate_tracker_on_hold(
            hold_number=h,
            y_raw=hold_data["y_raw"],
            R_raw=hold_data["R_raw"],
            dates=hold_data["dates"],
            w=w_current,
            fit_te_target=fit_te_target,
            rebalance_label=rebalance_label,
            rebalance_decision=decision,
            roll_window=roll_window,
            portfolio_label=portfolio_label,
        )
        hold_rows.append(hold_eval["row"])
        path_frames.append(hold_eval["paths"])

        recovery_state_after_hold = compute_recovery_state_for_decision(
            hold_rows,
            hold_number=h + 1,
            target_te_bp=RECOVERY_TARGET_TE_BP,
            target_gap_bp=RECOVERY_TARGET_GAP_BP,
            shock_jump_bp=RECOVERY_SHOCK_JUMP_BP,
            exit_gap_bp=RECOVERY_EXIT_GAP_BP,
            p_recovery=RECOVERY_P_DIR,
            max_turnover=RECOVERY_MAX_ONE_WAY_TURNOVER,
            max_moves=RECOVERY_MAX_APPROVED_MOVES,
            max_new_buys=RECOVERY_MAX_NEW_BUY_NAMES,
        )
        recovery_state_after_hold["evaluated_after_hold"] = int(h)
        experiment_state.setdefault("recovery_state_rows", []).append(recovery_state_after_hold)

        row = hold_eval["row"]
        print(f"Decision entering hold: {rebalance_label}")
        if decision is not None and bool(decision.get("no_trade", False)):
            print(f"No-trade reason: {decision.get('no_trade_reason', '')}")
        print(f"TE(realised) = {row['TE_RMSE']:.6e} ({row['TE_bp']:.3f} bp)")
        print(f"turnover entering hold = {row['turnover']:.6e} | trades = {row['n_trades']} | nnz = {row['nnz']}")

    df_hold_summary = pd.DataFrame(hold_rows)
    df_decision_summary = pd.DataFrame(decision_rows)

    if len(path_frames) > 0:
        paths = pd.concat(path_frames, ignore_index=True)
        paths["cum_index_global"] = np.cumprod(1.0 + paths["index_return"].to_numpy())
        paths["cum_tracker_global"] = np.cumprod(1.0 + paths["tracker_return"].to_numpy())
        paths["cum_active_global"] = np.cumsum(paths["active_return"].to_numpy())
        paths["cum_index"] = paths["cum_index_global"]
        paths["cum_tracker"] = paths["cum_tracker_global"]
        paths["cum_active"] = paths["cum_active_global"]
    else:
        paths = pd.DataFrame()

    result = {
        "hold_start_number": int(hold_start_number),
        "hold_end_number": int(hold_end_number),
        "df_hold_summary": df_hold_summary,
        "df_decision_summary": df_decision_summary,
        "paths": paths,
        "hold_start_dates": hold_start_dates,
        "weights_by_hold": weights_by_hold,
        "decisions_by_hold": decisions_by_hold,
        "final_w": w_current.copy(),
        "asset_cols": inferred_asset_cols,
        "idx_col": idx_col,
        "rolling_windows_used": [rolling_windows[h - 1] for h in weights_by_hold.keys()],
        "portfolio_label": portfolio_label,
        "roll_window": int(roll_window),
        "experiment_state": experiment_state,
        "recovery_state_log": pd.DataFrame(experiment_state.get("recovery_state_rows", [])),
        "recovery_candidates": (
            pd.concat(experiment_state.get("recovery_candidate_frames", []), ignore_index=True)
            if experiment_state.get("recovery_candidate_frames", []) else pd.DataFrame()
        ),
    }

    if make_plots and len(hold_rows) > 0:
        plot_sequential_tracker_results(
            result,
            figdir=figdir,
            roll_window=roll_window,
            portfolio_label=portfolio_label,
        )

    return result


# ---------------------------------------------------------------
# 3) Composition export helper

# ---------------------------------------------------------------

def export_weight_compositions_for_local_run(
    *,
    seq_out: dict,
    outdir: Path | str,
    run_label: str,
    top_n: int = 50,
    nnz_tol: float = 1e-12,
) -> dict:
    """
    Export per-hold portfolio compositions for later comparison with
    competing methods.
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    run_label_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(run_label))

    asset_names = list(seq_out["asset_cols"])
    weights_by_hold = seq_out["weights_by_hold"]

    full_rows = []
    top_rows = []
    summary_rows = []

    for hold_number in sorted(weights_by_hold):
        w = np.asarray(weights_by_hold[hold_number], dtype=float).reshape(-1)
        abs_w = np.abs(w)
        l1 = float(abs_w.sum())
        order = np.argsort(-abs_w)
        nnz = int(np.count_nonzero(abs_w > nnz_tol))

        summary_rows.append({
            "hold": int(hold_number),
            "sum_weights": float(w.sum()),
            "nnz": nnz,
            "n_positive": int(np.count_nonzero(w > nnz_tol)),
            "n_negative": int(np.count_nonzero(w < -nnz_tol)),
            "l1_abs": l1,
            "max_abs_weight": float(abs_w.max()) if w.size else 0.0,
            "top10_L1_share": float(abs_w[order[:10]].sum() / l1) if l1 > 0 else np.nan,
            "top25_L1_share": float(abs_w[order[:25]].sum() / l1) if l1 > 0 else np.nan,
        })

        for j, ticker in enumerate(asset_names):
            full_rows.append({
                "hold": int(hold_number),
                "ticker": ticker,
                "weight": float(w[j]),
                "abs_weight": float(abs_w[j]),
                "is_active_holding": bool(abs_w[j] > nnz_tol),
            })

        for rank, j in enumerate(order[:top_n], start=1):
            top_rows.append({
                "hold": int(hold_number),
                "rank_abs_weight": int(rank),
                "ticker": asset_names[j],
                "weight": float(w[j]),
                "abs_weight": float(abs_w[j]),
            })

    df_full = pd.DataFrame(full_rows)
    df_top = pd.DataFrame(top_rows)
    df_summary = pd.DataFrame(summary_rows)

    full_csv = outdir / f"{run_label_safe}_full_compositions.csv"
    top_csv = outdir / f"{run_label_safe}_top{top_n}_compositions.csv"
    summary_csv = outdir / f"{run_label_safe}_composition_summary.csv"

    df_full.to_csv(full_csv, index=False)
    df_top.to_csv(top_csv, index=False)
    df_summary.to_csv(summary_csv, index=False)

    metadata = {
        "run_label": run_label_safe,
        "tracker_cache_path": str(TRACKER_CACHE_PATH),
        "tracker_cache_label": str(tracker_cache_label),
        "hold_start_number": int(seq_out["hold_start_number"]),
        "hold_end_number": int(seq_out["hold_end_number"]),
        "rebalance_entering_start_hold": bool(REBALANCE_ENTERING_START_HOLD),
        "selected_support_rule_fit": str(selected_support_rule_fit),
        "initial_fit_te_target": float(initial_fit_te_target),
        "initial_nnz": int(np.count_nonzero(np.abs(initial_w_for_local_run) > 1e-12)),
        "initial_sum": float(initial_w_for_local_run.sum()),
        "variant_kind": str(VARIANT_KIND),
        "variant_description": str(VARIANT_DESCRIPTION),
    }
    metadata_json = outdir / f"{run_label_safe}_composition_metadata.json"
    with open(metadata_json, "w") as f:
        json.dump(metadata, f, indent=2)

    print("\n[Composition export]")
    print(f"full compositions = {full_csv}")
    print(f"top holdings      = {top_csv}")
    print(f"summary           = {summary_csv}")
    print(f"metadata          = {metadata_json}")

    return {
        "full_csv": full_csv,
        "top_csv": top_csv,
        "summary_csv": summary_csv,
        "metadata_json": metadata_json,
        "df_full": df_full,
        "df_top": df_top,
        "df_summary": df_summary,
    }


# ---------------------------------------------------------------
# 4) Run full-horizon controlled-recovery experiment
# ---------------------------------------------------------------

seq_tracker_result = run_local_hold_window(
    hold_start_number=HOLD_START_NUMBER,
    hold_end_number=HOLD_END_NUMBER,
    rebalance_entering_start_hold=REBALANCE_ENTERING_START_HOLD,
    df_ret=df_ret,
    rolling_windows=rolling_windows,
    initial_w=initial_w_for_local_run,
    idx_name=idx_name_for_seq,
    asset_cols=asset_cols_for_seq,
    rebalance_step_fn=rebalance_step_fn_asoc_controlled_recovery_local,
    initial_fit_te_target=initial_fit_te_target,
    roll_window=20,
    figdir=figdir_seq,
    portfolio_label="paper-main-tracker-maintenance",
    make_plots=MAKE_PLOTS,
    verbose=VERBOSE,
)

print("\n[Paper main tracker maintenance run] returned.")
print("Object: seq_tracker_result")
print(f"Output directory: {run_output_dir_seq}")

# Convenience handles expected from previous notebooks.
df_hold_summary_final = seq_tracker_result["df_hold_summary"]
df_decision_summary_final = seq_tracker_result["df_decision_summary"]
decisions_final = seq_tracker_result["decisions_by_hold"]
weights_final = seq_tracker_result["weights_by_hold"]
paths_final = seq_tracker_result["paths"]

composition_exports = export_weight_compositions_for_local_run(
    seq_out=seq_tracker_result,
    outdir=composition_dir_seq,
    run_label=run_label_seq,
    top_n=50,
)

# ---------------------------------------------------------------
# 5) Variant-specific floor/funding logs, small-weight summary, metadata
# ---------------------------------------------------------------

floor_action_frames = []
if isinstance(initial_floor_actions_df, pd.DataFrame) and len(initial_floor_actions_df) > 0:
    floor_action_frames.append(initial_floor_actions_df.copy())

if VARIANT_KIND == "dust_funding005":
    funding_actions_df = extract_dust_funding_action_rows(
        seq_tracker_result,
        run_label=run_label_seq,
        dust_floor=DUST_FLOOR,
    )
    if len(funding_actions_df) > 0:
        floor_action_frames.append(funding_actions_df)

# For recovery funding-top-up branches, record extra funders in the floor/action log.
_recovery_candidates_for_actions = seq_tracker_result.get("recovery_candidates", pd.DataFrame())
if (
    isinstance(_recovery_candidates_for_actions, pd.DataFrame)
    and len(_recovery_candidates_for_actions) > 0
    and "used_as_extra_funder" in _recovery_candidates_for_actions.columns
):
    _fund_rows = _recovery_candidates_for_actions[
        _recovery_candidates_for_actions["used_as_extra_funder"].astype(bool)
    ].copy()
    if len(_fund_rows) > 0:
        _fund_actions = pd.DataFrame({
            "variant": run_label_seq,
            "hold": _fund_rows["hold"].astype(int),
            "action_stage": "recovery_funding_topup",
            "ticker": _fund_rows["ticker"].astype(str),
            "old_weight": _fund_rows["current_weight"].astype(float),
            "new_weight": _fund_rows["current_weight"].astype(float) + _fund_rows["implemented_dw"].astype(float),
            "delta_weight": _fund_rows["implemented_dw"].astype(float),
            "reason": _fund_rows["funding_reason"].astype(str),
            "floor": float(DUST_FLOOR),
            "realloc": "local_recovery_funding_topup",
            "mass_pruned_total": np.nan,
            "extra_funding_amount": _fund_rows["extra_funding_amount"].astype(float),
            "funding_rank": _fund_rows["funding_rank"],
        })
        floor_action_frames.append(_fund_actions)

if floor_action_frames:
    df_floor_actions = pd.concat(floor_action_frames, ignore_index=True)
else:
    df_floor_actions = pd.DataFrame(
        columns=[
            "variant", "hold", "action_stage", "ticker", "old_weight", "new_weight",
            "delta_weight", "reason", "floor", "realloc", "mass_pruned_total",
        ]
    )

floor_actions_csv = run_output_dir_seq / f"{run_label_seq}_floor_actions.csv"
df_floor_actions.to_csv(floor_actions_csv, index=False)

small_weight_summary_df = build_small_weight_summary(
    seq_tracker_result,
    run_label=run_label_seq,
)
small_weight_summary_csv = run_output_dir_seq / f"{run_label_seq}_small_weight_summary.csv"
small_weight_summary_df.to_csv(small_weight_summary_csv, index=False)


# Recovery diagnostics requested for this branch.
recovery_state_log_df = seq_tracker_result.get("recovery_state_log", pd.DataFrame())
recovery_state_log_csv = run_output_dir_seq / f"{run_label_seq}_recovery_state_log.csv"
recovery_state_log_df.to_csv(recovery_state_log_csv, index=False)

recovery_candidates_df = seq_tracker_result.get("recovery_candidates", pd.DataFrame())
recovery_candidates_csv = run_output_dir_seq / f"{run_label_seq}_recovery_candidates.csv"
recovery_candidates_df.to_csv(recovery_candidates_csv, index=False)


# Final numerical outputs.  These replace the development notebook's
# incremental experiment-management outputs and are written only once, after the full run.
hold_summary_csv = run_output_dir_seq / f"{run_label_seq}_hold_summary.csv"
decision_summary_csv = run_output_dir_seq / f"{run_label_seq}_decision_summary.csv"
paths_csv = run_output_dir_seq / f"{run_label_seq}_daily_paths.csv"
final_weights_csv = run_output_dir_seq / f"{run_label_seq}_final_weights.csv"

df_hold_summary_final.to_csv(hold_summary_csv, index=False)
df_decision_summary_final.to_csv(decision_summary_csv, index=False)
paths_final.to_csv(paths_csv, index=False)

pd.DataFrame({
    "ticker": asset_cols_for_seq,
    "weight": seq_tracker_result["final_w"],
    "abs_weight": np.abs(seq_tracker_result["final_w"]),
    "is_active": np.abs(seq_tracker_result["final_w"]) > 1e-12,
}).sort_values("abs_weight", ascending=False).to_csv(final_weights_csv, index=False)


def collect_delta_w_audit_tables(seq_out: dict) -> dict:
    """Collect compact Delta-w audit tables from in-memory rebalance history."""
    hist = seq_out.get("experiment_state", {}).get("rebalance_history", {})
    all_grid_rows = []
    selected_rows = []
    mala_rows = []
    frontier_rows = []
    selected_move_rows = []

    for hold_number, hobj in sorted(hist.items()):
        cgrid = hobj.get("cgrid_out", {})
        if isinstance(cgrid.get("df_dw_grid", None), pd.DataFrame):
            df_grid_h = cgrid["df_dw_grid"].copy()
            df_grid_h.insert(0, "hold", int(hold_number))
            all_grid_rows.append(df_grid_h)
        if isinstance(cgrid.get("selected_row", None), dict):
            row = dict(cgrid["selected_row"])
            row["hold"] = int(hold_number)
            selected_rows.append(row)

        mala = hobj.get("mala_out", {})
        if isinstance(mala, dict) and mala:
            te_summary = mala.get("te_summary", {}) or {}
            top_idx = np.asarray(mala.get("top_idx", []), dtype=int).reshape(-1)
            ess_coords = np.asarray(mala.get("ess_coords", []), dtype=float).reshape(-1)
            sent = []
            for j in top_idx[:4]:
                if 0 <= int(j) < ess_coords.size:
                    sent.append(f"j={int(j)}:{ess_coords[int(j)]:.1f}")
            mala_rows.append({
                "hold": int(hold_number),
                "sigma2_dw": float(mala.get("sigma2_dw", np.nan)),
                "kappa_dw": float(mala.get("kappa_dw", np.nan)),
                "lambda_MY": float(mala.get("lambda_MY", np.nan)),
                "gamma": float(mala.get("gamma", np.nan)),
                "accept_rate": float(mala.get("acc_rate", np.nan)),
                "n_iter": int(mala.get("n_iter", 0) or 0),
                "thin": int(mala.get("thin", 0) or 0),
                "burn_kept": int(mala.get("burn_kept", 0) or 0),
                "wall_seconds": float(mala.get("wall_seconds", np.nan)),
                "ESS_TE": float(mala.get("ess_te", np.nan)),
                "TE_mean_bp": 1e4 * float(te_summary.get("mean", np.nan)),
                "TE_median_bp": 1e4 * float(te_summary.get("median", np.nan)),
                "TE_q05_bp": 1e4 * float(te_summary.get("q05", np.nan)),
                "TE_q95_bp": 1e4 * float(te_summary.get("q95", np.nan)),
                "sentinel_ess": "; ".join(sent),
            })

        rope = hobj.get("rope_out", {})
        if isinstance(rope.get("df_frontier", None), pd.DataFrame):
            df_front_h = rope["df_frontier"].copy()
            df_front_h.insert(0, "hold", int(hold_number))
            frontier_rows.append(df_front_h)
        selected_candidate = rope.get("selected_candidate", {}) if isinstance(rope, dict) else {}
        if isinstance(selected_candidate, dict) and isinstance(selected_candidate.get("df_moves", None), pd.DataFrame):
            df_moves_h = selected_candidate["df_moves"].copy()
            df_moves_h.insert(0, "hold", int(hold_number))
            selected_move_rows.append(df_moves_h)

    return {
        "delta_w_cgrid_all_holds": pd.concat(all_grid_rows, ignore_index=True) if all_grid_rows else pd.DataFrame(),
        "delta_w_selected_rows": pd.DataFrame(selected_rows),
        "delta_w_mala_diagnostics": pd.DataFrame(mala_rows),
        "delta_w_rope_frontier_all_holds": pd.concat(frontier_rows, ignore_index=True) if frontier_rows else pd.DataFrame(),
        "delta_w_selected_candidate_moves": pd.concat(selected_move_rows, ignore_index=True) if selected_move_rows else pd.DataFrame(),
    }


dw_audit_tables = collect_delta_w_audit_tables(seq_tracker_result)
for name, df_out in dw_audit_tables.items():
    df_out.to_csv(run_output_dir_seq / f"{run_label_seq}_{name}.csv", index=False)

variant_metadata = {
    "run_label": run_label_seq,
    "run_mode": str(RUN_MODE),
    "run_production": bool(RUN_PRODUCTION),
    "random_seed": int(RANDOM_SEED),
    "python_version": __import__("sys").version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "tracker": TRACKER_RUN_SHORT_LABEL,
    "tracker_cache_path": str(TRACKER_CACHE_PATH),
    "tracker_cache_label": str(tracker_cache_label),
    "hold_start_number": int(HOLD_START_NUMBER),
    "hold_end_number": int(HOLD_END_NUMBER),
    "variant_kind": VARIANT_KIND,
    "variant_description": VARIANT_DESCRIPTION,
    "dust_floor": float(DUST_FLOOR),
    "initial_floor_applied_in_maintenance": bool(APPLY_INITIAL_FLOOR),
    "initial_floor_expected_in_cache": True,
    "initial_floor_value": None if INITIAL_FLOOR is None else float(INITIAL_FLOOR),
    "initial_reallocation": INITIAL_REALLOC,
    "initial_turnover_from_cached_tracker": float(0.5 * np.sum(np.abs(initial_w_for_local_run - initial_tracker_w))),
    "initial_nnz_before_variant": int(np.count_nonzero(np.abs(initial_tracker_w) > 1e-12)),
    "initial_nnz_after_variant": int(np.count_nonzero(np.abs(initial_w_for_local_run) > 1e-12)),
    "normal_mode_candidate_rule": str(asoc_rebalance_config_calm.selected_rope_candidate),
    "baseline_no_recovery_counterfactual": False,
    "controlled_recovery_enabled": True,
    "controlled_recovery_base": VARIANT_KIND,
    "recovery_p_dir": float(RECOVERY_P_DIR),
        "recovery_p_dir_existing": float(RECOVERY_P_DIR_EXISTING),
        "recovery_p_dir_new_buy": float(RECOVERY_P_DIR_NEW_BUY),
        "recovery_p_dir_sell": float(RECOVERY_P_DIR_SELL),
    "recovery_eps_delta": float(RECOVERY_EPS_DELTA),
    "recovery_max_approved_moves": int(RECOVERY_MAX_APPROVED_MOVES),
    "recovery_max_new_buy_names": int(RECOVERY_MAX_NEW_BUY_NAMES),
    "recovery_max_one_way_turnover": float(RECOVERY_MAX_ONE_WAY_TURNOVER),
    "recovery_target_one_way_turnover": float(RECOVERY_TARGET_ONE_WAY_TURNOVER),
    "recovery_buy_topup_enabled": bool(RECOVERY_BUY_TOPUP_ENABLED),
    "recovery_max_extra_funding_names": int(RECOVERY_MAX_EXTRA_FUNDING_NAMES),
    "recovery_protect_top_n_funders": int(RECOVERY_PROTECT_TOP_N_FUNDERS),
    "recovery_protect_weight_above": float(RECOVERY_PROTECT_WEIGHT_ABOVE),
    "recovery_trigger_target_te_bp": float(RECOVERY_TARGET_TE_BP),
    "recovery_trigger_target_gap_bp": float(RECOVERY_TARGET_GAP_BP),
    "recovery_trigger_shock_jump_bp": float(RECOVERY_SHOCK_JUMP_BP),
    "no_retrospective_competitor_information_used": True,
    "global_renormalisation_used": False,
    "initial_floor_redistribution_logged_in_maintenance": bool(APPLY_INITIAL_FLOOR),
    "numerical_settings": {
        "dw_sapg_n_iter": int(DW_SAPG_N_ITER),
        "dw_fista_max_iter": int(DW_FISTA_MAX_ITER),
        "dw_mala_n_burn_steps": int(DW_MALA_N_BURN_STEPS),
        "dw_mala_n_iter_long": int(DW_MALA_N_ITER_LONG),
        "dw_mala_thin": int(DW_MALA_THIN),
        "dw_mala_burn_kept": int(DW_MALA_BURN_KEPT),
        "dw_mala_max_lag_ess": int(DW_MALA_MAX_LAG_ESS),
    },
    "buy_only_dust_tol": float(asoc_rebalance_config_calm.buy_only_dust_tol),
    "max_buy_only_funding_names": int(asoc_rebalance_config_calm.max_buy_only_funding_names),
    "outputs": {
        "hold_summary_csv": str(hold_summary_csv),
        "decision_summary_csv": str(decision_summary_csv),
        "paths_csv": str(paths_csv),
        "floor_actions_csv": str(floor_actions_csv),
        "small_weight_summary_csv": str(small_weight_summary_csv),
        "recovery_state_log_csv": str(recovery_state_log_csv),
        "recovery_candidates_csv": str(recovery_candidates_csv),
        "composition_full_csv": str(composition_exports["full_csv"]),
        "composition_top_csv": str(composition_exports["top_csv"]),
        "composition_summary_csv": str(composition_exports["summary_csv"]),
        "final_weights_csv": str(final_weights_csv),
        **{name: str(run_output_dir_seq / f"{run_label_seq}_{name}.csv") for name in dw_audit_tables.keys()},
    },
}

variant_metadata_json = run_output_dir_seq / f"{run_label_seq}_metadata.json"
with open(variant_metadata_json, "w") as f:
    json.dump(variant_metadata, f, indent=2)


print("\n[Variant exports]")
print(f"floor actions        = {floor_actions_csv}")
print(f"small weight summary = {small_weight_summary_csv}")
print(f"metadata             = {variant_metadata_json}")
print(f"recovery state log   = {recovery_state_log_csv}")
print(f"recovery candidates  = {recovery_candidates_csv}")

print("\n[Recovery state log]")
print(recovery_state_log_df.to_string(index=False) if len(recovery_state_log_df) else "(empty)")

print("\n[Recovery candidate rows saved]")
print(f"rows = {len(recovery_candidates_df)}")

print("\n[Small-weight summary]")
print(small_weight_summary_df.to_string(index=False))

print("\n[Floor/funding action log]")
if len(df_floor_actions) > 0:
    print(df_floor_actions.to_string(index=False))
else:
    print("No floor/funding actions were logged for this variant.")


print("\n[Hold summary]")
display_cols_hold = [
    "hold", "portfolio", "start", "end", "TE_bp", "MAE_bp",
    "mean_active_bp", "std_active_bp", "turnover", "n_trades",
    "nnz", "n_shorts", "hit_rate_vs_FIT_TE", "max_active_drawdown",
]
display_cols_hold = [c for c in display_cols_hold if c in df_hold_summary_final.columns]
print(
    df_hold_summary_final[display_cols_hold].to_string(
        index=False,
        formatters={
            "TE_bp": lambda x: f"{x:.3f}",
            "MAE_bp": lambda x: f"{x:.3f}",
            "mean_active_bp": lambda x: f"{x:+.3f}",
            "std_active_bp": lambda x: f"{x:.3f}",
            "turnover": lambda x: f"{x:.6e}",
            "hit_rate_vs_FIT_TE": lambda x: "" if pd.isna(x) else f"{x:.3f}",
            "max_active_drawdown": lambda x: f"{x:+.6e}",
        },
    )
)

print("\n[Decision summary]")
display_cols_decision = [
    "hold", "decision_type", "no_trade", "implementation_case", "turnover",
    "n_trades", "candidate_name", "S_size", "fit_TE_new_bp", "no_trade_reason",
]
display_cols_decision = [c for c in display_cols_decision if c in df_decision_summary_final.columns]
print(
    df_decision_summary_final[display_cols_decision].to_string(
        index=False,
        formatters={
            "turnover": lambda x: f"{x:.6e}",
            "fit_TE_new_bp": lambda x: "" if pd.isna(x) else f"{x:.3f}",
        },
    )
)

print("\n[Composition summary]")
print(composition_exports["df_summary"].to_string(index=False))


In [ ]:
# ===============================================================
# Paper-number validation for production maintenance runs
# ===============================================================
# This cell checks the headline values reported in the manuscript.
# Tolerances are deliberately small but nonzero to allow for tiny stochastic
# variation when long MALA runs are repeated on a different platform.

EXPECTED_MAINTENANCE = {
    "mean_te_bp": 20.556,
    "median_te_bp": 19.409,
    "max_te_bp": 36.074,
    "total_turnover_pct": 5.0,
    "final_active_names": 55,
}

if RUN_PRODUCTION:
    df_val = df_hold_summary_final.copy()
    observed = {
        "mean_te_bp": float(df_val["TE_bp"].mean()),
        "median_te_bp": float(df_val["TE_bp"].median()),
        "max_te_bp": float(df_val["TE_bp"].max()),
        "total_turnover_pct": 100.0 * float(df_val["turnover"].sum()),
        "final_active_names": int(np.count_nonzero(np.asarray(seq_tracker_result["final_w"]) > 1e-12)),
    }

    tolerances = {
        "mean_te_bp": 0.05,
        "median_te_bp": 0.05,
        "max_te_bp": 0.05,
        "total_turnover_pct": 0.02,
        "final_active_names": 0,
    }

    print("\n[Paper-number validation: maintenance]")
    for key, expected in EXPECTED_MAINTENANCE.items():
        obs = observed[key]
        tol = tolerances[key]
        ok = abs(float(obs) - float(expected)) <= tol
        status = "OK" if ok else "CHECK"
        print(f"{status:5s} {key:24s} observed={obs} expected={expected} tol={tol}")
else:
    print("[Paper-number validation skipped: RUN_PRODUCTION=False smoke mode]")
